In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

TB_KDE_LOO = False
os.environ["KDE_OUTPUT_DIR"] = (r"C:\eshaghi\outputs_LOO" if TB_KDE_LOO
                                else r"C:\eshaghi\outputs_EXT")
import time
import json
import itertools
from pathlib import Path
from typing import Optional, Dict, Tuple
from collections import Counter
import collections
import copy
from sklearn.kernel_approximation import Nystroem
import numpy as np
from scipy.special import logsumexp as _lse_loo

def kde_logdens_at_samples(X, bw):
    X = np.asarray(X, float)
    m, d = X.shape
    if not TB_KDE_LOO:
        from sklearn.neighbors import KernelDensity
        return KernelDensity(kernel='gaussian', bandwidth=bw).fit(X).score_samples(X)
    sq = np.sum(X ** 2, axis=1)
    D2 = np.maximum(sq[:, None] + sq[None, :] - 2.0 * X @ X.T, 0.0)
    logK = -D2 / (2.0 * bw ** 2) - 0.5 * d * np.log(2.0 * np.pi * bw ** 2)
    np.fill_diagonal(logK, -np.inf)
    return _lse_loo(logK, axis=1) - np.log(m - 1)

import pandas as pd
from joblib import Parallel, delayed
from sklearn.datasets import load_svmlight_file
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, LabelBinarizer, MinMaxScaler
from sklearn.metrics import pairwise_distances
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KernelDensity, NearestNeighbors
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from numpy.linalg import LinAlgError
import zlib
import scipy.linalg as sla
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

RANDOM_STATES      = [123, 456, 789]
RANDOM_STATE        = RANDOM_STATES[0]

LABEL_NOISE_RATIOS = [0.0, 0.1, 0.2, 0.3, 0.4]

PRUNE_ROB_WINS      = False
CURRENT_LABEL_NOISE = LABEL_NOISE_RATIOS[0]

SELECTION_MODE     = "legacy"
INNER_SCHEME       = "holdout"
INNER_VAL_FRACTION = 0.30
INNER_FOLDS        = 3
N_OUTER_FOLDS      = 5

RUN_ABLATION       = True

RUN_TIMING         = True
TIMING_REPEATS     = 3

OUTPUT_DIR         = Path(os.environ.get("KDE_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

N_JOBS             = int(os.environ.get("N_JOBS", "-1"))

FLAT_PARALLEL      = os.environ.get("FLAT_PARALLEL", "1") == "1"

DATASET_BATCH      = int(os.environ.get("DATASET_BATCH", "0"))

DATASET_PARALLEL   = int(os.environ.get("DATASET_PARALLEL", "1"))
SHARD_ID           = int(os.environ.get("SHARD_ID", "0"))
N_SHARDS           = int(os.environ.get("N_SHARDS", "1"))

MAKE_REPORTS       = os.environ.get("MAKE_REPORTS", "1") == "1"
RESULTS_DIR        = OUTPUT_DIR
RAW_SELECTION_CSV  = OUTPUT_DIR / "raw_selection_grid.csv"
NESTED_OUTER_CSV   = OUTPUT_DIR / "nested_outer_folds.csv"
SELECTED_CFG_CSV   = OUTPUT_DIR / "selected_hyperparameters.csv"
TIMING_CSV         = OUTPUT_DIR / "single_model_timing.csv"

PROGRESS_CSV       = OUTPUT_DIR / "progress_summary.csv"

COMPLETED_CSV      = OUTPUT_DIR / "completed_units.csv"
RESUME             = True

DISPLAY_NAMES = {
    "Australian":      "Australian",
    "Banknote":        "Banknote",
    "Breastcancer":    "Breast cancer",
    "Diabets_Local":   "Pima diabetes",
    "Haberman":        "Haberman",
    "Heart_Local":     "Heart",
    "Ionosphere":      "Ionosphere",
    "Iris_bin":        "Iris",
    "Sonar_UCI":       "Sonar",
    "liver-disorders": "Liver disorders",
    "spolice":         "Splice",
}

def disp(name: str) -> str:
    return DISPLAY_NAMES.get(name, name)

ALL_METHODS   = ["Std", "LSW", "GBF", "GB", "CQS", "Rob"]
ABLATION_ARMS = ["RobHua", "RobSelf"]
METHOD_LABELS = {
    "Std": "LSTSVM", "LSW": "LSWLTSVM", "APL": "APL-RLSTSVM", "GBF": "GBFTSVM", "CQS": "CL1QTSVM",
    "Rob": "KDEWLSTSVM", "GB": "LS-GBLSTSVM",
    "RobHua":  "KDEWLSTSVM (one-sided)",
    "RobSelf": "KDEWLSTSVM (opposite-class only)",
}

VERBOSE_NOISE_CHECK = True

NOISE_SEED_DECORRELATE = True

_SEED_OFFSETS = {"split": 0, "feature": 10_000, "label": 20_000}

def derive_seed(base_seed: int, tag: str, fold_index: int = 0) -> int:
    if not NOISE_SEED_DECORRELATE or tag == "split":
        return int(base_seed)
    return int(base_seed) + _SEED_OFFSETS[tag] + 97 * int(fold_index)

def safe_solve(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    try:
        c, low = sla.cho_factor(A, check_finite=False)
        return sla.cho_solve((c, low), B, check_finite=False)
    except Exception:
        try:
            return np.linalg.solve(A, B)
        except LinAlgError:
            return np.linalg.lstsq(A, B, rcond=None)[0]

class KernelCache:
    def __init__(self, X_train: np.ndarray, X_test: np.ndarray):
        self.D_tr = pairwise_distances(X_train, squared=True)
        self.D_te = pairwise_distances(X_test, X_train, squared=True)
        self._cache: Dict[float, tuple] = {}

    def get(self, gamma: float):
        if gamma not in self._cache:
            K_tr = np.exp(-gamma * self.D_tr)
            K_te = np.exp(-gamma * self.D_te)
            self._cache[gamma] = (K_tr, K_te)
        return self._cache[gamma]

In [ ]:
def to_pm1(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y).ravel()
    vals = np.unique(y)
    if set(vals) <= {-1, +1}:
        return y.astype(int)
    if len(vals) != 2:
        raise ValueError("Binary labels required")
    return np.where(y == vals[0], -1, +1).astype(int)

def add_noise(y: np.ndarray, noise_ratio: float, seed: int = RANDOM_STATE) -> np.ndarray:
    if noise_ratio <= 0:
        return y
    rng = np.random.default_rng(seed)
    n_noisy = int(noise_ratio * y.shape[0])
    noisy_idx = rng.choice(y.shape[0], size=n_noisy, replace=False)
    y_noisy = y.copy()
    y_noisy[noisy_idx] *= -1
    return y_noisy

def add_feature_noise(
    X: np.ndarray,
    col_noise_ratio: float,
    row_noise_ratio: float,
    seed: int = RANDOM_STATE
) -> np.ndarray:
    if col_noise_ratio <= 0 or row_noise_ratio <= 0:
        return X
    rng = np.random.default_rng(seed)
    n_samples, n_features = X.shape
    n_noisy_cols = max(0, int(col_noise_ratio * n_features))
    noisy_cols = rng.choice(n_features, size=n_noisy_cols, replace=False)
    X_noisy = X.copy()
    for col in noisy_cols:
        n_noisy_rows = max(0, int(row_noise_ratio * n_samples))
        noisy_rows = rng.choice(n_samples, size=n_noisy_rows, replace=False)
        X_noisy[noisy_rows, col] += rng.normal(0.0, 0.2, size=n_noisy_rows)
    return X_noisy

def _is_already_scaled(X: np.ndarray, tol: float = 0.01) -> bool:
    col_min = X.min(axis=0)
    col_max = X.max(axis=0)
    in_neg1_pos1 = (np.all(col_min >= -1.0 - tol) and np.all(col_max <=  1.0 + tol))
    in_0_pos1    = (np.all(col_min >= -tol)         and np.all(col_max <=  1.0 + tol))
    if not (in_neg1_pos1 or in_0_pos1):
        return False
    col_range = col_max - col_min
    has_meaningful_range = np.any(col_range > 0.5)
    if in_0_pos1 and not in_neg1_pos1:
        label = "[0,1]"
    elif in_neg1_pos1 and np.any(col_min < -tol):
        label = "[-1,1]"
    else:
        label = "[0,1] or [-1,1]"
    if has_meaningful_range:
        print(f"  [Scaler] detected scaled range {label}")
        return True
    return False

def load_libsvm(path: str):
    X, y = load_svmlight_file(path)
    X = X.toarray()
    X = X[:, X.std(axis=0) != 0]

    y = np.where(y == np.min(y), -1, +1)
    return X, y

class Dataset:
    def __init__(
        self,
        name: str,
        loader,
        label_noise_ratio: float = 0.2,
        feature_col_noise_ratio: float = 0.4,
        feature_row_noise_ratio: float = 0.4,
        seed: int = RANDOM_STATE
    ):
        self.name = name
        X, y = loader()
        self.X = X
        self.y = to_pm1(y)
        self.label_noise_ratio = label_noise_ratio
        self.feature_col_noise_ratio = feature_col_noise_ratio
        self.feature_row_noise_ratio = feature_row_noise_ratio
        self.seed = seed

def get_precomputed_folds(ds: Dataset, kernel: str, n_splits: int = 5,
                          label_noise_ratio: float = None):
    X, y = ds.X, ds.y
    noise = ds.label_noise_ratio if label_noise_ratio is None else label_noise_ratio
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=derive_seed(ds.seed, "split"))
    folds = []
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
        X_train_clean, X_test = X[train_index], X[test_index]
        y_train_clean, y_test = y[train_index], y[test_index]

        _k = X_train_clean.std(axis=0) > 0
        _lo, _hi = X_train_clean[:, _k].min(0), X_train_clean[:, _k].max(0)
        _r = np.where(_hi - _lo > 0, _hi - _lo, 1.0)
        X_train_clean = 2.0 * (X_train_clean[:, _k] - _lo) / _r - 1.0
        X_test = 2.0 * (X_test[:, _k] - _lo) / _r - 1.0
        X_train_noisy = add_feature_noise(
            X_train_clean,
            col_noise_ratio=ds.feature_col_noise_ratio,
            row_noise_ratio=ds.feature_row_noise_ratio,
            seed=derive_seed(ds.seed, "feature", fold_idx)
        )
        y_train_noisy = add_noise(
            y_train_clean, noise_ratio=noise,
            seed=derive_seed(ds.seed, "label", fold_idx)
        )
        if VERBOSE_NOISE_CHECK:
            flipped = int(np.sum(y_train_noisy != y_train_clean))
            n_tr    = len(y_train_clean)
            print(f"  [NoiseCheck] ds={ds.name} kernel={kernel} seed={ds.seed} "
                  f"fold={fold_idx} target={noise:.2f} n_train={n_tr} "
                  f"flipped={flipped} ({100.0 * flipped / max(1, n_tr):.1f}%)")
        cache = KernelCache(X_train_noisy, X_test) if kernel != 'linear' else None
        folds.append((X_train_noisy, y_train_noisy, X_test, y_test, cache))
    return folds

def _prepare_features(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    X = X[:, X.std(axis=0) != 0]
    if X.shape[1] == 0:
        raise ValueError("پس از حذف ستون‌های با واریانس صفر، ویژگی‌ای نماند.")

    return X

def _finalize_binary(X: np.ndarray, y: np.ndarray):
    X = _prepare_features(X)
    y = np.asarray(y, dtype=int).ravel()
    if X.shape[0] != y.shape[0]:
        raise ValueError(f"ناسازگاری ابعاد: X={X.shape[0]} ولی y={y.shape[0]}")
    if set(np.unique(y).tolist()) != {-1, 1}:
        raise ValueError(f"برچسب‌ها باید ±۱ باشند؛ یافت شد: {np.unique(y).tolist()}")
    if min((y == 1).sum(), (y == -1).sum()) < 5:
        raise ValueError("یکی از دو کلاس کمتر از ۵ نمونه دارد؛ دیتاست نامناسب است.")
    return X, y

def binarize_multiclass(cls, rule):
    cls = np.asarray(cls).ravel()
    kind = rule[0]
    if kind == "largest_vs_rest":
        vals, cnts = np.unique(cls, return_counts=True)
        pos = vals[int(np.argmax(cnts))]
        pos = pos.item() if hasattr(pos, "item") else pos
        keep = np.ones(len(cls), dtype=bool)
        if int((cnts == cnts.max()).sum()) > 1:
            print(f"  [Binarize][WARN] چند کلاس هم‌جمعیت‌اند (تعداد={cnts.max()})؛ "
                  f"انتخاب کلاس مثبت به ترتیب برچسب انجام شد و دلخواه است. "
                  f"برای این دیتاست قانون صریح (vs_rest / pair / group) اعلام کنید.")
        desc = f"largest_vs_rest (کلاس مثبت = {pos!r})"
    elif kind == "vs_rest":
        pos = rule[1]
        keep = np.ones(len(cls), dtype=bool)
        if not np.any(cls == pos):
            raise ValueError(f"کلاس مثبت {pos!r} در داده وجود ندارد؛ "
                             f"کلاس‌های موجود: {np.unique(cls).tolist()}")
        desc = f"vs_rest (کلاس مثبت = {pos!r})"
    elif kind == "group":
        pos_set = set(rule[1])
        keep = np.ones(len(cls), dtype=bool)
        desc = f"group (کلاس‌های مثبت = {sorted(pos_set)})"
    elif kind == "pair":
        a, b = rule[1], rule[2]
        keep = np.isin(cls, [a, b])
        desc = f"pair ({a!r} = +1 در برابر {b!r} = -1)"
    else:
        raise ValueError(f"قانون دوکلاسه‌سازی ناشناخته: {rule!r}")

    cls_k = cls[keep]
    if kind == "group":
        y = np.where(np.isin(cls_k, list(rule[1])), +1, -1)
    elif kind == "pair":
        y = np.where(cls_k == rule[1], +1, -1)
    elif kind == "vs_rest":
        y = np.where(cls_k == rule[1], +1, -1)
    else:
        vals, cnts = np.unique(cls, return_counts=True)
        y = np.where(cls_k == vals[int(np.argmax(cnts))], +1, -1)
    return keep, y.astype(int), desc

def binarize_and_finalize(X, cls, rule):
    keep, y, desc = binarize_multiclass(cls, rule)
    X, y = _finalize_binary(np.asarray(X, dtype=float)[keep], y)
    return X, y, desc

In [ ]:
class GranularBall:
    def __init__(self, data):
        self.data          = data[:, :]
        self.data_no_label = data[:, :-2]
        self.num, self.dim = self.data_no_label.shape
        self.center        = self.data_no_label.mean(0)
        self.label, self.purity, self.r = self.__get_label_and_purity_and_r()

    def __get_label_and_purity_and_r(self):
        count  = Counter(self.data[:, -2])
        label  = max(count, key=count.get)
        purity = count[label] / self.num
        arr    = np.array(self.data_no_label) - self.center
        r      = np.sum(np.sqrt(np.sum(np.square(arr), 1))) / len(self.data_no_label)
        return label, purity, r

    def split_2balls(self):

        clu = KMeans(n_clusters=2, n_init=1, max_iter=50, tol=1e-3,
                     random_state=0).fit(self.data_no_label)
        lbl = clu.labels_
        if sum(lbl == 0) and sum(lbl == 1):
            return GranularBall(self.data[lbl == 0]), GranularBall(self.data[lbl == 1])
        return GranularBall(self.data[0:1]), GranularBall(self.data[1:])

class GBList:
    def __init__(self, data=None):
        self.data           = data[:, :]
        self.granular_balls = [GranularBall(self.data)]

    def init_granular_balls(self, purity=1, min_sample=1):
        ll = len(self.granular_balls)
        i  = 0
        while True:
            if self.granular_balls[i].purity < purity and self.granular_balls[i].num > min_sample:
                b1, b2 = self.granular_balls[i].split_2balls()
                self.granular_balls[i] = b1
                self.granular_balls.append(b2)
                ll += 1
            else:
                i += 1
            if i >= ll:
                break
        self.data = self.get_data()

    def get_center(self):
        return np.array([b.center for b in self.granular_balls])

    def get_r(self):
        return np.array([b.r for b in self.granular_balls])

    def get_data(self):
        return np.vstack([b.data for b in self.granular_balls])

    def del_ball(self, purty=0., num_data=0):
        self.granular_balls = [b for b in self.granular_balls
                               if b.purity >= purty and b.num >= num_data]
        self.data = self.get_data()

def generate_ball_data(data, pur, delbals):
    num, _ = data[:, :-1].shape
    index  = np.arange(num).reshape(num, 1)
    data   = np.hstack((data, index))
    gb     = GBList(data)
    gb.init_granular_balls(purity=pur)
    gb.del_ball(num_data=delbals)
    centers  = gb.get_center().tolist()
    rs       = gb.get_r().tolist()
    balldata = []
    for i, ball in enumerate(gb.granular_balls):
        balldata.append([centers[i], rs[i], ball.label])
    return balldata

def gen_balls(data, pur, delbals):
    return [[b[0], b[1], b[2]] for b in generate_ball_data(data, pur, delbals)]

class GBLSTSVM:
    def __init__(self, c1=1000, c2=0.00001, pur=0.985, delbals=2,
                 kernel='linear', gamma=1.0):
        self.c1       = c1
        self.c2       = c2
        self.pur      = pur
        self.delbals  = delbals
        self.kernel   = kernel
        self.gamma    = gamma
        self.w1 = self.b1 = self.w2 = self.b2 = None
        self.reg_term = 0.0001

    def fit(self, X: np.ndarray, y: np.ndarray, K_tr: Optional[np.ndarray] = None):
        y    = y.reshape(-1, 1)
        data = np.hstack((X if self.kernel == 'linear' else K_tr, y))
        R_balls = gen_balls(data, pur=self.pur, delbals=self.delbals)
        if not R_balls:
            return

        Center = np.array([b[0] for b in R_balls])
        Radius = np.array([b[1] for b in R_balls]).reshape(-1, 1)
        Label  = np.array([b[2] for b in R_balls]).reshape(-1, 1)

        A_train = np.hstack((Center, Radius, Label))
        C1 = A_train[A_train[:, -1] == 1]
        C2 = A_train[A_train[:, -1] != 1]
        if C1.shape[0] == 0 or C2.shape[0] == 0:
            return

        A   = C1[:, :-2]
        B   = C2[:, :-2]
        R1  = np.asmatrix(C1[:, -2]).T
        R2  = np.asmatrix(C2[:, -2]).T
        e1  = np.ones((A.shape[0], 1))
        e2  = np.ones((B.shape[0], 1))
        mat_H = np.column_stack((A, e1))
        mat_G = np.column_stack((B, e2))

        HtH = mat_H.T @ mat_H
        GtG = mat_G.T @ mat_G
        reg = self.reg_term * np.eye(mat_H.shape[1])

        A_p1  = GtG + (1 / self.c1) * HtH + reg
        H_rhs = np.vstack([B.T @ (e2 + R2), e2.T @ (e2 + R2)])
        self.w1_raw = -safe_solve(A_p1, H_rhs)
        self.w1 = np.asarray(self.w1_raw[:-1]).ravel()

        self.b1 = np.asarray(self.w1_raw[-1]).item()

        A_p2  = HtH + (1 / self.c2) * GtG + reg
        G_rhs = np.vstack([A.T @ (e1 + R1), e1.T @ (e1 + R1)])
        self.w2_raw = safe_solve(A_p2, G_rhs)
        self.w2 = np.asarray(self.w2_raw[:-1]).ravel()
        self.b2 = np.asarray(self.w2_raw[-1]).item()

    def predict(self, X: np.ndarray, K_te: Optional[np.ndarray] = None):
        if self.w1 is None:
            return np.zeros(len(X), dtype=int)
        Z  = X if self.kernel == 'linear' else K_te
        y1 = Z.dot(self.w1) + self.b1
        y2 = Z.dot(self.w2) + self.b2

        return np.where(np.abs(y1) <= np.abs(y2), +1, -1).ravel()

class StdLSTSVM:
    def __init__(self, c1=0.1, c2=0.1, kernel='linear', gamma=1.0, **_ignored):

        self.c1, self.c2 = c1, c2
        self.kernel      = kernel
        self.gamma       = gamma
        self.w1 = self.w2 = None
        self.b1 = self.b2 = None

    def fit(self, X: np.ndarray, y: np.ndarray, K_tr: Optional[np.ndarray] = None):
        y    = y.ravel()
        idxA = np.where(y == +1)[0]
        idxB = np.where(y == -1)[0]
        if not len(idxA) or not len(idxB):
            return
        A, B = X[idxA], X[idxB]
        e1 = np.ones((len(A), 1))
        e2 = np.ones((len(B), 1))

        if self.kernel == 'linear':
            E = np.hstack([A, e1])
            F = np.hstack([B, e2])
        else:
            E = np.hstack([K_tr[idxA], e1])
            F = np.hstack([K_tr[idxB], e2])

        EtE = E.T @ E
        FtF = F.T @ F
        reg = 1e-4 * np.eye(E.shape[1])

        z1 = safe_solve(FtF + (1 / self.c1) * EtE + reg, -F.T @ e2)
        z2 = safe_solve(EtE + (1 / self.c2) * FtF + reg,  E.T @ e1)

        self.w1, self.b1 = z1[:-1].ravel(), z1[-1, 0]
        self.w2, self.b2 = z2[:-1].ravel(), z2[-1, 0]

    def predict(self, X: np.ndarray, K_te: Optional[np.ndarray] = None):
        if self.w1 is None:
            return np.zeros(len(X), dtype=int)
        Z  = X if self.kernel == 'linear' else K_te
        d1 = np.abs(Z.dot(self.w1) + self.b1)
        d2 = np.abs(Z.dot(self.w2) + self.b2)
        return np.where(d1 < d2, +1, -1)

class RobustGMMLSTSVM:

    def __init__(self, c1=0.1, c2=0.1, n_components=2, kernel='linear',
                 gamma=1.0, method='gmm', bandwidth=1.0, variant='full'):
        self.variant       = variant
        self.c1, self.c2   = c1, c2
        self.n_comp        = n_components
        self.kernel        = kernel
        self.gamma         = gamma
        self.method        = method
        self.bandwidth     = bandwidth
        self.w1 = self.w2  = None
        self.b1 = self.b2  = None
        self._external_weights = None

    def set_external_weights(self, w_pos: np.ndarray, w_neg: np.ndarray):
        self._external_weights = (w_pos, w_neg)

    def _weights(self, X_cls):
        if len(X_cls) < 2:
            return np.ones(len(X_cls))
        if self.method == 'gmm':
            model = GaussianMixture(
                n_components=min(self.n_comp, len(X_cls)),
                covariance_type='full', random_state=42,
                max_iter=50, n_init=1
            )
            logd = model.fit(X_cls).score_samples(X_cls)
        else:

            logd = kde_logdens_at_samples(X_cls, self.bandwidth)
        return np.exp(logd)

    def fit(self, X: np.ndarray, y: np.ndarray, K_tr: Optional[np.ndarray] = None):
        y    = y.ravel()
        idxA = np.where(y == +1)[0]
        idxB = np.where(y == -1)[0]
        if not len(idxA) or not len(idxB):
            return
        A, B = X[idxA], X[idxB]

        if self._external_weights is not None:
            w1, w2 = self._external_weights
        else:
            w1 = self._weights(A)
            w2 = self._weights(B)

        e1 = np.ones((len(A), 1))
        e2 = np.ones((len(B), 1))

        if self.kernel == 'linear':
            E = np.hstack([A, e1])
            F = np.hstack([B, e2])
        else:
            E = np.hstack([K_tr[idxA], e1])
            F = np.hstack([K_tr[idxB], e2])

        Ew1E  = (E * w1[:, np.newaxis]).T @ E
        Fw2F  = (F * w2[:, np.newaxis]).T @ F
        Fw2e2 = (F * w2[:, np.newaxis]).T @ e2
        Ew1e1 = (E * w1[:, np.newaxis]).T @ e1

        reg  = 1e-4 * np.eye(E.shape[1])

        if self.variant == 'full':

            LHS1 = Fw2F + (1 / self.c1) * Ew1E + reg
            LHS2 = Ew1E + (1 / self.c2) * Fw2F + reg
        elif self.variant == 'hua':

            LHS1 = (F.T @ F) + (1 / self.c1) * Ew1E + reg
            LHS2 = (E.T @ E) + (1 / self.c2) * Fw2F + reg
        elif self.variant == 'self':

            LHS1 = Fw2F + (1 / self.c1) * (E.T @ E) + reg
            LHS2 = Ew1E + (1 / self.c2) * (F.T @ F) + reg
        else:
            raise ValueError(f"variant ناشناخته: {self.variant}")

        z1 = safe_solve(LHS1, -Fw2e2)
        z2 = safe_solve(LHS2, Ew1e1)

        self.w1, self.b1 = z1[:-1].ravel(), z1[-1, 0]
        self.w2, self.b2 = z2[:-1].ravel(), z2[-1, 0]

    def predict(self, X: np.ndarray, K_te: Optional[np.ndarray] = None):
        if self.w1 is None:
            return np.zeros(len(X), dtype=int)
        Z  = X if self.kernel == 'linear' else K_te
        d1 = np.abs(Z.dot(self.w1) + self.b1)
        d2 = np.abs(Z.dot(self.w2) + self.b2)
        return np.where(d1 < d2, +1, -1)

class LSWLTSVM:
    def __init__(self, c1=0.1, c2=0.1, k_neighbor=5, t_param=1.0,
                 kernel='linear', gamma=1.0):
        self.c1, self.c2   = c1, c2
        self.k, self.t     = int(k_neighbor), t_param
        self.kernel        = kernel
        self.gamma         = gamma
        self.w1 = self.w2  = None
        self.b1 = self.b2  = None

    def _within_weight_diag(self, X_cls) -> np.ndarray:
        n = X_cls.shape[0]
        if n < 2:
            return np.ones(n)
        k    = min(self.k, n - 1)
        nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree').fit(X_cls)
        dists, idxs = nbrs.kneighbors(X_cls)

        rows   = np.repeat(np.arange(n), k)
        cols   = idxs[:, 1:].ravel()
        ws     = np.exp(-(dists[:, 1:].ravel() ** 2) / self.t)

        W = np.zeros((n, n))
        W[rows, cols] = ws
        W[cols, rows] = ws

        return W.sum(axis=1)

    def fit(self, X, y, K_tr: Optional[np.ndarray] = None):
        y    = y.ravel()
        idx1 = np.where(y == +1)[0]
        idx2 = np.where(y == -1)[0]
        if not len(idx1) or not len(idx2):
            return
        X1, X2 = X[idx1], X[idx2]

        d1 = self._within_weight_diag(X1)
        d2 = self._within_weight_diag(X2)

        e1 = np.ones((len(X1), 1))
        e2 = np.ones((len(X2), 1))

        if self.kernel == 'linear':
            E = np.hstack([X1, e1])
            F = np.hstack([X2, e2])
        else:
            E = np.hstack([K_tr[idx1], e1])
            F = np.hstack([K_tr[idx2], e2])

        Ed1E  = (E * d1[:, np.newaxis]).T @ E
        Fd2F  = (F * d2[:, np.newaxis]).T @ F
        Fd2e2 = (F * d2[:, np.newaxis]).T @ e2
        Ed1e1 = (E * d1[:, np.newaxis]).T @ e1

        reg_E = 1e-4 * np.eye(E.shape[1])
        reg_F = 1e-4 * np.eye(F.shape[1])

        z1 = safe_solve(F.T @ F + (1 / self.c1) * Ed1E + reg_E, -Fd2e2)
        z2 = safe_solve(E.T @ E + (1 / self.c2) * Fd2F + reg_F,  Ed1e1)

        self.w1, self.b1 = z1[:-1].ravel(), z1[-1, 0]
        self.w2, self.b2 = z2[:-1].ravel(), z2[-1, 0]

    def predict(self, X, K_te: Optional[np.ndarray] = None):
        if self.w1 is None:
            return np.zeros(len(X), dtype=int)
        Z  = X if self.kernel == 'linear' else K_te
        d1 = np.abs(Z.dot(self.w1) + self.b1)
        d2 = np.abs(Z.dot(self.w2) + self.b2)
        return np.where(d1 < d2, +1, -1)

class APL_RLSTSVM:
    def __init__(self, c1=0.1, c2=0.1, theta=1.0, tau=0.5, max_iter=20,
                 kernel='linear', gamma=1.0):
        self.c1, self.c2   = c1, c2
        self.theta, self.tau = theta, tau
        self.max_iter      = max_iter
        self.kernel        = kernel
        self.gamma         = gamma
        self.eps           = 1e-4
        self.w1 = self.w2  = None
        self.b1 = self.b2  = None

    def _lambda(self, xi):
        xi  = xi.ravel()
        lam = np.zeros_like(xi)
        pos = xi >= 0
        neg = ~pos
        if pos.any():
            lam[pos] = (1 + self.tau * self.theta ** 2) * (xi[pos] + 2 * self.tau) \
                       / ((xi[pos] + self.tau) ** 2)
        if neg.any():
            x = -xi[neg]
            lam[neg] = (self.theta + (1 - self.theta) ** 2 * self.tau) * (x + 2 * self.tau) \
                       / ((x + self.tau) ** 2)
        return lam.reshape(-1, 1)

    def fit(self, X, y, K_tr: Optional[np.ndarray] = None):
        y    = y.ravel()
        idxA = np.where(y == +1)[0]
        idxB = np.where(y == -1)[0]
        if not len(idxA) or not len(idxB):
            return
        A, B   = X[idxA], X[idxB]
        m1, m2 = len(idxA), len(idxB)
        n      = len(X)
        e1, e2 = np.ones((m1, 1)), np.ones((m2, 1))

        if self.kernel == 'linear':
            H   = np.hstack([A, e1])
            G   = np.hstack([B, e2])
            u1  = (1 / n) * np.ones((H.shape[1], 1))
            u2  = u1.copy()
            HTH = H.T @ H
            GTG = G.T @ G
            I_H = self.eps * np.eye(H.shape[1])
            I_G = self.eps * np.eye(G.shape[1])

            conv_tol = 1e-6
            for _ in range(self.max_iter):
                u1_prev, u2_prev = u1, u2
                w1, b1 = u1[:-1], u1[-1:]
                w2, b2 = u2[:-1], u2[-1:]
                xi1 = B @ w1 + e2 * b1 + e2
                xi2 = -A @ w2 - e1 * b2 + e1
                L1  = np.diag(self._lambda(xi1).ravel())
                L2  = np.diag(self._lambda(xi2).ravel())
                u1  = -safe_solve(G.T @ L1 @ G + (1 / self.c1) * HTH + I_H, G.T @ L1 @ e2)
                u2  =  safe_solve(H.T @ L2 @ H + (1 / self.c2) * GTG + I_G, H.T @ L2 @ e1)
                if max(np.max(np.abs(u1 - u1_prev)),
                       np.max(np.abs(u2 - u2_prev))) < conv_tol:
                    break

            self.w1, self.b1 = u1[:-1].ravel(), u1[-1, 0]
            self.w2, self.b2 = u2[:-1].ravel(), u2[-1, 0]

        else:
            P    = np.hstack([K_tr[idxA], e1])
            Q    = np.hstack([K_tr[idxB], e2])
            v1   = (1 / n) * np.ones((n, 1))
            b1_v = np.array([[1 / n]])
            v2, b2_v = v1.copy(), b1_v.copy()
            I_aug = self.eps * np.eye(n + 1)

            conv_tol = 1e-6
            for _ in range(self.max_iter):
                v1_prev, b1_prev = v1, b1_v
                v2_prev, b2_prev = v2, b2_v
                xi1 = K_tr[idxB] @ v1 + e2 * b1_v + e2
                xi2 = -K_tr[idxA] @ v2 - e1 * b2_v + e1
                L1  = np.diag(self._lambda(xi1).ravel())
                L2  = np.diag(self._lambda(xi2).ravel())

                u1  = -safe_solve(Q.T @ L1 @ Q + (1 / self.c1) * P.T @ P + I_aug,
                                   Q.T @ L1 @ e2)
                u2  =  safe_solve(P.T @ L2 @ P + (1 / self.c2) * Q.T @ Q + I_aug,
                                   P.T @ L2 @ e1)
                v1, b1_v = u1[:-1], u1[-1:]
                v2, b2_v = u2[:-1], u2[-1:]
                if max(np.max(np.abs(v1 - v1_prev)), np.max(np.abs(b1_v - b1_prev)),
                       np.max(np.abs(v2 - v2_prev)), np.max(np.abs(b2_v - b2_prev))) < conv_tol:
                    break

            self.w1, self.b1 = v1.ravel(), b1_v[0, 0]
            self.w2, self.b2 = v2.ravel(), b2_v[0, 0]

    def predict(self, X, K_te: Optional[np.ndarray] = None):
        if self.w1 is None:
            return np.zeros(len(X), dtype=int)
        if self.kernel == 'linear':
            y1, y2 = X @ self.w1 + self.b1, X @ self.w2 + self.b2
        else:
            y1, y2 = K_te.dot(self.w1) + self.b1, K_te.dot(self.w2) + self.b2
        return np.where(np.abs(y1) <= np.abs(y2), +1, -1)

In [ ]:
def build_param_grids(kernel: str) -> dict:
    C_vals      = [10 ** i for i in range(-4, 4)]
    Theta_vals  = [10 ** i for i in range(-8, 5)]
    Tau_vals    = np.around(np.arange(0.1, 1.0, 0.1), 1).tolist()
    Gamma_range = [0.001, 0.01, 0.1, 1, 10] if kernel == 'rbf' else [None]
    C_pairs     = [(c, c) for c in C_vals]
    Pur_vals    = [0.90, 0.95, 0.985, 1.0]

    return {
        "Std": list(itertools.product(C_pairs, Gamma_range)),
        "Rob": list(itertools.product(
            ['kde'], C_pairs, [1, 2, 3],
            np.around(np.arange(0.1, 2.0, 0.2), 1).tolist(),
            Gamma_range
        )),
        "LSW": list(itertools.product(
            C_pairs, list(range(1, 9)),
            [2 ** i for i in range(-1, 6)],
            Gamma_range
        )),
        "APL": list(itertools.product(C_pairs, Theta_vals, Tau_vals, Gamma_range)),
        "GB":  list(itertools.product(C_pairs, Pur_vals, [2], Gamma_range))
    }

_ROB_VARIANT_OF = {"Rob": "full", "RobHua": "hua", "RobSelf": "self"}

def _build_binary_cls_and_params(method: str, params, kernel: str):
    if method == 'Std':
        (c1, c2), g = params
        return StdLSTSVM, dict(c1=c1, c2=c2, kernel=kernel, gamma=g or 1.0)
    elif method in _ROB_VARIANT_OF:
        kern_method, (c1, c2), ncomp, bw, g = params
        return RobustGMMLSTSVM, dict(
            c1=c1, c2=c2, n_components=ncomp, kernel=kernel,
            gamma=g or 1.0, method=kern_method, bandwidth=bw,
            variant=_ROB_VARIANT_OF[method]
        )
    elif method == 'LSW':
        (c1, c2), k, t, g = params
        return LSWLTSVM, dict(c1=c1, c2=c2, k_neighbor=k, t_param=t,
                              kernel=kernel, gamma=g or 1.0)
    elif method == 'APL':
        (c1, c2), th, ta, g = params
        return APL_RLSTSVM, dict(c1=c1, c2=c2, theta=th, tau=ta,
                                  max_iter=50, kernel=kernel, gamma=g or 1.0)
    elif method == 'GB':
        (c1, c2), pur, delbals, g = params
        return GBLSTSVM, dict(c1=c1, c2=c2, pur=pur, delbals=delbals,
                              kernel=kernel, gamma=g or 1.0)
    raise ValueError(f"روش ناشناخته: {method}")

def _grid_of(method: str, grids: dict):
    return grids["Rob"] if method in _ROB_VARIANT_OF else grids[method]

def evaluate(method: str, params, ds_name: str, kernel: str, precomputed_folds) -> dict:
    accuracies = []
    total_time = 0.0
    cls, kw = _build_binary_cls_and_params(method, params, kernel)
    for X_train, y_train, X_test, y_test, cache in precomputed_folds:
        model = cls(**kw)
        K_tr = K_te = None
        if kernel == 'rbf' and cache is not None:
            K_tr, K_te = cache.get(model.gamma)
        t0 = time.time()
        model.fit(X_train, y_train, K_tr=K_tr)
        y_pred = model.predict(X_test, K_te=K_te)
        accuracies.append(accuracy_score(y_test, y_pred))
        total_time += time.time() - t0
    return {
        'dataset':       ds_name,
        'kernel':        kernel,
        'method':        method,
        'params':        json.dumps(params),
        'mean_accuracy': np.mean(accuracies),
        'std_accuracy':  np.std(accuracies),
        'time_sec':      total_time
    }

_REG_EPS = 1e-4

def _split_params(method: str, params):
    if method == 'Std':
        (c1, c2), g = params
        return (g,), (c1, c2)
    if method in _ROB_VARIANT_OF:
        km, (c1, c2), ncomp, bw, g = params
        return (km, bw, g), (c1, c2)
    if method == 'LSW':
        (c1, c2), k, t, g = params
        return (k, t, g), (c1, c2)
    if method == 'GB':
        (c1, c2), pur, delbals, g = params
        return (pur, delbals, g), (c1, c2)
    return None, None

def _twin_predict_from_z(z1, z2, Z):
    w1, b1 = z1[:-1].ravel(), z1[-1, 0]
    w2, b2 = z2[:-1].ravel(), z2[-1, 0]
    d1 = np.abs(Z.dot(w1) + b1)
    d2 = np.abs(Z.dot(w2) + b2)
    return np.where(d1 < d2, +1, -1)

def evaluate_group(method: str, shared_key, param_list, ds_name: str,
                   kernel: str, precomputed_folds, rob_arms=("Rob",)) -> list:
    arms = list(rob_arms) if method == 'Rob' else [method]
    unique_C   = list(dict.fromkeys(_split_params(method, p)[1] for p in param_list))
    acc_per    = {(a, C): [] for a in arms for C in unique_C}
    time_per   = {(a, C): 0.0 for a in arms for C in unique_C}

    for X_train, y_train, X_test, y_test, kcache in precomputed_folds:
        t_sh = time.time()
        yv   = np.asarray(y_train).ravel()
        idxA = np.where(yv == +1)[0]
        idxB = np.where(yv == -1)[0]

        gamma = shared_key[-1]
        K_tr = K_te = None
        if kernel == 'rbf' and kcache is not None:
            K_tr, K_te = kcache.get(gamma if gamma else 1.0)
        Z_te = X_test if kernel == 'linear' else K_te

        degenerate = (len(idxA) == 0 or len(idxB) == 0)
        if not degenerate and method != 'GB':
            A, B = X_train[idxA], X_train[idxB]
            e1 = np.ones((len(A), 1))
            e2 = np.ones((len(B), 1))
            if kernel == 'linear':
                E = np.hstack([A, e1])
                F = np.hstack([B, e2])
            else:
                E = np.hstack([K_tr[idxA], e1])
                F = np.hstack([K_tr[idxB], e2])
            reg = _REG_EPS * np.eye(E.shape[1])
            EtE, FtF = E.T @ E, F.T @ F

            if method == 'Std':
                rhs1, rhs2 = F.T @ e2, E.T @ e1
            elif method == 'Rob':
                bw = shared_key[1]
                helper = RobustGMMLSTSVM(method='kde', bandwidth=bw)
                w1v = helper._weights(A)
                w2v = helper._weights(B)
                Ew1E = (E * w1v[:, np.newaxis]).T @ E
                Fw2F = (F * w2v[:, np.newaxis]).T @ F
                rhs1 = (F * w2v[:, np.newaxis]).T @ e2
                rhs2 = (E * w1v[:, np.newaxis]).T @ e1
            elif method == 'LSW':
                k, t = shared_key[0], shared_key[1]
                helper = LSWLTSVM(k_neighbor=k, t_param=t)
                d1v = helper._within_weight_diag(A)
                d2v = helper._within_weight_diag(B)
                Ed1E = (E * d1v[:, np.newaxis]).T @ E
                Fd2F = (F * d2v[:, np.newaxis]).T @ F
                rhs1 = (F * d2v[:, np.newaxis]).T @ e2
                rhs2 = (E * d1v[:, np.newaxis]).T @ e1
        elif not degenerate and method == 'GB':
            pur, delbals = shared_key[0], shared_key[1]
            data = np.hstack((X_train if kernel == 'linear' else K_tr,
                              yv.reshape(-1, 1)))
            R_balls = gen_balls(data, pur=pur, delbals=delbals)
            gb_ok = bool(R_balls)
            if gb_ok:
                Center = np.array([b[0] for b in R_balls])
                Radius = np.array([b[1] for b in R_balls]).reshape(-1, 1)
                Label  = np.array([b[2] for b in R_balls]).reshape(-1, 1)
                A_tr = np.hstack((Center, Radius, Label))
                C1b  = A_tr[A_tr[:, -1] == 1]
                C2b  = A_tr[A_tr[:, -1] != 1]
                gb_ok = (C1b.shape[0] > 0 and C2b.shape[0] > 0)
            if gb_ok:
                Ag, Bg = C1b[:, :-2], C2b[:, :-2]
                R1  = np.asarray(C1b[:, -2]).reshape(-1, 1)
                R2  = np.asarray(C2b[:, -2]).reshape(-1, 1)
                ge1 = np.ones((Ag.shape[0], 1))
                ge2 = np.ones((Bg.shape[0], 1))
                Hm  = np.column_stack((Ag, ge1))
                Gm  = np.column_stack((Bg, ge2))
                HtH, GtG = Hm.T @ Hm, Gm.T @ Gm
                H_rhs  = np.vstack([Bg.T @ (ge2 + R2), ge2.T @ (ge2 + R2)])
                G_rhs  = np.vstack([Ag.T @ (ge1 + R1), ge1.T @ (ge1 + R1)])
                gb_reg = _REG_EPS * np.eye(Hm.shape[1])
            degenerate = degenerate or (not gb_ok)

        share = (time.time() - t_sh) / max(1, len(unique_C) * len(arms))

        for C in unique_C:
            c1, c2 = C
            for arm in arms:
                t0 = time.time()
                if degenerate:
                    preds = np.zeros(len(X_test), dtype=int)
                elif method == 'Std':
                    z1 = safe_solve(FtF + (1 / c1) * EtE + reg, -rhs1)
                    z2 = safe_solve(EtE + (1 / c2) * FtF + reg,  rhs2)
                    preds = _twin_predict_from_z(z1, z2, Z_te)
                elif method == 'Rob':
                    v = _ROB_VARIANT_OF[arm]
                    if v == 'full':
                        L1, L2 = Fw2F + (1 / c1) * Ew1E, Ew1E + (1 / c2) * Fw2F
                    elif v == 'hua':
                        L1, L2 = FtF + (1 / c1) * Ew1E,  EtE + (1 / c2) * Fw2F
                    else:
                        L1, L2 = Fw2F + (1 / c1) * EtE,  Ew1E + (1 / c2) * FtF
                    z1 = safe_solve(L1 + reg, -rhs1)
                    z2 = safe_solve(L2 + reg,  rhs2)
                    preds = _twin_predict_from_z(z1, z2, Z_te)
                elif method == 'LSW':
                    z1 = safe_solve(FtF + (1 / c1) * Ed1E + reg, -rhs1)
                    z2 = safe_solve(EtE + (1 / c2) * Fd2F + reg,  rhs2)
                    preds = _twin_predict_from_z(z1, z2, Z_te)
                elif method == 'GB':
                    w1_raw = -safe_solve(GtG + (1 / c1) * HtH + gb_reg, H_rhs)
                    w2_raw =  safe_solve(HtH + (1 / c2) * GtG + gb_reg, G_rhs)
                    w1 = np.asarray(w1_raw[:-1]).ravel()
                    b1 = np.asarray(w1_raw[-1]).item()
                    w2 = np.asarray(w2_raw[:-1]).ravel()
                    b2 = np.asarray(w2_raw[-1]).item()
                    y1 = Z_te.dot(w1) + b1
                    y2 = Z_te.dot(w2) + b2
                    preds = np.where(np.abs(y1) <= np.abs(y2), +1, -1).ravel()
                acc_per[(arm, C)].append(accuracy_score(y_test, preds))
                time_per[(arm, C)] += (time.time() - t0) + share

    rows = []
    for params in param_list:
        _, C = _split_params(method, params)
        for arm in arms:
            rows.append({
                'dataset':       ds_name,
                'kernel':        kernel,
                'method':        arm,
                'params':        json.dumps(params),
                'mean_accuracy': np.mean(acc_per[(arm, C)]),
                'std_accuracy':  np.std(acc_per[(arm, C)]),
                'time_sec':      time_per[(arm, C)]
            })
    return rows

GROUPABLE_METHODS = ("Std", "LSW", "Rob", "GB")

def evaluate_all_methods(ds_name: str, kernel: str, folds, rob_arms=("Rob",),
                         n_jobs: int = None) -> list:

    n_jobs  = N_JOBS if n_jobs is None else n_jobs
    grids   = build_param_grids(kernel)
    results = []
    for method, grid in grids.items():
        try:
            if method in GROUPABLE_METHODS:
                groups, fallback = {}, []
                for p in grid:
                    if method == 'Rob' and p[0] != 'kde':
                        fallback.append(p)
                        continue
                    key, _ = _split_params(method, p)
                    groups.setdefault(key, []).append(p)
                tasks = [
                    delayed(evaluate_group)(method, key, plist, ds_name,
                                             kernel, folds, rob_arms)
                    for key, plist in groups.items()
                ]
                for rows in Parallel(n_jobs=n_jobs)(tasks):
                    results.extend(rows)
                if fallback:
                    tasks = (delayed(evaluate)(method, p, ds_name, kernel, folds)
                             for p in fallback)
                    results.extend(Parallel(n_jobs=n_jobs)(tasks))
            else:
                tasks = (delayed(evaluate)(method, p, ds_name, kernel, folds)
                         for p in grid)
                results.extend(Parallel(n_jobs=n_jobs, batch_size=8)(tasks))
        except Exception as e:
            print(f"  [SKIP-METHOD] {method} | dataset={ds_name} | kernel={kernel}: "
                  f"{type(e).__name__}: {e}")
    return results

def build_inner_folds(X_tr, y_tr, kernel: str, seed: int):
    y = np.asarray(y_tr).ravel()
    inner = []
    if INNER_SCHEME == "holdout":
        idx_tr, idx_val = train_test_split(
            np.arange(len(y)), test_size=INNER_VAL_FRACTION,
            stratify=y, random_state=seed
        )
        splits = [(idx_tr, idx_val)]
    elif INNER_SCHEME == "kfold":
        skf = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed)
        splits = list(skf.split(X_tr, y))
    else:
        raise ValueError(f"INNER_SCHEME ناشناخته: {INNER_SCHEME}")

    for idx_tr, idx_val in splits:
        Xi, yi = X_tr[idx_tr], y[idx_tr]
        Xv, yv = X_tr[idx_val], y[idx_val]
        if len(np.unique(yi)) < 2 or len(np.unique(yv)) < 2:
            continue
        cache = KernelCache(Xi, Xv) if kernel != 'linear' else None
        inner.append((Xi, yi, Xv, yv, cache))
    return inner

def _refit_and_score(method: str, params, kernel: str,
                     X_tr, y_tr, X_te, y_te, cache):
    cls, kw = _build_binary_cls_and_params(method, params, kernel)
    model = cls(**kw)
    K_tr = K_te = None
    if kernel == 'rbf' and cache is not None:
        K_tr, K_te = cache.get(model.gamma)
    t0 = time.time()
    model.fit(X_tr, y_tr, K_tr=K_tr)
    fit_time = time.time() - t0
    y_pred = model.predict(X_te, K_te=K_te)
    return accuracy_score(y_te, y_pred), fit_time

def run_nested_on_dataset(ds: Dataset, kernel: str, label_noise_ratio: float,
                          rob_arms=("Rob",), save_raw: bool = True,
                          n_jobs: int = None):
    outer = get_precomputed_folds(ds, kernel, n_splits=N_OUTER_FOLDS,
                                  label_noise_ratio=label_noise_ratio)
    methods = list(build_param_grids(kernel).keys())
    if 'Rob' in methods:
        methods = [m for m in methods if m != 'Rob'] + list(rob_arms)

    per_fold_acc  = {m: [] for m in methods}
    per_fold_time = {m: [] for m in methods}
    chosen        = {m: [] for m in methods}
    outer_rows, raw_rows = [], []

    for fold_idx, (X_tr, y_tr, X_te, y_te, cache) in enumerate(outer):
        inner = build_inner_folds(
            X_tr, y_tr, kernel,
            seed=derive_seed(ds.seed, "label", 500 + fold_idx)
        )
        if not inner:
            print(f"  [WARN] {ds.name} fold={fold_idx}: تقسیم درونی ممکن نشد؛ رد شد.")
            continue

        sel = pd.DataFrame(evaluate_all_methods(ds.name, kernel, inner, rob_arms,
                                                n_jobs=n_jobs))
        if sel.empty:
            continue
        if save_raw:
            s = sel.copy()
            s["outer_fold"] = fold_idx
            raw_rows.append(s)

        for m in methods:
            sub = sel[sel["method"] == m]
            if sub.empty:
                continue
            best = sub.loc[sub["mean_accuracy"].idxmax()]
            params = json.loads(best["params"])
            if isinstance(params, list):
                params = tuple(tuple(x) if isinstance(x, list) else x for x in params)
            try:
                acc, ftime = _refit_and_score(m, params, kernel,
                                              X_tr, y_tr, X_te, y_te, cache)
            except Exception as e:
                print(f"  [SKIP-REFIT] {m} | {ds.name} | fold={fold_idx}: "
                      f"{type(e).__name__}: {e}")
                continue
            per_fold_acc[m].append(acc)
            per_fold_time[m].append(ftime)
            chosen[m].append(best["params"])
            outer_rows.append({
                "dataset": ds.name, "kernel": kernel, "method": m,
                "outer_fold": fold_idx, "val_accuracy": float(best["mean_accuracy"]),
                "test_accuracy": acc, "fit_time_sec": ftime,
                "params": best["params"]
            })

    summary_rows, cfg_rows = [], []
    for m in methods:
        if not per_fold_acc[m]:
            continue
        summary_rows.append({
            "dataset": ds.name, "kernel": kernel, "method": m,
            "params": Counter(chosen[m]).most_common(1)[0][0],
            "mean_accuracy": float(np.mean(per_fold_acc[m])),
            "std_accuracy":  float(np.std(per_fold_acc[m])),
            "time_sec":      float(np.mean(per_fold_time[m])),
        })
        cfg_rows.append({
            "dataset": ds.name, "kernel": kernel, "method": m,
            "modal_params": Counter(chosen[m]).most_common(1)[0][0],
            "all_fold_params": " | ".join(chosen[m]),
            "n_distinct": len(set(chosen[m])),
        })

    raw_df = pd.concat(raw_rows, ignore_index=True) if raw_rows else pd.DataFrame()
    return summary_rows, outer_rows, cfg_rows, raw_df

def run_benchmark_on_dataset(ds: Dataset, kernel: str,
                             label_noise_ratio: float = None,
                             rob_arms=("Rob",), n_jobs: int = None):
    if label_noise_ratio is None:
        label_noise_ratio = CURRENT_LABEL_NOISE

    if SELECTION_MODE == "nested":
        return run_nested_on_dataset(ds, kernel, label_noise_ratio, rob_arms,
                                     n_jobs=n_jobs)

    folds = get_precomputed_folds(ds, kernel, n_splits=N_OUTER_FOLDS,
                                  label_noise_ratio=label_noise_ratio)
    rows = evaluate_all_methods(ds.name, kernel, folds, rob_arms, n_jobs=n_jobs)
    df = pd.DataFrame(rows)
    best = df.loc[df.groupby("method")["mean_accuracy"].idxmax()]
    return best.to_dict("records"), [], [], df

def single_model_timing(preloaded_map, selected_cfg: pd.DataFrame,
                        noise: float = 0.2, seed: int = 80,
                        repeats: int = None, methods=None):
    repeats = repeats or TIMING_REPEATS
    methods = methods or ALL_METHODS
    rows = []
    for name, ds in preloaded_map.items():
        ds.seed = seed
        ds.label_noise_ratio = noise
        for kernel in ("linear", globals().get("TB_NONLINEAR", "rbf")):
            folds = get_precomputed_folds(ds, kernel, n_splits=N_OUTER_FOLDS,
                                          label_noise_ratio=noise)
            X_tr, y_tr, X_te, y_te, cache = folds[0]
            K_tr = None
            for m in methods:
                sub = selected_cfg[(selected_cfg.dataset == name)
                                   & (selected_cfg.kernel == kernel)
                                   & (selected_cfg.method == m)]
                if sub.empty:
                    continue
                params = json.loads(sub.iloc[0]["modal_params"])
                if isinstance(params, list):
                    params = tuple(tuple(x) if isinstance(x, list) else x
                                   for x in params)
                cls, kw = _build_binary_cls_and_params(m, params, kernel)
                if kernel == 'rbf' and cache is not None:
                    K_tr, _ = cache.get(kw.get("gamma", 1.0))
                times = []
                for _ in range(repeats):
                    model = cls(**kw)
                    t0 = time.time()
                    try:
                        model.fit(X_tr, y_tr, K_tr=K_tr)
                    except Exception as e:
                        print(f"  [SKIP-TIME] {m} | {name} | {kernel}: {e}")
                        times = []
                        break
                    times.append(time.time() - t0)
                if times:
                    rows.append({"dataset": name, "kernel": kernel, "method": m,
                                 "n_train": len(y_tr),
                                 "fit_time_sec": float(np.median(times))})
    df = pd.DataFrame(rows)
    if not df.empty:
        df.to_csv(TIMING_CSV, index=False)
        print(f"[Timing] ذخیره شد: {TIMING_CSV}")
        print(df.groupby(["kernel", "method"])["fit_time_sec"]
                .agg(["median", "mean", "std"]).round(5).to_string())
    return df

def _build_units(preloaded_map, todo, kernel, noise, seed):
    units = {}
    for name in todo:
        ds = preloaded_map[name]
        ds.seed = seed
        try:
            outer = get_precomputed_folds(ds, kernel, n_splits=N_OUTER_FOLDS,
                                          label_noise_ratio=noise)
        except Exception as e:
            print(f"  [SKIP-FOLDS] {name}: {type(e).__name__}: {e}")
            continue
        folds = []
        for fold_idx, (X_tr, y_tr, X_te, y_te, cache) in enumerate(outer):
            inner = build_inner_folds(
                X_tr, y_tr, kernel,
                seed=derive_seed(ds.seed, "label", 500 + fold_idx))
            if not inner:
                print(f"  [WARN] {name} fold={fold_idx}: تقسیم درونی ممکن نشد.")
                continue
            folds.append({"idx": fold_idx, "outer": (X_tr, y_tr, X_te, y_te, cache),
                          "inner": inner})
        if folds:
            units[name] = folds
    return units

def _flat_selection_tasks(units, kernel, rob_arms):
    grids = build_param_grids(kernel)
    tasks = []
    for name, folds in units.items():
        for f in folds:
            for method, grid in grids.items():
                if method in GROUPABLE_METHODS:
                    groups, fallback = {}, []
                    for p in grid:
                        if method == 'Rob' and p[0] != 'kde':
                            fallback.append(p)
                            continue
                        key, _ = _split_params(method, p)
                        groups.setdefault(key, []).append(p)
                    for key, plist in groups.items():
                        tasks.append(("group", name, f["idx"], method, key,
                                      plist, f["inner"]))
                    for p in fallback:
                        tasks.append(("single", name, f["idx"], method, None,
                                      p, f["inner"]))
                else:
                    for p in grid:
                        tasks.append(("single", name, f["idx"], method, None,
                                      p, f["inner"]))
    return tasks

def _run_selection_task(kind, name, fold_idx, method, key, payload, inner,
                        kernel, rob_arms):
    try:
        if kind == "group":
            rows = evaluate_group(method, key, payload, name, kernel, inner,
                                  rob_arms)
        else:
            rows = [evaluate(method, payload, name, kernel, inner)]
    except Exception as e:
        return name, fold_idx, [], f"{type(e).__name__}: {e}"
    return name, fold_idx, rows, None

def _refit_task(method, params_json, kernel, X_tr, y_tr, X_te, y_te, cache,
                name, fold_idx, val_acc):
    params = json.loads(params_json)
    if isinstance(params, list):
        params = tuple(tuple(x) if isinstance(x, list) else x for x in params)
    try:
        acc, ftime = _refit_and_score(method, params, kernel,
                                      X_tr, y_tr, X_te, y_te, cache)
    except Exception as e:
        return name, fold_idx, method, None, None, params_json, val_acc, \
               f"{type(e).__name__}: {e}"
    return name, fold_idx, method, acc, ftime, params_json, val_acc, None

def run_batch_flat(preloaded_map, todo, kernel, noise, seed, rob_arms=("Rob",)):
    units = _build_units(preloaded_map, todo, kernel, noise, seed)
    if not units:
        return {}

    methods = list(build_param_grids(kernel).keys())
    if 'Rob' in methods:
        methods = [m for m in methods if m != 'Rob'] + list(rob_arms)

    tasks = _flat_selection_tasks(units, kernel, rob_arms)
    print(f"  [FLAT] فاز انتخاب: {len(tasks)} وظیفه روی {len(units)} دیتاست "
          f"در یک استخر واحد (n_jobs={N_JOBS})")
    out = Parallel(n_jobs=N_JOBS, batch_size='auto')(
        delayed(_run_selection_task)(k, n, fi, m, key, pl, inn, kernel, rob_arms)
        for (k, n, fi, m, key, pl, inn) in tasks)

    sel_rows = {}
    for name, fold_idx, rows, err in out:
        if err:
            print(f"  [SKIP-SEL] {name} fold={fold_idx}: {err}")
            continue
        sel_rows.setdefault((name, fold_idx), []).extend(rows)

    refit_specs = []
    for (name, fold_idx), rows in sel_rows.items():
        df = pd.DataFrame(rows)
        if df.empty:
            continue
        fold = next(f for f in units[name] if f["idx"] == fold_idx)
        X_tr, y_tr, X_te, y_te, cache = fold["outer"]
        for m in methods:
            sub = df[df["method"] == m]
            if sub.empty:
                continue
            best = sub.loc[sub["mean_accuracy"].idxmax()]
            refit_specs.append((m, best["params"], kernel, X_tr, y_tr, X_te,
                                y_te, cache, name, fold_idx,
                                float(best["mean_accuracy"])))

    print(f"  [FLAT] فاز برازش مجدد: {len(refit_specs)} وظیفه")
    refits = Parallel(n_jobs=N_JOBS, batch_size='auto')(
        delayed(_refit_task)(*spec) for spec in refit_specs)

    acc_map, time_map, chosen_map, outer_rows = {}, {}, {}, {}
    for name, fold_idx, m, acc, ftime, pjson, val_acc, err in refits:
        if err:
            print(f"  [SKIP-REFIT] {m} | {name} | fold={fold_idx}: {err}")
            continue
        acc_map.setdefault((name, m), []).append(acc)
        time_map.setdefault((name, m), []).append(ftime)
        chosen_map.setdefault((name, m), []).append(pjson)
        outer_rows.setdefault(name, []).append({
            "dataset": name, "kernel": kernel, "method": m,
            "outer_fold": fold_idx, "val_accuracy": val_acc,
            "test_accuracy": acc, "fit_time_sec": ftime, "params": pjson})

    results = {}
    for name in units:
        summary_rows, cfg_rows = [], []
        for m in methods:
            key = (name, m)
            if key not in acc_map or not acc_map[key]:
                continue
            summary_rows.append({
                "dataset": name, "kernel": kernel, "method": m,
                "params": Counter(chosen_map[key]).most_common(1)[0][0],
                "mean_accuracy": float(np.mean(acc_map[key])),
                "std_accuracy": float(np.std(acc_map[key])),
                "time_sec": float(np.mean(time_map[key])),
            })
            cfg_rows.append({
                "dataset": name, "kernel": kernel, "method": m,
                "modal_params": Counter(chosen_map[key]).most_common(1)[0][0],
                "all_fold_params": " | ".join(chosen_map[key]),
                "n_distinct": len(set(chosen_map[key])),
            })
        if summary_rows:
            results[name] = (summary_rows, outer_rows.get(name, []), cfg_rows)
    return results

In [ ]:
RESULTS_CSV = "benchmark_results.csv"

DATA_DIR = Path("C:/eshaghi")

LOCAL_BINARY_FILES = [
    ("Australian",      "Australian.txt"),

    ("Diabets_Local",   "Diabets.txt"),
    ("Sonar",           "Sonar.txt"),
    ("Heart_Local",     "Heart.txt"),
    ("spolice",         "spolice.txt"),

    ("Breastcancer",    "Breastcancer.txt"),
    ("Ionosphere",      "Ionosphere.txt"),
    ("liver-disorders", "liver-disorders.txt"),
]

def make_local_binary_loader(path: Path):
    def _loader():
        return load_libsvm(str(path))
    return _loader

DATASETS = []
_LOCAL_MISSING = []

if not DATA_DIR.exists():
    print(f"[Local] مسیر '{DATA_DIR}' وجود ندارد؛ هیچ فایل محلی اضافه نشد. "
          f"اجرا با منابع آنلاین ادامه می‌یابد.")
else:
    for _name, _fname in LOCAL_BINARY_FILES:
        _p = DATA_DIR / _fname
        if _p.exists():
            DATASETS.append({"name": _name, "data": make_local_binary_loader(_p),
                             "source": "local", "kind": "binary", "rule": "-"})
        else:
            _LOCAL_MISSING.append(_fname)

    print(f"[Local] {len(DATASETS)} فایل محلی پیدا شد در '{DATA_DIR}'.")
    if _LOCAL_MISSING:
        print(f"[Local] یافت نشد ({len(_LOCAL_MISSING)}): {_LOCAL_MISSING}")

def load_and_process_libsvm(path: str):
    return load_libsvm(path)

def load_dataset(cfg: dict) -> Dataset:
    X, y = cfg["data"]()
    return Dataset(
        cfg["name"], lambda X=X, y=y: (X, y),
        label_noise_ratio=CURRENT_LABEL_NOISE,
        seed=RANDOM_STATE
    )

def preload_all_binary_datasets(cfgs: list) -> dict:
    preloaded_map = {}
    report, failures = [], []
    for cfg in cfgs:
        name = cfg["name"]
        print(f"[Preload] {name}")
        try:
            ds = load_dataset(cfg)
            preloaded_map[name] = ds
            n_pos = int((ds.y == 1).sum())
            n_neg = int((ds.y == -1).sum())
            report.append({
                "dataset": name,
                "source": cfg.get("source", "-"),
                "kind": cfg.get("kind", "-"),
                "n": ds.X.shape[0], "d": ds.X.shape[1],
                "n_pos": n_pos, "n_neg": n_neg,
                "pos_ratio": round(n_pos / (n_pos + n_neg), 3),
                "rule": cfg.get("rule", "-"),
            })
        except Exception as e:
            print(f"  [SKIP] {name}: {type(e).__name__}: {e}")
            failures.append({"dataset": name, "source": cfg.get("source", "-"),
                             "error": f"{type(e).__name__}: {e}"})
            continue

    print(f"\n{'='*100}")
    print(f"[Preload] دیتاست‌های آماده: {len(preloaded_map)} از {len(cfgs)}")
    print(f"{'='*100}")
    if report:
        print(pd.DataFrame(report).to_string(index=False))
    if failures:
        print(f"\n[Preload] رد‌شده‌ها ({len(failures)}):")
        print(pd.DataFrame(failures).to_string(index=False))
    print()
    return preloaded_map

In [ ]:
import urllib.request

UCI_MIRRORS = [
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/",
]
CACHE_DIR      = Path("uci_cache")
CACHE_DIR.mkdir(exist_ok=True)
ONLINE_TIMEOUT = 60
ONLINE_RETRIES = 3

def _fetch_text(fname: str) -> str:
    cached = CACHE_DIR / fname
    if cached.exists():
        return cached.read_text(encoding="utf-8")
    last_err = None
    for base in UCI_MIRRORS:
        for attempt in range(1, ONLINE_RETRIES + 1):
            try:
                req = urllib.request.Request(base + fname,
                                             headers={"User-Agent": "python-urllib"})
                with urllib.request.urlopen(req, timeout=ONLINE_TIMEOUT) as r:
                    raw = r.read().decode("utf-8", errors="replace")
                cached.write_text(raw, encoding="utf-8")
                print(f"  [Cache] '{fname}' دانلود و در '{CACHE_DIR}' ذخیره شد.")
                return raw
            except Exception as e:
                last_err = e
                if attempt < ONLINE_RETRIES:
                    time.sleep(2 * attempt)
    raise RuntimeError(
        f"دریافت آنلاین '{fname}' ناموفق بود ({type(last_err).__name__}: {last_err}). "
        f"اگر شبکه در دسترس نیست، فایل را دستی در پوشه‌ی '{CACHE_DIR}' بگذارید."
    )

def _fetch_csv_rows(fname: str):
    return [ln.split(",") for ln in _fetch_text(fname).strip().splitlines() if ln.strip()]

def _split_numeric_last_label(fname: str):
    arr = np.array(_fetch_csv_rows(fname), dtype=float)
    return arr[:, :-1], arr[:, -1].astype(int)

def _split_numeric_last_string(fname: str):
    rows = _fetch_csv_rows(fname)
    X = np.array([r[:-1] for r in rows], dtype=float)
    cls = np.array([r[-1].strip() for r in rows])
    return X, cls

def _raw_haberman():    return _split_numeric_last_label("haberman.csv")
def _raw_banknote():    return _split_numeric_last_label("banknote_authentication.csv")
def _raw_sonar():       return _split_numeric_last_string("sonar.csv")
def _raw_iris():        return _split_numeric_last_string("iris.csv")

BINARIZE_RULES = {

    "Haberman":       ("vs_rest", 1),
    "Banknote":       ("vs_rest", 1),
    "Sonar_UCI":      ("vs_rest", "M"),

    "Iris_bin":       ("pair", "Iris-versicolor", "Iris-virginica"),
}

def make_online_binary_loader(name: str, raw_fn):
    def _loader():
        X, cls = raw_fn()
        Xb, yb, desc = binarize_and_finalize(X, cls, BINARIZE_RULES[name])
        print(f"  [Binarize] {name}: {desc}")
        return Xb, yb
    return _loader

_ONLINE_RAW = [

    ("Haberman",  _raw_haberman, "binary"),
    ("Sonar_UCI", _raw_sonar,    "binary"),
    ("Banknote",  _raw_banknote, "binary"),
    ("Iris_bin",  _raw_iris,     "multiclass->binary"),
]

UCI_ONLINE_DATASETS = [
    {"name": _n, "data": make_online_binary_loader(_n, _f),
     "source": "online", "kind": _k, "rule": str(BINARIZE_RULES[_n])}
    for _n, _f, _k in _ONLINE_RAW
]

APPEND_UCI_ONLINE = True

if APPEND_UCI_ONLINE:
    _existing = {c["name"] for c in DATASETS}
    _added = 0
    for _cfg in UCI_ONLINE_DATASETS:
        if _cfg["name"] not in _existing:
            DATASETS.append(_cfg)
            _added += 1
    print(f"[UCI-ONLINE] {_added} دیتاست آنلاین دوکلاسه به DATASETS اضافه شد.")

print(f"\n[Registry] مجموع دیتاست‌های دوکلاسه: {len(DATASETS)}")

def print_run_scale_estimate(n_datasets: int = None):
    n_ds = len(DATASETS) if n_datasets is None else n_datasets
    n_units = n_ds * len(RANDOM_STATES) * len(LABEL_NOISE_RATIOS) * 2
    print(f"\n{'='*70}")
    print("[Scale] برآورد حجم اجرای مسیر دوکلاسه")
    print(f"  دیتاست‌ها      : {n_ds}")
    print(f"  سیدها          : {len(RANDOM_STATES)}  {RANDOM_STATES}")
    print(f"  سطوح نویز      : {len(LABEL_NOISE_RATIOS)}  {LABEL_NOISE_RATIOS}")
    print(f"  کرنل‌ها         : 2  ['linear', 'rbf']")
    print(f"  واحد کار (dataset x seed x noise x kernel) = {n_units}")
    print("  هر واحد کار = کل گرید پنج روش روی ۳ فولد.")
    print("  توجه: هزینه‌ی فاز RBF با n^3 رشد می‌کند. برای دیتاست‌های")
    print("  بزرگ‌تر از حدود ۲۰۰۰ نمونه (مثل Segment) فاز RBF می‌تواند")
    print("  ساعت‌ها طول بکشد. پیش از اجرای کامل، SMOKE_TEST را True کنید.")
    print(f"{'='*70}\n")

In [ ]:
def print_best_results(df: pd.DataFrame, ds_name: str, kernel: str):
    sub = df[(df["dataset"] == ds_name) & (df["kernel"] == kernel)]
    if sub.empty:
        return
    best = (sub.sort_values("mean_accuracy", ascending=False)
               .groupby("method", as_index=False)
               .head(1))
    print(f"\n[Best per Method] dataset={ds_name} | kernel={kernel}")
    print(best[["method", "mean_accuracy", "std_accuracy", "time_sec", "params"]].to_string(index=False))

class ResultsCollector:
    def __init__(self):
        self.rows = []

    def add_seed_results(self, seed: int, noise: float, df_seed: pd.DataFrame):
        if df_seed.empty:
            return
        idx = df_seed.groupby(["dataset", "kernel", "method"])["mean_accuracy"].idxmax()
        best = df_seed.loc[idx]
        for _, r in best.iterrows():
            self.rows.append({
                "seed": seed, "noise": noise,
                "dataset": r["dataset"], "kernel": r["kernel"],
                "method": r["method"],
                "mean_accuracy": r["mean_accuracy"],
                "std_accuracy": r.get("std_accuracy", np.nan),
                "time_sec": r.get("time_sec", np.nan),
            })

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)

    def paper_style_table(self, kernel: str, noise: float,
                           methods=ALL_METHODS) -> Optional[pd.DataFrame]:
        df = self.to_dataframe()
        if df.empty:
            return None
        sub = df[(df["kernel"] == kernel) & (df["noise"] == noise)]
        if sub.empty:
            return None
        piv = sub.groupby(["dataset", "method"])["mean_accuracy"].mean().unstack("method")
        std = sub.groupby(["dataset", "method"])["mean_accuracy"].std().unstack("method")
        piv = piv.reindex(columns=[m for m in methods if m in piv.columns])
        std = std.reindex(columns=piv.columns)
        if piv.empty:
            return None

        ranks = piv.rank(axis=1, ascending=False, method="average")

        display = pd.DataFrame(index=[disp(i) for i in piv.index])
        for m in piv.columns:
            display[METHOD_LABELS.get(m, m)] = [
                f"{acc*100:5.2f}\u00b1{(0.0 if np.isnan(sd) else sd)*100:.2f} ({rk:.2f})"
                for acc, sd, rk in zip(piv[m], std[m], ranks[m])
            ]

        final_rank = ranks.mean(axis=0)
        acc_mean   = piv.mean(axis=0) * 100
        display.loc["Final Rank"]    = [f"{final_rank[m]:.2f}" for m in piv.columns]
        display.loc["Accuracy Mean"] = [f"{acc_mean[m]:.2f}" for m in piv.columns]

        print(f"\n{'='*98}\nجدول سبک-مقاله | kernel={kernel} | label_noise={noise}\n{'='*98}")
        print(display.to_string())
        return display

    def boxplots(self, output_dir: Path, methods=ALL_METHODS):
        df = self.to_dataframe()
        if df.empty:
            print("[Boxplot] هیچ داده‌ای برای رسم وجود ندارد.")
            return
        output_dir.mkdir(exist_ok=True, parents=True)
        colors = {'Std': '#2196F3', 'LSW': '#FF9800', 'APL': '#9C27B0', 'GBF': '#9C27B0', 'CQS': '#795548',
                  'Rob': '#4CAF50', 'GB': '#E91E63'}
        for (ds_name, kernel, noise), sub in df.groupby(["dataset", "kernel", "noise"]):
            present = [m for m in methods if m in sub["method"].unique()]
            if len(present) < 2:
                continue
            data = [sub.loc[sub["method"] == m, "mean_accuracy"].values for m in present]
            fig, ax = plt.subplots(figsize=(1.6 * len(present) + 2, 4.5))
            bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color='black', linewidth=2))
            for patch, m in zip(bp['boxes'], present):
                patch.set_facecolor(colors.get(m, '#888888'))
                patch.set_alpha(0.75)
            ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in present], rotation=20, ha='right')
            ax.set_ylabel('Accuracy')
            ax.set_title(f'{ds_name} | {kernel} kernel | noise={int(noise*100)}%\n'
                          f'(distribution across {sub["seed"].nunique()} random seeds)',
                          fontsize=10)
            ax.grid(axis='y', linestyle='--', alpha=0.5)
            plt.tight_layout()
            fname = output_dir / f"boxplot_{ds_name}_{kernel}_noise{int(noise*100)}.pdf"
            plt.savefig(fname, bbox_inches='tight')
            plt.close(fig)
            print(f"[Boxplot] ذخیره شد: {fname}")

    def wilcoxon_report_by_dataset(self, reference: str = "Rob",
                                   alpha: float = 0.05, methods=None):
        methods = methods or ALL_METHODS
        df = self.to_dataframe()
        if df.empty:
            print("[Wilcoxon-ds] داده‌ای موجود نیست.")
            return pd.DataFrame()
        out = []
        print(f"\n{'='*78}\n[تحلیل اصلی] Wilcoxon signed-rank، واحد = دیتاست "
              f"(مرجع = {METHOD_LABELS.get(reference, reference)})\n{'='*78}")
        for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
            piv = (sub[sub["method"].isin(methods)]
                   .groupby(["dataset", "method"])["mean_accuracy"]
                   .mean().unstack("method").dropna())
            n_ds = piv.shape[0]
            print(f"\n-- kernel={kernel} | noise={noise} | N={n_ds} دیتاست --")
            if reference not in piv.columns or n_ds < 5:
                print("  (تعداد دیتاست برای آزمون کافی نیست)")
                continue
            for comp in piv.columns:
                if comp == reference:
                    continue
                diff = piv[reference].values - piv[comp].values
                if np.all(diff == 0):
                    print(f"  vs {METHOD_LABELS.get(comp, comp):32s}: تمام تفاضل‌ها صفر")
                    continue
                stat, p = wilcoxon(diff, alternative='greater')
                sig = '*' if p < alpha else 'n.s.'
                wins = int(np.sum(diff > 0))
                print(f"  vs {METHOD_LABELS.get(comp, comp):32s}: W={stat:.1f}  "
                      f"p={p:.4f}  {sig}   (برد/باخت = {wins}/{n_ds - wins})")
                out.append({"kernel": kernel, "noise": noise, "reference": reference,
                            "comparison": comp, "n_datasets": n_ds, "W": stat,
                            "p_value": p, "significant": p < alpha,
                            "wins": wins, "losses": n_ds - wins})
        res = pd.DataFrame(out)
        if not res.empty:
            res.to_csv(RESULTS_DIR / "wilcoxon_by_dataset.csv", index=False)
        return res

    def wilcoxon_report(self, reference: str = "Rob", alpha: float = 0.05):
        print("\n[توجه] تحلیل زیر واحد نمونه را جفت (dataset, seed) می‌گیرد و "
              "anti-conservative است؛ تحلیل اصلی مقاله wilcoxon_report_by_dataset "
              "است.")
        df = self.to_dataframe()
        if df.empty:
            print("[Wilcoxon] داده‌ای موجود نیست.")
            return
        print(f"\n{'='*78}\nآزمون Wilcoxon signed-rank (مرجع = {METHOD_LABELS.get(reference, reference)})\n{'='*78}")
        for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
            print(f"\n-- kernel={kernel} | noise={noise} --")
            piv = sub.groupby(["dataset", "seed", "method"])["mean_accuracy"].mean().unstack("method")
            piv = piv.dropna()
            if reference not in piv.columns or piv.empty:
                print("  (داده کافی نیست)")
                continue
            ref_vals = piv[reference].values
            for comp in piv.columns:
                if comp == reference:
                    continue
                diff = ref_vals - piv[comp].values
                if np.all(diff == 0):
                    print(f"  vs {METHOD_LABELS.get(comp, comp):15s}: تمام تفاضل‌ها صفر - رد شد")
                    continue
                try:
                    stat, p_val = wilcoxon(diff, alternative='greater')
                    sig = '***' if p_val < alpha else 'n.s.'
                    print(f"  vs {METHOD_LABELS.get(comp, comp):15s}: W={stat:.1f}  p={p_val:.4f}  {sig}")
                except ValueError as e:
                    print(f"  vs {METHOD_LABELS.get(comp, comp):15s}: تست ممکن نشد ({e})")

    def friedman_report(self, methods=ALL_METHODS):
        from scipy.stats import friedmanchisquare
        df = self.to_dataframe()
        if df.empty:
            print("[Friedman] داده‌ای موجود نیست.")
            return
        print(f"\n{'='*78}\nآزمون Friedman بین همه‌ی روش‌ها\n{'='*78}")
        for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
            piv = (sub.groupby(["dataset", "seed", "method"])["mean_accuracy"]
                      .mean().unstack("method"))
            piv = piv.reindex(columns=[m for m in methods if m in piv.columns]).dropna()
            if piv.shape[1] < 3 or piv.shape[0] < 3:
                print(f"-- kernel={kernel} | noise={noise}: داده کافی نیست")
                continue
            stat, p = friedmanchisquare(*[piv[m].values for m in piv.columns])
            print(f"-- kernel={kernel} | noise={noise}: chi2={stat:.3f}, p={p:.3e}")

    def noise_curve_plot(self, output_dir: Path, methods=ALL_METHODS,
                          fname: str = "accuracy_vs_noise.pdf"):
        df = self.to_dataframe()
        if df.empty or df["noise"].nunique() < 2:
            print("[NoiseCurve] حداقل دو سطح نویز لازم است؛ نمودار رسم نشد.")
            return
        output_dir.mkdir(exist_ok=True, parents=True)
        styles = {"Std": ("o", "-"), "LSW": ("s", "--"), "APL": ("^", "-."), "GBF": ("^", "-."), "CQS": ("v", ":"),
                  "GB": ("d", ":"), "Rob": ("*", "-")}
        for kernel, sub_k in df.groupby("kernel"):
            fig, ax = plt.subplots(figsize=(6.4, 4.2))
            for m in methods:
                sub = sub_k[sub_k["method"] == m]
                if sub.empty:
                    continue
                curve = sub.groupby("noise")["mean_accuracy"].mean().sort_index()
                mk, ls = styles.get(m, ("o", "-"))
                lw = 2.4 if m == "Rob" else 1.4
                ax.plot(curve.index * 100, curve.values * 100, marker=mk,
                        linestyle=ls, linewidth=lw,
                        markersize=9 if m == "Rob" else 6,
                        label=METHOD_LABELS.get(m, m))
            ax.set_xlabel("Label-noise level (%)")
            ax.set_ylabel("Mean accuracy over datasets (%)")
            ax.grid(linestyle="--", alpha=0.5)
            ax.legend()
            plt.tight_layout()
            out = output_dir / f"{Path(fname).stem}_{kernel}{Path(fname).suffix}"
            plt.savefig(out, bbox_inches="tight")
            plt.close(fig)
            print(f"[NoiseCurve] ذخیره شد: {out}")

    def to_excel(self, path: Path):
        df = self.to_dataframe()
        if df.empty:
            print("[Excel] داده‌ای برای ذخیره وجود ندارد.")
            return
        try:
            with pd.ExcelWriter(path, engine="openpyxl") as writer:
                df.to_excel(writer, sheet_name="raw_seed_results", index=False)
                summary = (df.groupby(["dataset", "kernel", "noise", "method"])
                             .agg(acc_mean=("mean_accuracy", "mean"),
                                  acc_std_over_seeds=("mean_accuracy", "std"),
                                  cv_fold_std_mean=("std_accuracy", "mean"),
                                  n_seeds=("mean_accuracy", "count"),
                                  train_time_mean=("time_sec", "mean"))
                             .reset_index())
                summary.to_excel(writer, sheet_name="summary", index=False)
            print(f"[Excel] ذخیره شد: {path}")
        except Exception as e:
            print(f"[Excel] خطا در ذخیره: {type(e).__name__}: {e}")

def _nemenyi_q_alpha(k: int, alpha: float = 0.05) -> float:
    try:
        from scipy.stats import studentized_range
        return float(studentized_range.ppf(1.0 - alpha, k, np.inf) / np.sqrt(2.0))
    except Exception:
        table = {
            0.05: {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850,
                   7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164},
            0.10: {2: 1.645, 3: 2.052, 4: 2.291, 5: 2.459, 6: 2.589,
                   7: 2.693, 8: 2.780, 9: 2.855, 10: 2.920},
        }
        return table.get(alpha, table[0.05])[min(max(k, 2), 10)]

def _nemenyi_cliques(sorted_vals: np.ndarray, cd: float):
    k = len(sorted_vals)
    spans = []
    for i in range(k):
        j = k - 1
        while j > i and (sorted_vals[j] - sorted_vals[i]) > cd:
            j -= 1
        if j > i:
            spans.append((i, j))
    return [s for s in spans
            if not any(s != t and t[0] <= s[0] and s[1] <= t[1] for t in spans)]

def _cd_diagram(avg_ranks: pd.Series, n_datasets: int, alpha: float,
                title: str, out_path: Path):
    k = len(avg_ranks)
    cd = _nemenyi_q_alpha(k, alpha) * np.sqrt(k * (k + 1) / (6.0 * n_datasets))
    avg = avg_ranks.sort_values()
    names = [METHOD_LABELS.get(m, m) for m in avg.index]
    vals  = avg.values

    lo, hi = 1.0, float(k)
    n_left = int(np.ceil(k / 2))
    step, top = 0.30, 0.0
    height = 1.6 + step * max(n_left, k - n_left) + 0.22 * max(1, len(_nemenyi_cliques(vals, cd)))
    fig, ax = plt.subplots(figsize=(7.6, height))

    ax.plot([lo, hi], [top, top], color="black", lw=1.4)
    for t in np.arange(lo, hi + 1e-9, 0.5):
        big = abs(t - round(t)) < 1e-9
        ax.plot([t, t], [top, top + (0.09 if big else 0.05)], color="black", lw=1.1)
        if big:
            ax.text(t, top + 0.14, f"{int(round(t))}", ha="center", va="bottom", fontsize=9)

    for idx in range(k):
        y = top - step * ((idx + 1) if idx < n_left else (k - idx))
        x_end = lo - 0.22 if idx < n_left else hi + 0.22
        ax.plot([vals[idx], vals[idx], x_end], [top, y, y], color="0.25", lw=1.2)
        ax.text(x_end + (-0.05 if idx < n_left else 0.05), y,
                f"{names[idx]} ({vals[idx]:.2f})",
                ha="right" if idx < n_left else "left", va="center", fontsize=9)

    for c_i, (i, j) in enumerate(_nemenyi_cliques(vals, cd)):
        y = top - 0.13 - 0.10 * c_i
        ax.plot([vals[i] - 0.03, vals[j] + 0.03], [y, y], color="black", lw=4.0,
                solid_capstyle="butt")

    y_cd = top + 0.34
    ax.plot([lo, lo + cd], [y_cd, y_cd], color="crimson", lw=2.0)
    ax.plot([lo, lo], [y_cd - 0.04, y_cd + 0.04], color="crimson", lw=2.0)
    ax.plot([lo + cd, lo + cd], [y_cd - 0.04, y_cd + 0.04], color="crimson", lw=2.0)
    ax.text(lo + cd / 2, y_cd + 0.06, f"CD = {cd:.2f}", ha="center", va="bottom",
            fontsize=9, color="crimson")

    ax.text(lo - 0.22, top + 0.30, "better", ha="right", va="center", fontsize=9,
            style="italic", color="0.35")
    ax.set_xlim(lo - 1.9, hi + 1.9)
    ax.set_ylim(top - step * max(n_left, k - n_left) - 0.55, y_cd + 0.34)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return cd

def _collector_critical_difference(self, output_dir: Path, methods=ALL_METHODS,
                                   alpha: float = 0.05):
    df = self.to_dataframe()
    if df.empty:
        print("[CD] داده‌ای موجود نیست.")
        return
    output_dir.mkdir(exist_ok=True, parents=True)
    for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
        piv = sub.groupby(["dataset", "method"])["mean_accuracy"].mean().unstack("method")
        piv = piv.reindex(columns=[m for m in methods if m in piv.columns]).dropna()
        n_ds, k = piv.shape
        if n_ds < 3 or k < 3:
            print(f"[CD] kernel={kernel} | noise={noise}: داده کافی نیست "
                  f"(دیتاست={n_ds}, روش={k}؛ حداقل ۳ و ۳ لازم است).")
            continue
        ranks = piv.rank(axis=1, ascending=False, method="average")
        avg   = ranks.mean(axis=0)
        out   = output_dir / f"critical_difference_{kernel}_noise{int(noise*100)}.pdf"
        cd = _cd_diagram(avg, n_ds, alpha,
                         f"Nemenyi CD diagram | {kernel} kernel | "
                         f"label noise = {int(noise*100)}%  (N = {n_ds} datasets)", out)
        print(f"[CD] kernel={kernel} | noise={noise} | N={n_ds} | CD={cd:.3f} -> {out}")
        if n_ds < 10:
            print("     [هشدار] با تعداد کم دیتاست، توان آزمون Nemenyi پایین است "
                  "و نبودِ اختلاف معنادار را نباید هم‌ارزی روش‌ها تفسیر کرد.")

def _collector_noise_curve_per_dataset(self, output_dir: Path, methods=ALL_METHODS):
    df = self.to_dataframe()
    if df.empty or df["noise"].nunique() < 2:
        print("[NoiseCurveByDS] حداقل دو سطح نویز لازم است؛ رسم نشد.")
        return
    output_dir.mkdir(exist_ok=True, parents=True)
    styles = {"Std": ("o", "-"), "LSW": ("s", "--"), "APL": ("^", "-."), "GBF": ("^", "-."), "CQS": ("v", ":"),
              "GB": ("d", ":"), "Rob": ("*", "-")}
    for kernel, sub_k in df.groupby("kernel"):
        ds_names = sorted(sub_k["dataset"].unique())
        ncol = min(3, len(ds_names))
        nrow = int(np.ceil(len(ds_names) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.5 * nrow),
                                 squeeze=False)
        for ax_i, ds_name in enumerate(ds_names):
            ax  = axes[ax_i // ncol][ax_i % ncol]
            sub = sub_k[sub_k["dataset"] == ds_name]
            for m in methods:
                s = sub[sub["method"] == m]
                if s.empty:
                    continue
                curve = s.groupby("noise")["mean_accuracy"].mean().sort_index()
                mk, ls = styles.get(m, ("o", "-"))
                ax.plot(curve.index * 100, curve.values * 100, marker=mk, linestyle=ls,
                        linewidth=2.4 if m == "Rob" else 1.4,
                        markersize=9 if m == "Rob" else 6,
                        label=METHOD_LABELS.get(m, m))
            ax.set_title(ds_name, fontsize=10)
            ax.grid(linestyle="--", alpha=0.5)
            ax.set_xlabel("Label-noise level (%)")
            ax.set_ylabel("Accuracy (%)")
        for ax_i in range(len(ds_names), nrow * ncol):
            axes[ax_i // ncol][ax_i % ncol].axis("off")
        handles, labels = axes[0][0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=min(5, len(labels)),
                   frameon=False, bbox_to_anchor=(0.5, -0.02))
        fig.suptitle(f"Accuracy vs label noise per dataset | {kernel} kernel", fontsize=11)
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        out = output_dir / f"accuracy_vs_noise_by_dataset_{kernel}.pdf"
        plt.savefig(out, bbox_inches="tight")
        plt.close(fig)
        print(f"[NoiseCurveByDS] ذخیره شد: {out}")

def _collector_training_time_plot(self, output_dir: Path, methods=ALL_METHODS):
    df = self.to_dataframe()
    if df.empty or "time_sec" not in df.columns or df["time_sec"].isna().all():
        print("[TimePlot] داده‌ی زمانی موجود نیست.")
        return
    output_dir.mkdir(exist_ok=True, parents=True)
    present = [m for m in methods if m in df["method"].unique()]
    kernels = sorted(df["kernel"].unique())
    colors  = {'Std': '#2196F3', 'LSW': '#FF9800', 'APL': '#9C27B0', 'GBF': '#9C27B0', 'CQS': '#795548',
               'Rob': '#4CAF50', 'GB': '#E91E63'}
    x = np.arange(len(present))
    width = 0.8 / max(1, len(kernels))
    fig, ax = plt.subplots(figsize=(1.5 * len(present) + 3, 4.4))
    for k_i, kernel in enumerate(kernels):
        sub  = df[df["kernel"] == kernel]
        vals = [sub.loc[sub["method"] == m, "time_sec"].mean() for m in present]
        errs = [sub.loc[sub["method"] == m, "time_sec"].std() for m in present]
        ax.bar(x + k_i * width - 0.4 + width / 2, vals, width,
               yerr=np.nan_to_num(errs), capsize=3, label=f"{kernel} kernel",
               color=[colors.get(m, "#888888") for m in present],
               alpha=0.95 if kernel == "linear" else 0.55,
               edgecolor="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in present], rotation=20, ha="right")
    ax.set_yscale("log")
    ax.set_ylabel("Amortized training time per model (s, log scale)")
    ax.set_title("Computational cost of the best configuration\n"
                 "(grid-amortized; see note in code)", fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.legend(frameon=False)
    plt.tight_layout()
    out = output_dir / "training_time.pdf"
    plt.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"[TimePlot] ذخیره شد: {out}")
    print(df.groupby(["kernel", "method"])["time_sec"]
            .agg(["mean", "std"]).round(4).to_string())

def _collector_add_full_grid(self, seed: int, noise: float, df_seed: pd.DataFrame,
                             method: str = "Rob"):
    if not hasattr(self, "grid_rows"):
        self.grid_rows = []
    if df_seed is None or df_seed.empty:
        return
    sub = df_seed[df_seed["method"] == method]
    for _, r in sub.iterrows():
        try:
            p = json.loads(r["params"])
            bw = p[3]
        except Exception:
            continue
        self.grid_rows.append({
            "seed": seed, "noise": noise, "dataset": r["dataset"],
            "kernel": r["kernel"], "bandwidth": float(bw),
            "mean_accuracy": float(r["mean_accuracy"]),
        })

def _collector_bandwidth_sensitivity_plot(self, output_dir: Path):
    rows = getattr(self, "grid_rows", [])
    if not rows:
        print("[BandwidthPlot] گریدی ثبت نشده است؛ add_full_grid فراخوانی نشده.")
        return
    df = pd.DataFrame(rows)
    output_dir.mkdir(exist_ok=True, parents=True)
    for kernel, sub_k in df.groupby("kernel"):
        best = (sub_k.groupby(["dataset", "seed", "noise", "bandwidth"])["mean_accuracy"]
                     .max().reset_index())
        fig, ax = plt.subplots(figsize=(6.6, 4.3))
        for noise, s in best.groupby("noise"):
            curve = s.groupby("bandwidth")["mean_accuracy"].mean().sort_index()
            ax.plot(curve.index, curve.values * 100, marker="o", linewidth=1.8,
                    label=f"label noise = {int(noise*100)}%")
        ax.set_xlabel("KDE bandwidth")
        ax.set_ylabel("Mean accuracy over datasets and seeds (%)")
        ax.set_title(f"KDEWLSTSVM sensitivity to the KDE bandwidth | {kernel} kernel",
                     fontsize=10)
        ax.grid(linestyle="--", alpha=0.5)
        ax.legend(frameon=False)
        plt.tight_layout()
        out = output_dir / f"bandwidth_sensitivity_{kernel}.pdf"
        plt.savefig(out, bbox_inches="tight")
        plt.close(fig)
        print(f"[BandwidthPlot] ذخیره شد: {out}")

ResultsCollector.critical_difference_plot   = _collector_critical_difference
ResultsCollector.noise_curve_per_dataset    = _collector_noise_curve_per_dataset
ResultsCollector.training_time_plot         = _collector_training_time_plot
ResultsCollector.add_full_grid              = _collector_add_full_grid
ResultsCollector.bandwidth_sensitivity_plot = _collector_bandwidth_sensitivity_plot

def _collector_ablation_table(self, methods=("Std", "RobHua", "RobSelf", "Rob")):
    df = self.to_dataframe()
    if df.empty:
        print("[Ablation] داده‌ای موجود نیست.")
        return pd.DataFrame()
    out = []
    for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
        piv = (sub[sub["method"].isin(methods)]
               .groupby(["dataset", "method"])["mean_accuracy"]
               .mean().unstack("method"))
        piv = piv.reindex(columns=[m for m in methods if m in piv.columns]).dropna()
        if piv.empty:
            continue
        ranks = piv.rank(axis=1, ascending=False, method="average")
        for m in piv.columns:
            out.append({"kernel": kernel, "noise": noise, "method": m,
                        "mean_accuracy": piv[m].mean() * 100,
                        "avg_rank": ranks[m].mean(),
                        "n_datasets": piv.shape[0]})
    res = pd.DataFrame(out)
    if res.empty:
        return res
    print(f"\n{'='*78}\n[ابلیشن] اثر جای‌گذاری وزن‌ها\n{'='*78}")
    for kernel in sorted(res.kernel.unique()):
        piv_acc = (res[res.kernel == kernel]
                   .pivot(index="noise", columns="method", values="mean_accuracy"))
        piv_rk = (res[res.kernel == kernel]
                  .pivot(index="noise", columns="method", values="avg_rank"))
        cols = [m for m in methods if m in piv_acc.columns]
        print(f"\n--- kernel={kernel} | میانگین دقت (٪) ---")
        print(piv_acc[cols].rename(columns=METHOD_LABELS).round(2).to_string())
        print(f"--- kernel={kernel} | میانگین رتبه (کمتر بهتر) ---")
        print(piv_rk[cols].rename(columns=METHOD_LABELS).round(2).to_string())
    res.to_csv(RESULTS_DIR / "ablation_weight_placement.csv", index=False)

    for kernel in sorted(res.kernel.unique()):
        sub = res[res.kernel == kernel]
        fig, ax = plt.subplots(figsize=(6.4, 4.2))
        styles = {"Std": ("o", "-"), "RobHua": ("s", "--"),
                  "RobSelf": ("^", "-."), "Rob": ("*", "-")}
        for m in methods:
            s = sub[sub.method == m].sort_values("noise")
            if s.empty:
                continue
            mk, ls = styles.get(m, ("o", "-"))
            ax.plot(s["noise"] * 100, s["mean_accuracy"], marker=mk, linestyle=ls,
                    linewidth=2.4 if m == "Rob" else 1.4,
                    markersize=9 if m == "Rob" else 6,
                    label=METHOD_LABELS.get(m, m))
        ax.set_xlabel("Label-noise level (%)")
        ax.set_ylabel("Mean accuracy over datasets (%)")
        ax.set_title(f"Ablation: where the weights are applied | {kernel} kernel",
                     fontsize=10)
        ax.grid(linestyle="--", alpha=0.5)
        ax.legend(fontsize=8)
        plt.tight_layout()
        out_p = RESULTS_DIR / f"ablation_weight_placement_{kernel}.pdf"
        plt.savefig(out_p, bbox_inches="tight")
        plt.close(fig)
        print(f"[Ablation] شکل ذخیره شد: {out_p}")
    return res

ResultsCollector.ablation_table = _collector_ablation_table

def plot_single_model_timing(df_time: pd.DataFrame, output_dir: Path,
                             methods=None):
    if df_time is None or df_time.empty:
        print("[TimingPlot] داده‌ای موجود نیست.")
        return
    methods = methods or ALL_METHODS
    output_dir.mkdir(exist_ok=True, parents=True)
    present = [m for m in methods if m in df_time["method"].unique()]
    kernels = sorted(df_time["kernel"].unique())
    colors  = {'Std': '#2196F3', 'LSW': '#FF9800', 'APL': '#9C27B0', 'GBF': '#9C27B0', 'CQS': '#795548',
               'Rob': '#4CAF50', 'GB': '#E91E63'}
    x = np.arange(len(present))
    width = 0.8 / max(1, len(kernels))
    fig, ax = plt.subplots(figsize=(1.5 * len(present) + 3, 4.4))
    for k_i, kernel in enumerate(kernels):
        sub = df_time[df_time["kernel"] == kernel]
        vals = [sub.loc[sub["method"] == m, "fit_time_sec"].median() for m in present]
        lo   = [sub.loc[sub["method"] == m, "fit_time_sec"].quantile(0.25) for m in present]
        hi   = [sub.loc[sub["method"] == m, "fit_time_sec"].quantile(0.75) for m in present]
        err  = np.vstack([np.array(vals) - np.array(lo), np.array(hi) - np.array(vals)])
        ax.bar(x + k_i * width - 0.4 + width / 2, vals, width,
               yerr=np.nan_to_num(err), capsize=3, label=f"{kernel} kernel",
               color=[colors.get(m, "#888888") for m in present],
               alpha=0.95 if kernel == "linear" else 0.55,
               edgecolor="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in present], rotation=20, ha="right")
    ax.set_yscale("log")
    ax.set_ylabel("Training time of a single fit (s, log scale)")
    ax.set_title("Cost of one model fit with the selected configuration\n"
                 "(median over datasets; whiskers = interquartile range)", fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.legend(frameon=False)
    plt.tight_layout()
    out = output_dir / "training_time.pdf"
    plt.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"[TimingPlot] ذخیره شد: {out}")

def rebuild_collector_from_csv(path=None) -> ResultsCollector:
    path = Path(path or NESTED_OUTER_CSV)
    if not path.exists():
        raise FileNotFoundError(f"فایل یافت نشد: {path}")
    raw = pd.read_csv(path)
    agg = (raw.groupby(["seed", "noise", "dataset", "kernel", "method"], as_index=False)
              .agg(mean_accuracy=("test_accuracy", "mean"),
                   std_accuracy=("test_accuracy", "std"),
                   time_sec=("fit_time_sec", "mean")))
    agg["std_accuracy"] = agg["std_accuracy"].fillna(0.0)
    col = ResultsCollector()
    col.rows = agg.to_dict("records")
    print(f"[Rebuild] {len(col.rows)} سطر از {path} بازسازی شد.")
    return col

def regenerate_all_reports(path=None, output_dir: Path = None):
    output_dir = output_dir or OUTPUT_DIR

    if path is None and Path(PROGRESS_CSV).exists() \
            and Path(PROGRESS_CSV).stat().st_size > 0:
        col = rebuild_collector_from_progress(PROGRESS_CSV)
    else:
        col = rebuild_collector_from_csv(path)
    for noise in sorted(col.to_dataframe()["noise"].unique()):
        for kernel in ("linear", globals().get("TB_NONLINEAR", "rbf")):
            col.paper_style_table(kernel, noise)
    col.wilcoxon_report_by_dataset(reference="Rob")
    col.wilcoxon_two_sided_holm(reference="Rob")
    col.friedman_report()
    col.friedman_posthoc_holm(reference="Rob")
    col.noise_curve_plot(output_dir)
    col.critical_difference_plot(output_dir / "critical_difference")
    col.noise_curve_per_dataset(output_dir)
    col.bandwidth_sensitivity_plot(output_dir)
    if RUN_ABLATION:
        col.ablation_table()
    if Path(TIMING_CSV).exists():
        plot_single_model_timing(pd.read_csv(TIMING_CSV), output_dir)
    col.to_excel(output_dir / "binary_results_summary.xlsx")
    export_json_results(out_dir=output_dir / "json")
    return col

PROGRESS_COLS = ["seed", "noise", "dataset", "kernel", "method", "params",
                 "mean_accuracy", "std_accuracy", "time_sec"]

def _repair_csv(path):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return
    try:
        df = pd.read_csv(p, on_bad_lines='skip')
    except Exception as e:
        print(f"  [RESUME] خواندن {p} ممکن نشد ({e})؛ فایل دست‌نخورده ماند.")
        return
    before = sum(1 for _ in open(p, encoding='utf-8')) - 1
    if len(df) < before:
        print(f"  [RESUME] {before - len(df)} سطر ناقص در {p.name} حذف شد.")
    df.to_csv(p, index=False)

def _ensure_csv(path, cols):
    p = Path(path)
    p.parent.mkdir(exist_ok=True, parents=True)
    if not p.exists() or p.stat().st_size == 0:
        pd.DataFrame(columns=cols).to_csv(p, index=False)

def _append_rows(path, rows, cols):
    if not rows:
        return
    _ensure_csv(path, cols)

    p = Path(path)
    if p.stat().st_size > 0:
        with open(p, 'rb') as fb:
            fb.seek(-1, os.SEEK_END)
            if fb.read(1) != b'\n':
                with open(p, 'a', encoding='utf-8') as fa:
                    fa.write('\n')
                    fa.flush()
                    os.fsync(fa.fileno())
    df = pd.DataFrame(rows).reindex(columns=cols)
    with open(path, 'a', newline='', encoding='utf-8') as f:
        df.to_csv(f, header=False, index=False)
        f.flush()
        os.fsync(f.fileno())

def _noise_key(x):
    return round(float(x), 6)

UNIT_COLS = ["seed", "noise", "dataset", "kernel"]

def _mark_unit_complete(seed, noise, dataset, kernel):
    _append_rows(COMPLETED_CSV,
                 [{"seed": int(seed), "noise": _noise_key(noise),
                   "dataset": str(dataset), "kernel": str(kernel)}],
                 UNIT_COLS)

def load_done_keys(path=None) -> set:
    path = Path(path or COMPLETED_CSV)
    if not RESUME:
        return set()
    _repair_csv(path)
    if not path.exists() or path.stat().st_size == 0:
        return set()
    try:
        df = pd.read_csv(path, usecols=UNIT_COLS).dropna()
    except Exception:
        return set()
    if df.empty:
        return set()
    return {(_noise_key(n), str(k), int(s), str(d))
            for s, n, d, k in df[UNIT_COLS].itertuples(index=False)}

def rebuild_collector_from_progress(path=None) -> ResultsCollector:
    path = Path(path or PROGRESS_CSV)
    _repair_csv(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"فایل پیشرفت خالی یا موجود نیست: {path}")
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"فایل پیشرفت هیچ سطری ندارد: {path}")

    done_units = load_done_keys()
    if done_units:
        mask = [(_noise_key(n), str(k), int(s), str(d)) in done_units
                for s, n, d, k in df[UNIT_COLS].itertuples(index=False)]
        dropped = len(df) - int(np.sum(mask))
        if dropped:
            print(f"[RESUME] {dropped} سطر متعلق به واحدهای ناتمام نادیده گرفته شد.")
        df = df[mask]
    df = df.dropna(subset=["mean_accuracy"])

    df = df.drop_duplicates(subset=["seed", "noise", "dataset", "kernel", "method"],
                            keep="last")
    col = ResultsCollector()
    col.rows = df[["seed", "noise", "dataset", "kernel", "method",
                   "mean_accuracy", "std_accuracy", "time_sec"]].to_dict("records")
    print(f"[RESUME] {len(col.rows)} سطر نتیجه از {path.name} بازسازی شد.")
    return col

def _unit_in_shard(noise, kernel, seed, dataset) -> bool:
    if N_SHARDS <= 1:
        return True
    key = f"{_noise_key(noise)}|{kernel}|{int(seed)}|{dataset}"
    return (zlib.crc32(key.encode("utf-8")) % N_SHARDS) == SHARD_ID

def progress_report(path=None):
    keys = load_done_keys(path or COMPLETED_CSV)
    total = (len(LABEL_NOISE_RATIOS) * 2 * len(RANDOM_STATES)
             * max(1, len(DATASETS)))
    print(f"[RESUME] واحدهای کاری کامل‌شده: {len(keys)} از حدود {total} "
          f"({100.0 * len(keys) / max(1, total):.1f}%)")
    return keys

def _feature_noise_settings() -> dict:
    import inspect
    out = {}
    try:

        sig = inspect.signature(Dataset.__init__)
        for k, v in sig.parameters.items():
            if ("noise" in k or "sigma" in k) and \
                    v.default is not inspect.Parameter.empty:
                out[k] = v.default
    except Exception as e:
        out["dataset_defaults_error"] = f"{type(e).__name__}: {e}"
    try:
        sig = inspect.signature(add_feature_noise)
        out["add_feature_noise_signature"] = [k for k in sig.parameters]
    except Exception:
        pass
    return out or {"note": "پارامتری یافت نشد"}

def export_json_results(outer_csv=None, out_dir: Path = None,
                        per_dataset_files: bool = True):
    outer_csv = Path(outer_csv or NESTED_OUTER_CSV)
    out_dir = Path(out_dir or (OUTPUT_DIR / "json"))
    out_dir.mkdir(exist_ok=True, parents=True)

    if not outer_csv.exists() or outer_csv.stat().st_size == 0:
        print(f"[JSON] فایل {outer_csv} خالی است؛ خروجی JSON ساخته نشد.")
        return None
    _repair_csv(outer_csv)
    df = pd.read_csv(outer_csv)
    if df.empty:
        print("[JSON] داده‌ای برای خروجی نیست.")
        return None

    done_units = load_done_keys()
    if done_units:
        mask = [(_noise_key(n), str(k), int(s), str(d)) in done_units
                for s, n, d, k in df[UNIT_COLS].itertuples(index=False)]
        df = df[mask]

    written = []
    if per_dataset_files:
        for (ds_name, kernel), sub in df.groupby(["dataset", "kernel"]):
            payload = {}
            for method, m_sub in sub.groupby("method"):
                payload[METHOD_LABELS.get(method, method)] = {
                    str(nl): [float(a) for a in
                              nl_sub.sort_values(["seed", "outer_fold"])["test_accuracy"]]
                    for nl, nl_sub in m_sub.groupby("noise")
                }
            p = out_dir / f"accstore_{disp(ds_name).replace(' ', '_')}_{kernel}.json"
            p.write_text(json.dumps(payload, indent=1, ensure_ascii=False),
                         encoding="utf-8")
            written.append(p)

    summary = {}
    for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
        k1 = f"{kernel}"
        k2 = f"{noise}"
        summary.setdefault(k1, {})[k2] = {}
        for (ds_name, method), g in sub.groupby(["dataset", "method"]):
            acc = g["test_accuracy"].astype(float).values

            per_seed = g.groupby("seed")["test_accuracy"].mean().values
            summary[k1][k2].setdefault(disp(ds_name), {})[
                METHOD_LABELS.get(method, method)] = {
                "mean_accuracy": float(np.mean(per_seed)),
                "std_accuracy": float(np.std(per_seed)),
                "n_seeds": int(len(per_seed)),
                "n_outer_folds": int(len(acc)),
                "per_seed_accuracy": [float(v) for v in per_seed],
                "per_fold_accuracy": [float(v) for v in acc],
                "mean_fit_time_sec": float(g["fit_time_sec"].mean()),
            }

    selected = {}
    if Path(SELECTED_CFG_CSV).exists() and Path(SELECTED_CFG_CSV).stat().st_size > 0:
        _repair_csv(SELECTED_CFG_CSV)
        cfg = pd.read_csv(SELECTED_CFG_CSV)
        for (ds_name, kernel, method), g in cfg.groupby(
                ["dataset", "kernel", "method"]):
            selected.setdefault(kernel, {}).setdefault(disp(ds_name), {})[
                METHOD_LABELS.get(method, method)] = {
                "modal_params": str(g["modal_params"].value_counts().index[0]),
                "n_distinct_across_folds": int(g["n_distinct"].max()),
            }

    meta = {
        "selection_mode": SELECTION_MODE,
        "inner_scheme": INNER_SCHEME,
        "inner_val_fraction": INNER_VAL_FRACTION,
        "inner_folds": INNER_FOLDS,
        "n_outer_folds": N_OUTER_FOLDS,
        "seeds": list(RANDOM_STATES),
        "label_noise_levels": list(LABEL_NOISE_RATIOS),

        "feature_noise": _feature_noise_settings(),
        "methods": {m: METHOD_LABELS.get(m, m) for m in ALL_METHODS},
        "ablation_arms": {m: METHOD_LABELS.get(m, m) for m in ABLATION_ARMS},
        "note": ("دقت‌ها روی مجموعه‌ی آزمونِ تمیزِ هر فولد بیرونی سنجیده "
                 "شده‌اند و هایپرپارامترها فقط روی بخش اعتبارسنجیِ نویزی "
                 "انتخاب شده‌اند."),
    }

    timing = {}
    if Path(TIMING_CSV).exists() and Path(TIMING_CSV).stat().st_size > 0:
        t = pd.read_csv(TIMING_CSV)
        for (kernel, method), g in t.groupby(["kernel", "method"]):
            timing.setdefault(kernel, {})[METHOD_LABELS.get(method, method)] = {
                "median_fit_time_sec": float(g["fit_time_sec"].median()),
                "mean_fit_time_sec": float(g["fit_time_sec"].mean()),
                "std_fit_time_sec": float(g["fit_time_sec"].std(ddof=0)),
            }

    allp = out_dir / "results_all.json"
    allp.write_text(json.dumps(
        {"metadata": meta, "summary": summary,
         "selected_hyperparameters": selected, "timing": timing},
        indent=1, ensure_ascii=False), encoding="utf-8")
    written.append(allp)
    print(f"[JSON] {len(written)} فایل در {out_dir} نوشته شد "
          f"(از جمله {allp.name}).")
    return written

def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for r, idx in enumerate(order):
        run = max(run, (m - r) * p[idx])
        adj[idx] = min(1.0, run)
    return adj

def _collector_wilcoxon_two_sided(self, reference: str = "Rob",
                                  alpha: float = 0.05, methods=None):
    methods = methods or ALL_METHODS
    df = self.to_dataframe()
    if df.empty:
        print("[Wilcoxon2] داده‌ای موجود نیست.")
        return pd.DataFrame()

    out = []
    print(f"\n{'='*84}")
    print(f"Wilcoxon signed-rank دوطرفه با تصحیح Holm  "
          f"(مرجع = {METHOD_LABELS.get(reference, reference)}؛ واحد = دیتاست)")
    print(f"{'='*84}")
    for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
        piv = (sub[sub["method"].isin(methods)]
               .groupby(["dataset", "method"])["mean_accuracy"]
               .mean().unstack("method").dropna())
        n_ds = piv.shape[0]
        print(f"\n-- kernel={kernel} | noise={int(noise*100)}% | N={n_ds} دیتاست --")
        if reference not in piv.columns or n_ds < 5:
            print("  (تعداد دیتاست برای آزمون کافی نیست)")
            continue
        comps = [c for c in piv.columns if c != reference]
        rows, raw = [], []
        for comp in comps:
            diff = piv[reference].values - piv[comp].values
            if np.all(diff == 0):
                rows.append((comp, None, None, 0, 0.0, "tie"))
                raw.append(np.nan)
                continue
            stat, p = wilcoxon(diff, alternative="two-sided", zero_method="wilcox")
            rows.append((comp, float(stat), float(p), int((diff != 0).sum()),
                         float(np.median(diff)), None))
            raw.append(p)
        good = [i for i, v in enumerate(raw) if not np.isnan(v)]
        adj = np.full(len(raw), np.nan)
        if good:
            adj[good] = holm([raw[i] for i in good])

        for i, (comp, stat, p, neff, med, flag) in enumerate(rows):
            label = METHOD_LABELS.get(comp, comp)
            if flag == "tie":
                print(f"  vs {label:32s}: همه‌ی تفاضل‌ها صفر [tie]")
                out.append({"kernel": kernel, "noise": noise, "comparison": comp,
                            "W": None, "p": None, "p_holm": None, "n_eff": 0,
                            "median_diff": 0.0, "verdict": "tie"})
                continue
            pa = float(adj[i])
            if pa < alpha:
                verdict = "better" if med > 0 else "worse"
                mark = "***"
            else:
                verdict = "n.s.(" + ("+" if med > 0 else "-") + ")"
                mark = ""
            print(f"  vs {label:32s}: W={stat:7.1f}  p={p:.4f}  "
                  f"p_holm={pa:.4f}  n={neff:2d}  med_diff={med:+.4f}  "
                  f"{verdict:10s} {mark}")
            out.append({"kernel": kernel, "noise": noise, "comparison": comp,
                        "W": stat, "p": p, "p_holm": pa, "n_eff": neff,
                        "median_diff": med, "verdict": verdict})
    res = pd.DataFrame(out)
    if not res.empty:
        res.to_csv(RESULTS_DIR / "wilcoxon_two_sided_holm.csv", index=False)
        print(f"\n[Wilcoxon2] ذخیره شد: {RESULTS_DIR/'wilcoxon_two_sided_holm.csv'}")
    return res

ResultsCollector.wilcoxon_two_sided_holm = _collector_wilcoxon_two_sided

def _collector_friedman_posthoc(self, reference: str = "Rob",
                                alpha: float = 0.05, methods=None):
    from scipy.stats import friedmanchisquare
    methods = methods or ALL_METHODS
    df = self.to_dataframe()
    if df.empty:
        return pd.DataFrame()
    out = []
    print(f"\n{'='*84}\nFriedman + post-hoc هولم (واحد = دیتاست)\n{'='*84}")
    for (kernel, noise), sub in df.groupby(["kernel", "noise"]):
        piv = (sub[sub["method"].isin(methods)]
               .groupby(["dataset", "method"])["mean_accuracy"]
               .mean().unstack("method").dropna())
        n_ds = piv.shape[0]
        cols = [c for c in methods if c in piv.columns]
        if len(cols) < 3 or n_ds < 3:
            continue
        stat, p = friedmanchisquare(*[piv[c].values for c in cols])
        ranks = piv[cols].rank(axis=1, ascending=False, method="average")
        avg_rank = ranks.mean().to_dict()
        print(f"\n-- kernel={kernel} | noise={int(noise*100)}% | N={n_ds} --")
        print(f"  Friedman: chi2={stat:.3f}  p={p:.3e}")
        print("  میانگین رتبه: " + "  ".join(
            f"{METHOD_LABELS.get(c,c)}={avg_rank[c]:.2f}" for c in cols))
        if reference in cols:
            comps = [c for c in cols if c != reference]
            raw = []
            for c in comps:
                d = piv[reference].values - piv[c].values
                if np.all(d == 0):
                    raw.append(np.nan)
                else:
                    raw.append(wilcoxon(d, alternative="two-sided",
                                        zero_method="wilcox")[1])
            good = [i for i, v in enumerate(raw) if not np.isnan(v)]
            adj = np.full(len(raw), np.nan)
            if good:
                adj[good] = holm([raw[i] for i in good])
            for c, pa in zip(comps, adj):
                flag = "***" if (not np.isnan(pa) and pa < alpha) else "n.s."
                print(f"    post-hoc vs {METHOD_LABELS.get(c,c):32s}: "
                      f"p_holm={pa:.4f}  {flag}")
                out.append({"kernel": kernel, "noise": noise,
                            "friedman_chi2": stat, "friedman_p": p,
                            "comparison": c, "p_holm": float(pa),
                            "significant": bool(pa < alpha) if not np.isnan(pa) else None,
                            "avg_rank_reference": avg_rank[reference],
                            "avg_rank_comparison": avg_rank[c]})
    res = pd.DataFrame(out)
    if not res.empty:
        res.to_csv(RESULTS_DIR / "friedman_posthoc_holm.csv", index=False)
    return res

ResultsCollector.friedman_posthoc_holm = _collector_friedman_posthoc

In [ ]:
def _run_single_unit(ds, name, kernel, noise, seed, rob_arms, n_jobs):
    ds.seed = seed
    try:
        summ, outer, cfg, _raw = run_benchmark_on_dataset(
            ds, kernel, label_noise_ratio=noise, rob_arms=rob_arms,
            n_jobs=n_jobs)
        return name, summ, outer, cfg, None
    except Exception as e:
        return name, None, None, None, f"{type(e).__name__}: {e}"

def _iter_unit_results(preloaded_map, todo, kernel, noise, seed, rob_arms):
    if FLAT_PARALLEL and SELECTION_MODE == "nested" and todo:
        batch = DATASET_BATCH if DATASET_BATCH > 0 else len(todo)
        for i in range(0, len(todo), batch):
            chunk = todo[i:i + batch]
            try:
                res = run_batch_flat(preloaded_map, chunk, kernel, noise,
                                     seed, rob_arms)
            except Exception as e:
                print(f"  [SKIP-BATCH] {chunk}: {type(e).__name__}: {e}")
                continue
            for name in chunk:
                if name in res:
                    summ, outer, cfg = res[name]
                    yield name, summ, outer, cfg, None
                else:
                    yield name, None, None, None, "نتیجه‌ای تولید نشد"
        return

    if DATASET_PARALLEL <= 1 or len(todo) <= 1:
        for name in todo:
            yield _run_single_unit(preloaded_map[name], name, kernel, noise,
                                   seed, rob_arms, None)
        return

    print(f"  [CLOUD-DS] اجرای هم‌زمان {min(DATASET_PARALLEL, len(todo))} دیتاست "
          f"(n_jobs داخلی = 1)")
    tasks = (delayed(_run_single_unit)(preloaded_map[n], n, kernel, noise,
                                       seed, rob_arms, 1) for n in todo)
    try:

        results = Parallel(n_jobs=DATASET_PARALLEL, return_as="generator")(tasks)
    except TypeError:
        results = Parallel(n_jobs=DATASET_PARALLEL)(tasks)
    for r in results:
        yield r

def main_binary():
    global RANDOM_STATE, CURRENT_LABEL_NOISE

    preloaded_map = preload_all_binary_datasets(DATASETS)
    if not preloaded_map:
        print("[Binary] هیچ دیتاستی با موفقیت لود نشد؛ رد شدن از بخش دوکلاسه.")
        return None

    rob_arms = ("Rob",) + tuple(ABLATION_ARMS) if RUN_ABLATION else ("Rob",)

    _ensure_csv(PROGRESS_CSV, PROGRESS_COLS)
    _ensure_csv(COMPLETED_CSV, UNIT_COLS)
    _ensure_csv(NESTED_OUTER_CSV,
                ["seed", "noise", "dataset", "kernel", "method", "outer_fold",
                 "val_accuracy", "test_accuracy", "fit_time_sec", "params"])
    _ensure_csv(SELECTED_CFG_CSV,
                ["seed", "noise", "dataset", "kernel", "method",
                 "modal_params", "all_fold_params", "n_distinct"])

    done = load_done_keys()
    if done:
        print(f"[RESUME] {len(done)} واحد کاری از اجرای قبلی کامل است و رد می‌شود.")
    else:
        print("[RESUME] اجرای تازه (هیچ پیشرفت قبلی یافت نشد).")

    n_new = 0
    for noise in LABEL_NOISE_RATIOS:
        CURRENT_LABEL_NOISE = noise
        for _ds in preloaded_map.values():
            _ds.label_noise_ratio = noise
        print(f"\n================= LABEL_NOISE={CURRENT_LABEL_NOISE} =================")

        for kernel in globals().get("KERNELS_TO_RUN", ["linear", globals().get("TB_NONLINEAR", "rbf")]):
            print(f"\n{'#'*66}\n#  PHASE: kernel={kernel}\n{'#'*66}")
            active_names = list(preloaded_map.keys())

            for rs in RANDOM_STATES:
                RANDOM_STATE = rs
                np.random.seed(RANDOM_STATE)

                todo = [n for n in active_names
                        if (_noise_key(noise), kernel, int(rs), n) not in done
                        and _unit_in_shard(noise, kernel, rs, n)]
                if not todo:
                    print(f"----- kernel={kernel} | seed={rs} | همه‌ی دیتاست‌ها "
                          f"از قبل کامل‌اند؛ رد شد. -----")
                    continue
                print(f"\n----- kernel={kernel} | RANDOM_STATE={RANDOM_STATE} -----")
                print(f"[باقی‌مانده] {todo}")

                seed_rows = []
                for name, summ, outer, cfg, err in _iter_unit_results(
                        preloaded_map, todo, kernel, CURRENT_LABEL_NOISE,
                        RANDOM_STATE, rob_arms):
                    if err is not None:
                        print(f"  [SKIP-RUN] {name} | {kernel} | seed={rs}: {err}")
                        continue
                    if not summ:
                        print(f"  [SKIP-RUN] {name}: نتیجه‌ای تولید نشد.")
                        continue

                    prog_rows = [{**r, "seed": RANDOM_STATE,
                                  "noise": CURRENT_LABEL_NOISE} for r in summ]
                    _append_rows(PROGRESS_CSV, prog_rows, PROGRESS_COLS)
                    if outer:
                        _append_rows(
                            NESTED_OUTER_CSV,
                            [{**r, "seed": RANDOM_STATE,
                              "noise": CURRENT_LABEL_NOISE} for r in outer],
                            ["seed", "noise", "dataset", "kernel", "method",
                             "outer_fold", "val_accuracy", "test_accuracy",
                             "fit_time_sec", "params"])
                    if cfg:
                        _append_rows(
                            SELECTED_CFG_CSV,
                            [{**r, "seed": RANDOM_STATE,
                              "noise": CURRENT_LABEL_NOISE} for r in cfg],
                            ["seed", "noise", "dataset", "kernel", "method",
                             "modal_params", "all_fold_params", "n_distinct"])

                    _mark_unit_complete(RANDOM_STATE, CURRENT_LABEL_NOISE,
                                        name, kernel)
                    done.add((_noise_key(noise), kernel, int(rs), name))
                    n_new += 1
                    seed_rows.extend(summ)

                df_seed = pd.DataFrame(seed_rows)
                if not df_seed.empty:
                    for name in todo:
                        print_best_results(df_seed, name, kernel)

    print(f"\n[RESUME] {n_new} واحد کاری جدید در این اجرا تکمیل شد.")

    if not MAKE_REPORTS:
        print(f"[CLOUD] shard {SHARD_ID}/{N_SHARDS} تمام شد. تولید گزارش رد شد؛ "
              f"پس از اتمام همه‌ی shardها merge_reports.py را اجرا کنید.")
        return None

    if not globals().get("TB_FINAL_REPORTS", True):
        return None
    try:
        collector = rebuild_collector_from_progress()
    except Exception as e:
        print(f"[Binary] ساخت گزارش ممکن نشد: {e}")
        return None

    print("\n[Binary] تولید گزارش‌های نهایی ...")
    for noise in sorted(collector.to_dataframe()["noise"].unique()):
        for kernel in ("linear", globals().get("TB_NONLINEAR", "rbf")):
            collector.paper_style_table(kernel, noise)
    collector.boxplots(OUTPUT_DIR / "boxplots_binary")
    collector.wilcoxon_report_by_dataset(reference="Rob")
    collector.wilcoxon_two_sided_holm(reference="Rob")
    collector.wilcoxon_report(reference="Rob")
    collector.friedman_report()
    collector.friedman_posthoc_holm(reference="Rob")
    collector.noise_curve_plot(OUTPUT_DIR)
    collector.critical_difference_plot(OUTPUT_DIR / "critical_difference")
    collector.noise_curve_per_dataset(OUTPUT_DIR)
    if RUN_ABLATION:
        collector.ablation_table()
    collector.to_excel(OUTPUT_DIR / "binary_results_summary.xlsx")
    export_json_results(out_dir=OUTPUT_DIR / "json")

    if RUN_TIMING:
        if Path(TIMING_CSV).exists() and Path(TIMING_CSV).stat().st_size > 0:
            print(f"[Timing] {TIMING_CSV.name} از قبل موجود است؛ رد شد.")
            plot_single_model_timing(pd.read_csv(TIMING_CSV), OUTPUT_DIR)
        elif Path(SELECTED_CFG_CSV).exists():
            _repair_csv(SELECTED_CFG_CSV)
            cfg_df = pd.read_csv(SELECTED_CFG_CSV)
            if not cfg_df.empty:
                cfg_mode = (cfg_df.groupby(["dataset", "kernel", "method"])["modal_params"]
                                  .agg(lambda s: s.value_counts().index[0]).reset_index())
                df_time = single_model_timing(preloaded_map, cfg_mode)
                plot_single_model_timing(df_time, OUTPUT_DIR)

    print(f"\n[RESUME] پیشرفت در {PROGRESS_CSV} ذخیره شد. اگر اجرا نیمه‌کاره "
          f"بماند، کافی است همین کد را دوباره اجرا کنید تا از همان نقطه ادامه دهد.")
    print(f"[REV-REPRO] برای تغییر شکل‌ها یا آزمون‌ها بدون اجرای دوباره‌ی "
          f"بنچمارک، regenerate_all_reports() را اجرا کنید.")
    return collector

In [ ]:
def _margin_diff(model, XY: np.ndarray) -> np.ndarray:
    d1 = np.abs(XY.dot(model.w1) + model.b1)
    d2 = np.abs(XY.dot(model.w2) + model.b2)
    return d1 - d2

def plot_kde_weight_figure(X, y_noisy, flip_idx, bandwidth=0.7,
                            save_dir: Path = OUTPUT_DIR):
    save_dir.mkdir(exist_ok=True, parents=True)
    pos_mask = (y_noisy == 1)
    neg_mask = ~pos_mask
    X_pos = X[pos_mask]

    kde = KernelDensity(kernel='gaussian', bandwidth=bandwidth).fit(X_pos)

    x_min, x_max = X[:, 0].min() - 1.5, X[:, 0].max() + 1.5
    y_min, y_max = X[:, 1].min() - 1.5, X[:, 1].max() + 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    dens = np.exp(kde.score_samples(grid)).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    cf = ax.contourf(xx, yy, dens, levels=25, cmap='viridis', alpha=0.85)
    plt.colorbar(cf, ax=ax, label='Estimated density  $\\hat p(x \\mid +1)$')
    ax.scatter(X[pos_mask, 0], X[pos_mask, 1], marker='*', c='red',
               s=30, label='Class +1', edgecolors='none')
    ax.scatter(X[neg_mask, 0], X[neg_mask, 1], marker='o',
               facecolors='none', edgecolors='blue', s=22, label='Class -1')
    ax.scatter(X[flip_idx, 0], X[flip_idx, 1], marker='*', s=220,
               facecolors='none', edgecolors='lime', linewidths=2.4,
               label='Mislabeled point (true -1, labeled +1)')
    """ax.set_title('KDE class-conditional density (class +1)\nwith mislabeled point in low-density region')"""
    ax.set_xlabel('$X_1$'); ax.set_ylabel('$X_2$')
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout()
    p2d = save_dir / "synthetic_kde_density_2d.pdf"
    plt.savefig(p2d, bbox_inches='tight')
    plt.show()

    fig3 = plt.figure(figsize=(7, 6))
    ax3 = fig3.add_subplot(111, projection='3d')
    ax3.plot_surface(xx, yy, dens, cmap='viridis', linewidth=0, antialiased=True, alpha=0.95)
    ax3.set_xlabel('$X_1$'); ax3.set_ylabel('$X_2$'); ax3.set_zlabel('Density  $X_3$')
    ax3.set_title('KDE density surface (class +1)')
    plt.tight_layout()
    p3d = save_dir / "synthetic_kde_density_3d.pdf"
    plt.savefig(p3d, bbox_inches='tight')
    plt.show()

    print(f"[Figure 2] ذخیره شد: {p2d}\n[Figure 2] ذخیره شد: {p3d}")

def plot_decision_boundaries(X, y_noisy, flip_idx, ideal_model, methods: dict,
                              save_dir: Path = OUTPUT_DIR):
    save_dir.mkdir(exist_ok=True, parents=True)
    x_min, x_max = X[:, 0].min() - 1.5, X[:, 0].max() + 1.5
    y_min, y_max = X[:, 1].min() - 1.5, X[:, 1].max() + 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))
    XY = np.column_stack([xx.ravel(), yy.ravel()])

    pos_mask = (y_noisy == 1)
    neg_mask = ~pos_mask

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.scatter(X[pos_mask, 0], X[pos_mask, 1], marker='*', c='red',
               s=28, alpha=0.8, label='Class +1', zorder=2)
    ax.scatter(X[neg_mask, 0], X[neg_mask, 1], marker='o',
               facecolors='none', edgecolors='blue', s=22, alpha=0.7,
               label='Class -1', zorder=2)
    ax.scatter(X[flip_idx, 0], X[flip_idx, 1], marker='o', s=200,
               facecolors='none', edgecolors='lime', linewidths=2.4,
               label='Mislabeled point', zorder=3)

    from matplotlib.lines import Line2D
    legend_handles, legend_labels = ax.get_legend_handles_labels()

    if ideal_model.w1 is not None:
        Z_ideal = _margin_diff(ideal_model, XY).reshape(xx.shape)
        ax.contour(xx, yy, Z_ideal, levels=[0], colors='blue',
                   linewidths=2.6, linestyles='solid', zorder=1)
        legend_handles.append(Line2D([0], [0], color='blue', lw=2.6,
                                      label='Ideal LSTSVM (noise-free)'))

    colors = {
        "LSTSVM": "tab:blue",
        "KDEWLSTSVM": "tab:orange",
        "LSWLTSVM": "tab:green",
        _tb_rival_label(): "tab:red",
        "LS-GBLSTSVM": "tab:purple",
        "CL1QTSVM": "tab:brown",
    }
    for name, model in methods.items():
        if getattr(model, 'w1', None) is None:
            print(f"  [Warn] مدل {name} روی این داده fit نشد؛ رد شد.")
            continue
        Z = _margin_diff(model, XY).reshape(xx.shape)
        c = colors.get(name, 'black')
        ax.contour(xx, yy, Z, levels=[0], colors=c, linewidths=2.0,
                   linestyles='dashed', zorder=1)
        legend_handles.append(Line2D([0], [0], color=c, lw=2.0,
                                      linestyle='dashed', label=name))

    ax.legend(handles=legend_handles, fontsize=8, loc='best')
    """ax.set_title('Synthetic Dataset: Decision Boundaries under a Single Label-Noise Point\n'
                 '(solid blue = ideal LSTSVM on noise-free data)')"""
    ax.set_xlabel('$X_1$'); ax.set_ylabel('$X_2$')
    plt.tight_layout()
    fname = save_dir / "synthetic_decision_boundaries.pdf"
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f"[Figure 3] ذخیره شد: {fname}")

def run_synthetic_experiment(seed: int = 0, bandwidth: float = 0.7):
    print("\n" + "=" * 70)
    print("  Synthetic Dataset Analysis (Section 4.1 مقاله)")
    print("=" * 70)

    rng = np.random.default_rng(seed)
    mu_pos, cov_pos = np.array([2.0, 2.0]),  np.array([[0.6,  0.2], [0.2,  0.6]])

    mu_neg, cov_neg = np.array([-2.0, -2.0]), np.array([[1.0, -0.75], [-0.75, 1.0]])
    n_per_class = 200

    X_pos = rng.multivariate_normal(mu_pos, cov_pos, n_per_class)
    X_neg = rng.multivariate_normal(mu_neg, cov_neg, n_per_class)
    X = np.vstack([X_pos, X_neg])
    y_clean = np.hstack([np.ones(n_per_class), -np.ones(n_per_class)]).astype(int)

    neg_idx   = np.where(y_clean == -1)[0]

    _diff     = X[neg_idx] - mu_neg
    dists     = np.sqrt(np.einsum('ij,jk,ik->i', _diff, np.linalg.inv(cov_neg), _diff))
    flip_idx  = neg_idx[np.argmax(dists)]

    y_noisy = y_clean.copy()
    y_noisy[flip_idx] = +1
    _emp = np.cov(X[neg_idx].T)
    print(f"[Synthetic] کوواریانس تجربی کلاس -1: {np.round(_emp, 2).tolist()} | "
          f"همبستگی {(_emp[0,1] / np.sqrt(_emp[0,0] * _emp[1,1])):.2f}")
    print(f"[Synthetic] یک نمونه‌ی کلاس -1 (اندیس {flip_idx}) که در ناحیه‌ی "
          f"کم‌چگالی کلاس خودش قرار دارد، به کلاس +1 برچسب اشتباه خورد.")

    plot_kde_weight_figure(X, y_noisy, flip_idx, bandwidth=bandwidth)

    ideal_model = StdLSTSVM(c1=1.0, c2=1.0, kernel='linear')
    ideal_model.fit(X, y_clean)

    methods = {
        "LSTSVM":      StdLSTSVM(c1=1.0, c2=1.0, kernel='linear'),
        "KDEWLSTSVM":  RobustGMMLSTSVM(c1=1.0, c2=1.0, kernel='linear',
                                        method='kde', bandwidth=bandwidth),
        "LSWLTSVM":    LSWLTSVM(c1=1.0, c2=1.0, k_neighbor=5, t_param=1.0, kernel='linear'),
        _tb_rival_label(): _tb_rival(1.0),
        "CL1QTSVM": CL1QTSVM(c1=1.0, c2=1.0, kernel='linear'),

        "LS-GBLSTSVM": GBLSTSVM(c1=1.0, c2=1.0, pur=0.985, delbals=2, kernel='linear'),
    }
    for name, model in methods.items():
        model.fit(X, y_noisy)

    plot_decision_boundaries(X, y_noisy, flip_idx, ideal_model, methods)

    xx_m, yy_m = np.meshgrid(
        np.linspace(X[:, 0].min() - 1.5, X[:, 0].max() + 1.5, 200),
        np.linspace(X[:, 1].min() - 1.5, X[:, 1].max() + 1.5, 200))
    XY_m = np.column_stack([xx_m.ravel(), yy_m.ravel()])
    sign_ideal = np.sign(_margin_diff(ideal_model, XY_m))
    print("\n[Synthetic] Boundary disagreement with the ideal region (%):")
    for _name, _m in methods.items():
        if getattr(_m, 'w1', None) is None:
            continue
        _dis = np.mean(np.sign(_margin_diff(_m, XY_m)) != sign_ideal)
        print(f"  {_name:12s}: {_dis*100:.2f}%")

    print("\n[Synthetic] پایان تحلیل دیتاست مصنوعی.\n")
    return {"X": X, "y_clean": y_clean, "y_noisy": y_noisy,
            "flip_idx": flip_idx, "ideal_model": ideal_model, "methods": methods}

In [ ]:
def _adv_make_data(n_per_class=200, seed=0):
    rng = np.random.default_rng(seed)

    mu_p, cov_p = np.array([2, 2]),  np.array([[0.6, 0.2], [0.2, 0.6]])
    mu_n, cov_n = np.array([-2, -2]), np.array([[0.9, -0.2], [-0.2, 0.9]])
    Xp = rng.multivariate_normal(mu_p, cov_p, n_per_class)
    Xn = rng.multivariate_normal(mu_n, cov_n, n_per_class)
    X = np.vstack([Xp, Xn])
    y = np.hstack([np.ones(n_per_class, dtype=int), -np.ones(n_per_class, dtype=int)])
    return X, y

def _adv_far_side_idx(X, y, n_flip):
    idx_n = np.where(y == -1)[0]
    mu_p = X[y == +1].mean(axis=0)
    mu_n = X[y == -1].mean(axis=0)
    depth = (X[idx_n] - mu_p) @ (mu_n - mu_p)
    return idx_n[np.argsort(-depth)[:n_flip]]

def _adv_density_score(X, y, bandwidth=0.7):
    helper = RobustGMMLSTSVM(method='kde', bandwidth=bandwidth)
    score = np.zeros(len(y))
    for cls in (+1, -1):
        idx = np.where(y == cls)[0]
        score[idx] = helper._weights(X[idx])
    return score

def adversarial_flip(X, y, ratio, mechanism, seed=0, bandwidth=0.7):
    rng = np.random.default_rng(seed)
    n_flip = int(ratio * len(y))
    if n_flip == 0:
        return y.copy()
    if mechanism == 'random':
        flip_idx = rng.choice(len(y), size=n_flip, replace=False)
    elif mechanism == 'far_side_targeted':
        flip_idx = _adv_far_side_idx(X, y, n_flip)
    elif mechanism == 'high_density':

        score = _adv_density_score(X, y, bandwidth)
        flip_idx = []
        for cls in (+1, -1):
            idx = np.where(y == cls)[0]
            k = n_flip // 2 if cls == +1 else n_flip - n_flip // 2
            flip_idx.extend(idx[np.argsort(-score[idx])[:k]])
        flip_idx = np.array(flip_idx)
    else:
        raise ValueError(f"سازوکار ناشناخته: {mechanism}")
    y_noisy = y.copy()
    y_noisy[flip_idx] *= -1
    return y_noisy

def run_adversarial_experiment(
    ratios=(0.05, 0.10, 0.15, 0.20, 0.25, 0.30),
    mechanisms=('random', 'far_side_targeted', 'high_density'),
    seeds=(0, 1, 2, 3, 4),
    n_per_class=200,
    n_test_per_class=400,
    bandwidth=0.7,
    save_dir: Path = None
):
    save_dir = Path(OUTPUT_DIR) if save_dir is None else Path(save_dir)
    save_dir.mkdir(exist_ok=True, parents=True)

    _ADV_C_GRID = [10 ** i for i in range(-4, 4)]

    def _build(name, c):
        if name == "LSTSVM":
            return StdLSTSVM(c1=c, c2=c, kernel='linear')
        if name == "LSWLTSVM":
            return LSWLTSVM(c1=c, c2=c, k_neighbor=5, t_param=1.0, kernel='linear')
        if name == _tb_rival_label():
            return _tb_rival(c)
        if name == "CL1QTSVM":
            return CL1QTSVM(c1=c, c2=c, kernel='linear')
        if name == "KDEWLSTSVM":
            return RobustGMMLSTSVM(c1=c, c2=c, kernel='linear',
                                   method='kde', bandwidth=bandwidth)
        if name == "LS-GBLSTSVM":
            return GBLSTSVM(c1=c, c2=c, pur=0.985, delbals=2, kernel='linear')
        raise ValueError(name)

    method_names = ["LSTSVM", "LSWLTSVM", _tb_rival_label(), "KDEWLSTSVM", "LS-GBLSTSVM", "CL1QTSVM"]

    def select_and_fit(name, X_fit, y_fit, X_val, y_val):
        best_c, best_acc = _ADV_C_GRID[0], -1.0
        for c in _ADV_C_GRID:
            try:
                m = _build(name, c)
                m.fit(X_fit, y_fit)
                a = accuracy_score(y_val, m.predict(X_val))
            except Exception:
                continue
            if a > best_acc:
                best_acc, best_c = a, c
        model = _build(name, best_c)
        return model, best_c
    rows = []
    for seed in seeds:
        X_tr, y_tr = _adv_make_data(n_per_class, seed=seed)
        X_te, y_te = _adv_make_data(n_test_per_class, seed=10_000 + seed)
        for mech in mechanisms:
            for ratio in ratios:
                y_noisy = adversarial_flip(X_tr, y_tr, ratio, mech,
                                           seed=seed, bandwidth=bandwidth)

                idx_in, idx_val = train_test_split(
                    np.arange(len(y_noisy)), test_size=0.30,
                    stratify=y_noisy, random_state=seed)
                for name in method_names:
                    try:
                        model, c_sel = select_and_fit(
                            name, X_tr[idx_in], y_noisy[idx_in],
                            X_tr[idx_val], y_noisy[idx_val])
                        model.fit(X_tr, y_noisy)
                        acc = accuracy_score(y_te, model.predict(X_te))
                    except Exception as e:
                        print(f"  [SKIP] {name} | {mech} | ratio={ratio}: {e}")
                        acc, c_sel = np.nan, np.nan
                    rows.append({"seed": seed, "mechanism": mech,
                                 "ratio": ratio, "method": name,
                                 "accuracy": acc, "c_selected": c_sel})
    df = pd.DataFrame(rows)

    summary = (df.groupby(["mechanism", "ratio", "method"])["accuracy"]
                 .agg(["mean", "std"]).reset_index())
    print("\n[Adversarial] میانگین دقت آزمونِ تمیز (٪) روی سیدها:")
    for mech in mechanisms:
        piv = (summary[summary.mechanism == mech]
               .pivot(index="ratio", columns="method", values="mean")[method_names] * 100)
        print(f"\n--- mechanism = {mech} ---")
        print(piv.round(2).to_string())

    fig, axes = plt.subplots(1, len(mechanisms), figsize=(5.2 * len(mechanisms), 4.4),
                             sharey=True)
    if len(mechanisms) == 1:
        axes = [axes]
    styles = {"LSTSVM": ("o", "-"), "LSWLTSVM": ("s", "--"),
              _tb_rival_label(): ("^", "-."), "LS-GBLSTSVM": ("d", ":"), "CL1QTSVM": ("v", "-."),
              "KDEWLSTSVM": ("*", "-")}
    titles = {"random": "Random flipping",
              "far_side_targeted": "Far-side one-sided flipping",
              "high_density": "High-density-targeted flipping"}
    for ax, mech in zip(axes, mechanisms):
        sub = summary[summary.mechanism == mech]
        for name in method_names:
            s = sub[sub.method == name].sort_values("ratio")
            mk, ls = styles[name]
            lw = 2.4 if name == "KDEWLSTSVM" else 1.4
            ax.plot(s["ratio"] * 100, s["mean"] * 100, marker=mk, linestyle=ls,
                    linewidth=lw, markersize=9 if name == "KDEWLSTSVM" else 6,
                    label=name)
        ax.set_title(titles.get(mech, mech))
        ax.set_xlabel("Flipping ratio (%)")
        ax.grid(linestyle='--', alpha=0.5)
    axes[0].set_ylabel("Clean test accuracy (%)")
    axes[-1].legend(loc="lower left", fontsize=9)
    plt.tight_layout()
    fname = save_dir / "adversarial_label_noise.pdf"
    plt.savefig(fname, bbox_inches='tight')
    plt.close(fig)
    print(f"\n[Adversarial] شکل ذخیره شد: {fname}")
    df.to_csv(save_dir / "adversarial_raw.csv", index=False)
    summary.to_csv(save_dir / "adversarial_summary.csv", index=False)
    print(f"[Adversarial] نتایج خام در {save_dir/'adversarial_raw.csv'} ذخیره شد.")
    return df, summary

In [ ]:
import json as _tbjson
import time as _tbtime
from sklearn.model_selection import train_test_split as _tb_tts, StratifiedKFold as _tb_skf

TB_PROTOCOL = 'nested'
TB_OUTER_FOLDS = 5
TB_TEST_SIZE = 0.30
TB_CV_FOLDS = 5
SELECTION_MODE = "nested"
TB_C_GRID = [10.0 ** i for i in range(-4, 4)]

TB_C3_GRID = [1e-4, 1e-2, 1.0, 1e2, 1e4]
TB_PAIRS = [(c, cr) for c in TB_C_GRID for cr in TB_C3_GRID]
TB_C3_DEFAULT = 1e-4
TB_USE_EIGH = True
TB_APL_BATCH = 'linear'

TB_NONLINEAR = 'quad'
_SEED_OFFSETS.setdefault("inner", 30_000)
TB_ROB_NAME = {'full': 'Rob', 'hua': 'RobHua', 'self': 'RobSelf'}

def _tb_kde_logw(Xc, bw):
    if len(Xc) < 2:
        return np.zeros(len(Xc))
    return kde_logdens_at_samples(Xc, bw)

def _tb_kde_weights(XA, XB, bw):
    l1, l2 = _tb_kde_logw(XA, bw), _tb_kde_logw(XB, bw)
    off = max(l1.max(), l2.max())
    return np.exp(l1 - off), np.exp(l2 - off)

def _tb_mats(method, base, X, y, Z):
    y = np.asarray(y).ravel()
    iA, iB = np.where(y == 1)[0], np.where(y == -1)[0]
    if len(iA) == 0 or len(iB) == 0:
        return None
    e1, e2 = np.ones((len(iA), 1)), np.ones((len(iB), 1))
    E, F = np.hstack([Z[iA], e1]), np.hstack([Z[iB], e2])

    if method == 'Std':
        EtE, FtF = E.T @ E, F.T @ F
        return (FtF, EtE, F.T @ e2, EtE, FtF, E.T @ e1)

    if method in _ROB_VARIANT_OF:
        w1, w2 = _tb_kde_weights(X[iA], X[iB], base[0])
        EwE, FwF = (E * w1[:, None]).T @ E, (F * w2[:, None]).T @ F
        r1, r2 = (F * w2[:, None]).T @ e2, (E * w1[:, None]).T @ e1
        v = _ROB_VARIANT_OF[method]
        if v == 'full':
            return (FwF, EwE, r1, EwE, FwF, r2)
        if v == 'hua':
            return (F.T @ F, EwE, r1, E.T @ E, FwF, r2)
        return (FwF, E.T @ E, r1, EwE, F.T @ F, r2)

    if method == 'LSW':
        h = LSWLTSVM(k_neighbor=base[0], t_param=base[1])
        d1, d2 = h._within_weight_diag(X[iA]), h._within_weight_diag(X[iB])
        return (F.T @ F, (E * d1[:, None]).T @ E, (F * d2[:, None]).T @ e2,
                E.T @ E, (F * d2[:, None]).T @ F, (E * d1[:, None]).T @ e1)

    if method == 'GB':
        balls = gen_balls(np.hstack((Z, y.reshape(-1, 1))), pur=base[0], delbals=2)
        if not balls:
            return None
        Cn = np.array([b[0] for b in balls]); Rd = np.array([b[1] for b in balls])
        Lb = np.array([b[2] for b in balls])
        C_, D_ = Cn[Lb == 1], Cn[Lb != 1]
        if len(C_) == 0 or len(D_) == 0:
            return None
        Rp, Rm = Rd[Lb == 1].reshape(-1, 1), Rd[Lb != 1].reshape(-1, 1)
        g1, g2 = np.ones((len(C_), 1)), np.ones((len(D_), 1))
        Eg, Fg = np.hstack([C_, g1]), np.hstack([D_, g2])
        EtE, FtF = Eg.T @ Eg, Fg.T @ Fg
        return (FtF, EtE, Fg.T @ (g2 + Rm), EtE, FtF, Eg.T @ (g1 + Rp))

    if method == 'APL':
        lam = APL_RLSTSVM(theta=base[0], tau=base[1])._lambda
        return ('APL', E, F, e1, e2, lam, len(y))
    raise ValueError(method)

def _tb_solve1(M, rhs):
    try:
        return np.linalg.solve(M, rhs)
    except np.linalg.LinAlgError:
        return np.linalg.lstsq(M, rhs, rcond=None)[0]

def _tb_solve_batch(M, rhs):
    try:
        return np.linalg.solve(M, rhs)
    except np.linalg.LinAlgError:
        return np.stack([_tb_solve1(M[k], rhs[k]) for k in range(M.shape[0])])

def _tb_ls(P, Q, r, sign, pairs):
    r = np.asarray(r, float).ravel()
    p = P.shape[0]
    out = np.empty((len(pairs), p))
    if TB_USE_EIGH:
        by_c = {}
        for k, (c, cr) in enumerate(pairs):
            by_c.setdefault(c, []).append((k, cr))
        for c, lst in by_c.items():
            S = c * P + Q
            lam, V = np.linalg.eigh(0.5 * (S + S.T))
            lam = np.clip(lam, 0.0, None)
            Vr = V.T @ r
            for k, cr in lst:
                out[k] = sign * c * (V @ (Vr / (lam + cr)))
    else:
        I = np.eye(p)
        for k, (c, cr) in enumerate(pairs):
            out[k] = sign * _tb_solve1(P + Q / c + (cr / c) * I, r)
    return out

def _tb_apl_seq(E, F, e1, e2, lam_fn, n, which, pairs, max_iter):
    EtE, FtF = E.T @ E, F.T @ F
    p = E.shape[1]; I = np.eye(p)
    out = np.empty((len(pairs), p))
    for k, (c, cr) in enumerate(pairs):
        u = (1.0 / n) * np.ones((p, 1))
        for _ in range(max_iter):
            up = u
            if which == 1:
                L = np.asarray(lam_fn(F @ u + e2)).reshape(-1)
                FL = F * L[:, None]
                u = -_tb_solve1(FL.T @ F + EtE / c + (cr / c) * I, FL.T @ e2)
            else:
                L = np.asarray(lam_fn(e1 - E @ u)).reshape(-1)
                EL = E * L[:, None]
                u = _tb_solve1(EL.T @ E + FtF / c + (cr / c) * I, EL.T @ e1)
            if np.max(np.abs(u - up)) < 1e-6:
                break
        out[k] = u.ravel()
    return out

def _tb_apl_batch(E, F, e1, e2, lam_fn, n, which, pairs, max_iter):
    EtE, FtF = E.T @ E, F.T @ F
    p = E.shape[1]; K = len(pairs); I = np.eye(p)
    cs = np.array([c for c, _ in pairs]); crs = np.array([cr for _, cr in pairs])
    base = (EtE if which == 1 else FtF)[None] / cs[:, None, None] \
        + (crs / cs)[:, None, None] * I
    Zm, ev = (F, e2) if which == 1 else (E, e1)
    U = np.full((K, p), 1.0 / n); active = np.ones(K, bool)
    for _ in range(max_iter):
        idx = np.where(active)[0]
        if len(idx) == 0:
            break
        Ua = U[idx]
        xi = Zm @ Ua.T + ev if which == 1 else ev - Zm @ Ua.T
        L = np.asarray(lam_fn(xi)).reshape(xi.shape)
        ZL = Zm[None] * L.T[:, :, None]
        M = np.matmul(ZL.transpose(0, 2, 1), Zm) + base[idx]
        Un = _tb_solve_batch(M, np.matmul(ZL.transpose(0, 2, 1), ev))[:, :, 0]
        if which == 1:
            Un = -Un
        conv = np.max(np.abs(Un - Ua), axis=1) < 1e-6
        U[idx] = Un; active[idx[conv]] = False
    return U

def _tb_fit(method, base, X, y, Z, pairs1, pairs2, linear, max_iter=50):
    if method == 'CQS':
        return _tb_cqs_fit(X, y, np.hstack([Z]) if Z is not None else X, pairs1, pairs2)
    if method == 'GBF':
        return _tb_gbf_fit(base, y, Z, pairs1, pairs2)
    M = _tb_mats(method, base, X, y, Z)
    if M is None:
        return None
    if isinstance(M[0], str):
        _, E, F, e1, e2, lam, n = M
        f = _tb_apl_batch if (TB_APL_BATCH is True or (TB_APL_BATCH == 'linear' and linear)) \
            else _tb_apl_seq
        return f(E, F, e1, e2, lam, n, 1, pairs1, max_iter), \
            f(E, F, e1, e2, lam, n, 2, pairs2, max_iter)
    P1, Q1, r1, P2, Q2, r2 = M
    return _tb_ls(P1, Q1, r1, -1.0, pairs1), _tb_ls(P2, Q2, r2, +1.0, pairs2)

def _tb_qmap(X):
    X = np.asarray(X, float)
    i, j = np.triu_indices(X.shape[1])
    Q = X[:, i] * X[:, j]
    Q[:, i == j] *= 0.5
    return np.hstack([Q, X])

def _tb_factors(method, base, X, y, Z):
    y = np.asarray(y).ravel()
    iA, iB = np.where(y == 1)[0], np.where(y == -1)[0]
    if len(iA) == 0 or len(iB) == 0:
        return None
    one = lambda k: np.ones(k)
    E = np.hstack([Z[iA], one(len(iA))[:, None]])
    F = np.hstack([Z[iB], one(len(iB))[:, None]])
    e1, e2 = one(len(iA)), one(len(iB))
    if method == 'Std':
        return (F, e2, E, e1, e2), (E, e1, F, e2, e1)
    if method in _ROB_VARIANT_OF:
        w1, w2 = _tb_kde_weights(X[iA], X[iB], base[0])
        v = _ROB_VARIANT_OF[method]
        if v == 'full':
            return (F, w2, E, w1, w2), (E, w1, F, w2, w1)
        if v == 'hua':
            return (F, e2, E, w1, w2), (E, e1, F, w2, w1)
        return (F, w2, E, e1, w2), (E, w1, F, e2, w1)
    if method == 'LSW':
        h = LSWLTSVM(k_neighbor=base[0], t_param=base[1])
        d1, d2 = h._within_weight_diag(X[iA]), h._within_weight_diag(X[iB])
        return (F, e2, E, d1, d2), (E, e1, F, d2, d1)
    if method == 'GB':
        balls = gen_balls(np.hstack((Z, y.reshape(-1, 1))), pur=base[0], delbals=2)
        if not balls:
            return None
        Cn = np.array([b[0] for b in balls]); Rd = np.array([b[1] for b in balls])
        Lb = np.array([b[2] for b in balls])
        C_, D_ = Cn[Lb == 1], Cn[Lb != 1]
        if len(C_) == 0 or len(D_) == 0:
            return None
        Rp, Rm = Rd[Lb == 1], Rd[Lb != 1]
        Eg = np.hstack([C_, one(len(C_))[:, None]]); Fg = np.hstack([D_, one(len(D_))[:, None]])
        return (Fg, one(len(D_)), Eg, one(len(C_)), 1.0 + Rm), \
               (Eg, one(len(C_)), Fg, one(len(D_)), 1.0 + Rp)
    if method == 'APL':
        lam = APL_RLSTSVM(theta=base[0], tau=base[1])._lambda
        return ('APL', E, F, e1, e2, lam, len(y))
    raise ValueError(method)

def _tb_ls_fac(Xp, wP, Xq, wQ, rv, sign, pairs):
    p, m = Xp.shape[1], len(Xp) + len(Xq)
    if p <= m:
        P = (Xp * wP[:, None]).T @ Xp
        Q = (Xq * wQ[:, None]).T @ Xq
        return _tb_ls(P, Q, Xp.T @ rv, sign, pairs)
    M = np.vstack([Xp, Xq]); G = M @ M.T
    u = np.concatenate([rv, np.zeros(len(Xq))])
    out = np.empty((len(pairs), p))
    by_c = {}
    for k, (c, cr) in enumerate(pairs):
        by_c.setdefault(c, []).append((k, cr))
    for c, lst in by_c.items():
        s = np.sqrt(np.clip(np.concatenate([c * wP, wQ]), 0.0, None))
        A = s[:, None] * G * s[None, :]
        lam, U = np.linalg.eigh(0.5 * (A + A.T))
        lam = np.clip(lam, 0.0, None)
        Ut = U.T @ np.divide(u, s, out=np.zeros_like(u), where=s > 0)
        for k, cr in lst:
            out[k] = sign * c * (M.T @ (s * (U @ (Ut / (lam + cr)))))
    return out

def _tb_apl_q(E, F, e1, e2, lam_fn, n, which, pairs, max_iter=50, tol=1e-6):
    Zp, Zq, ev = (F, E, e2) if which == 1 else (E, F, e1)
    p, m = E.shape[1], len(E) + len(F)
    dual = p > m
    if dual:
        M = np.vstack([Zp, Zq]); G = M @ M.T; I_m = np.eye(m)
    else:
        ZqZq = Zq.T @ Zq; I_p = np.eye(p)
    out = np.empty((len(pairs), p))
    for k, (c, cr) in enumerate(pairs):
        u = (1.0 / n) * np.ones(p)
        for _ in range(max_iter):
            up = u
            xi = Zp @ u + ev if which == 1 else ev - Zp @ u
            L = np.asarray(lam_fn(xi.reshape(-1, 1))).reshape(-1)
            if dual:
                W = np.concatenate([L, np.full(len(Zq), 1.0 / c)])
                rhs = np.concatenate([L * ev, np.zeros(len(Zq))])
                u = M.T @ _tb_solve1(W[:, None] * G + (cr / c) * I_m, rhs)
            else:
                ZL = Zp * L[:, None]
                u = _tb_solve1(ZL.T @ Zp + ZqZq / c + (cr / c) * I_p, ZL.T @ ev)
            if which == 1:
                u = -u
            if np.max(np.abs(u - up)) < tol:
                break
        out[k] = u
    return out

def _tb_fit_q(method, base, X, y, Z, pairs1, pairs2, max_iter=50):
    if method == 'CQS':
        return _tb_cqs_fit(X, y, Z, pairs1, pairs2)
    if method == 'GBF':
        return _tb_gbf_fit(base, y, Z, pairs1, pairs2)
    Fc = _tb_factors(method, base, X, y, Z)
    if Fc is None:
        return None
    if isinstance(Fc[0], str):
        _, E, F, e1, e2, lam, n = Fc
        return (_tb_apl_q(E, F, e1, e2, lam, n, 1, pairs1, max_iter),
                _tb_apl_q(E, F, e1, e2, lam, n, 2, pairs2, max_iter))
    return (_tb_ls_fac(*Fc[0], -1.0, pairs1), _tb_ls_fac(*Fc[1], +1.0, pairs2))

from scipy.optimize import minimize as _tb_minimize

TB_ROBUST_RIVAL = 'GBF'
TB_GBF_RULE = 'unnormalized'
TB_GBF_EPS = 1e-5

TB_GB_RULE = 'unnormalized'

def _tb_balls_p(Z, y, pur, delbals=2):
    data = np.hstack((np.asarray(Z, float), np.asarray(y).reshape(-1, 1)))
    data = np.hstack((data, np.arange(len(data)).reshape(-1, 1)))
    gb = GBList(data); gb.init_granular_balls(purity=pur); gb.del_ball(num_data=delbals)
    return [(np.asarray(b.center, float), float(b.r), b.label, float(b.purity))
            for b in gb.granular_balls]

def _tb_gbf_scores(C, lab, pv, eps=None):
    eps = TB_GBF_EPS if eps is None else eps
    s = np.empty(len(C))
    for cls in (1, -1):
        idx = lab == cls
        if not idx.any():
            continue
        d = np.linalg.norm(C[idx] - C[idx].mean(0), axis=1)
        mu = 1.0 - d / (d.max() + eps)
        nu = np.sqrt(np.clip((1 - mu ** 2) * (1 - pv[idx]), 0, None))
        th = np.sqrt((1 - nu ** 2) / np.maximum(2 - mu ** 2 - nu ** 2, 1e-300))
        s[idx] = np.where(pv[idx] >= 1.0 - 1e-12, mu, th)
    return s

def _tb_box_qp(H, q, ub):
    H = 0.5 * (H + H.T)
    f = lambda a: (0.5 * a @ H @ a - q @ a, H @ a - q)
    r = _tb_minimize(f, np.zeros(len(q)), jac=True, method='L-BFGS-B',
                     bounds=list(zip(np.zeros(len(q)), ub)))
    return r.x

def _tb_kinv_ops(Eo, Fx):
    p, k = Eo.shape[1], len(Eo)
    if p <= k:
        lam, V = np.linalg.eigh(Eo.T @ Eo); lam = np.clip(lam, 0, None)
        VF = V.T @ Fx.T
        def ops(c3):
            KF = V @ (VF / (lam + c3)[:, None])
            return KF, Fx @ KF
    else:
        lam, U = np.linalg.eigh(Eo @ Eo.T); lam = np.clip(lam, 0, None)
        EF = Eo @ Fx.T; UEF = U.T @ EF; FF = Fx @ Fx.T
        def ops(c3):
            inner = U @ (UEF / (lam + c3)[:, None])
            KF = (Fx.T - Eo.T @ inner) / c3
            return KF, (FF - EF.T @ inner) / c3
    return ops

def _tb_gbf_fit(base, y, Z, pairs1, pairs2):
    balls = _tb_balls_p(Z, y, base[0])
    if not balls:
        return None
    C = np.array([b[0] for b in balls]); R = np.array([b[1] for b in balls])
    L = np.array([b[2] for b in balls]); Pv = np.array([b[3] for b in balls])
    ia, ib = L == 1, L != 1
    if not ia.any() or not ib.any():
        return None
    S = _tb_gbf_scores(C, np.where(ia, 1, -1), Pv)
    E = np.hstack([C[ia], np.ones((ia.sum(), 1))]); F = np.hstack([C[ib], np.ones((ib.sum(), 1))])
    out = []
    for Eo, Fx, rr, sc, sign, pairs in ((E, F, R[ib], S[ib], -1.0, pairs1),
                                        (F, E, R[ia], S[ia], +1.0, pairs2)):
        ops = _tb_kinv_ops(Eo, Fx)
        q = 1.0 + rr
        sol = np.empty((len(pairs), Eo.shape[1])); cache = {}
        for k, (c, c3) in enumerate(pairs):
            if c3 not in cache:
                cache[c3] = ops(c3)
            KF, H = cache[c3]
            sol[k] = sign * (KF @ _tb_box_qp(H, q, c * sc))
        out.append(sol)
    return out[0], out[1]

def _tb_predict(method, z1, z2, Zev):
    d1 = np.abs(Zev @ z1[:, :-1].T + z1[:, -1])
    d2 = np.abs(Zev @ z2[:, :-1].T + z2[:, -1])
    if method == 'GBF' and TB_GBF_RULE == 'paper':
        d1 = d1 / np.maximum(np.linalg.norm(z1[:, :-1], axis=1), 1e-300)
        d2 = d2 / np.maximum(np.linalg.norm(z2[:, :-1], axis=1), 1e-300)
        return np.where(d1 <= d2, 1, -1)
    if method == 'GB' and TB_GB_RULE == 'paper':
        d1 = d1 / np.maximum(np.linalg.norm(z1[:, :-1], axis=1), 1e-300)
        d2 = d2 / np.maximum(np.linalg.norm(z2[:, :-1], axis=1), 1e-300)
        return np.where(d1 <= d2, 1, -1)
    return np.where(d1 < d2, 1, -1)

def _tb_groups(method, kernel):
    G_ = [0.001, 0.01, 0.1, 1, 10] if kernel == 'rbf' else [None]
    if method in ('Std', 'CQS'):
        base = [()]
    elif method in _ROB_VARIANT_OF:
        base = [(bw,) for bw in np.around(np.arange(0.1, 2.0, 0.2), 1).tolist()]
    elif method == 'LSW':
        base = [(k, t) for k in range(1, 9) for t in [2 ** i for i in range(-1, 6)]]
    elif method in ('GB', 'GBF'):
        base = [(p,) for p in [0.90, 0.95, 0.985, 1.0]]
    elif method == 'APL':
        base = [(th, ta) for th in [10 ** i for i in range(-8, 5)]
                for ta in np.around(np.arange(0.1, 1.0, 0.1), 1).tolist()]
    return base, G_

def _tb_scaler_fit(X):
    X = np.asarray(X, float)
    keep = X.std(axis=0) > 0
    if not np.any(keep):
        keep = np.ones(X.shape[1], bool)
    lo, hi = X[:, keep].min(0), X[:, keep].max(0)
    return keep, lo, np.where(hi - lo > 0, hi - lo, 1.0)

def _tb_scaler_apply(sc, X):
    keep, lo, rng = sc
    return 2.0 * (np.asarray(X, float)[:, keep] - lo) / rng - 1.0

def _tb_make_outer(Xtr_raw, ytr_raw, Xte_raw, yte, ds, kernel, noise, fi):
    s0 = _tb_scaler_fit(Xtr_raw)
    Xtr0, Xte0 = _tb_scaler_apply(s0, Xtr_raw), _tb_scaler_apply(s0, Xte_raw)
    Xtr = add_feature_noise(Xtr0, col_noise_ratio=ds.feature_col_noise_ratio,
                            row_noise_ratio=ds.feature_row_noise_ratio,
                            seed=derive_seed(ds.seed, "feature", fi))
    ytr = add_noise(ytr_raw, noise_ratio=noise, seed=derive_seed(ds.seed, "label", fi))
    if VERBOSE_NOISE_CHECK:
        fl = int(np.sum(ytr != ytr_raw))
        print(f"  [NoiseCheck] ds={ds.name} kernel={kernel} seed={ds.seed} outer={fi} "
              f"target={noise:.2f} n_train={len(ytr)} flipped={fl} "
              f"({100.0 * fl / max(1, len(ytr)):.1f}%)")
    folds = list(_tb_skf(n_splits=TB_CV_FOLDS, shuffle=True,
                         random_state=derive_seed(ds.seed, "inner", fi)).split(Xtr, ytr))
    return Xtr, ytr, Xte0, yte, folds

def _tb_split(ds, kernel, noise):
    X, y = np.asarray(ds.X, float), np.asarray(ds.y).ravel()
    if TB_PROTOCOL == 'nested':
        skf = _tb_skf(n_splits=TB_OUTER_FOLDS, shuffle=True,
                      random_state=derive_seed(ds.seed, "split"))
        parts = list(skf.split(X, y))
    elif TB_PROTOCOL == 'holdout7030':
        parts = [_tb_tts(np.arange(len(y)), test_size=TB_TEST_SIZE, shuffle=True,
                         random_state=derive_seed(ds.seed, "split"))]
    else:
        raise ValueError(TB_PROTOCOL)
    return [_tb_make_outer(X[a], y[a], X[b], y[b], ds, kernel, noise, fi)
            for fi, (a, b) in enumerate(parts)]

def _tb_prep_pair(Xa, Xb, kernel):
    sc = _tb_scaler_fit(Xa)
    A, B = _tb_scaler_apply(sc, Xa), _tb_scaler_apply(sc, Xb)
    if kernel == 'rbf':
        return A, B, pairwise_distances(A, squared=True), pairwise_distances(B, A, squared=True)
    return A, B, None, None

def _tb_select_task(method, base, gammas, Xtr, ytr, folds, kernel):
    lin = kernel == 'linear'
    prepped = [(_tb_prep_pair(Xtr[tr], Xtr[va], kernel), ytr[tr], ytr[va]) for tr, va in folds]
    res = []
    for g in gammas:
        accs, ok = [], True
        for (Xi, Xv, Dii, Dvi), yi, yv in prepped:
            if kernel == 'quad':
                Zi, Zv = _tb_qmap(Xi), _tb_qmap(Xv)
                s = _tb_fit_q(method, base, Xi, yi, Zi, TB_PAIRS, TB_PAIRS)
            else:
                Zi = Xi if lin else np.exp(-g * Dii)
                Zv = Xv if lin else np.exp(-g * Dvi)
                s = _tb_fit(method, base, Xi, yi, Zi, TB_PAIRS, TB_PAIRS, lin)
            if s is None:
                ok = False; break
            accs.append((_tb_predict(method, s[0], s[1], Zv) == yv[:, None]).mean(0))
        res.append((g, np.mean(accs, axis=0) if ok else None))
    return res

def run_benchmark_on_dataset(ds, kernel, label_noise_ratio=None, rob_arms=("Rob",),
                             n_jobs=None):
    noise = ds.label_noise_ratio if label_noise_ratio is None else label_noise_ratio
    n_jobs = N_JOBS if n_jobs is None else n_jobs
    lin = kernel == 'linear'
    units = _tb_split(ds, kernel, noise)
    methods = list(TB_METHODS) + list(rob_arms)
    specs = [(fi, m, b, _tb_groups(m, kernel)[1]) for fi in range(len(units))
             for m in methods for b in _tb_groups(m, kernel)[0]]
    out = Parallel(n_jobs=n_jobs, batch_size='auto')(
        delayed(_tb_select_task)(m, b, G_, units[fi][0], units[fi][1], units[fi][4], kernel)
        for (fi, m, b, G_) in specs)
    best = {}
    for (fi, m, b, _), res in zip(specs, out):
        for g, acc in res:
            if acc is None:
                continue
            k = int(np.argmax(acc))
            if (fi, m) not in best or acc[k] > best[(fi, m)][0]:
                best[(fi, m)] = (float(acc[k]), b, g, TB_PAIRS[k])
    accs, times, chosen, outer = {}, {}, {}, []
    for fi, (Xtr, ytr, Xte, yte, _) in enumerate(units):
        Xtr_f, Xte_f, Dtr, Dte = _tb_prep_pair(Xtr, Xte, kernel)
        for m in methods:
            if (fi, m) not in best:
                continue
            va, b, g, (c, cr) = best[(fi, m)]
            t0 = _tbtime.time()
            if kernel == 'quad':
                Zt, Ze = _tb_qmap(Xtr_f), _tb_qmap(Xte_f)
                s = _tb_fit_q(m, b, Xtr_f, ytr, Zt, [(c, cr)], [(c, cr)])
            else:
                Zt = Xtr_f if lin else np.exp(-g * Dtr)
                Ze = Xte_f if lin else np.exp(-g * Dte)
                s = _tb_fit(m, b, Xtr_f, ytr, Zt, [(c, cr)], [(c, cr)], lin)
            ft = _tbtime.time() - t0
            if s is None:
                continue
            acc = float((_tb_predict(m, s[0], s[1], Ze)[:, 0] == yte).mean())
            pj = _tbjson.dumps({"base": list(b), "gamma": g, "c1": c, "c2": c,
                                "c3": cr, "c4": cr})
            accs.setdefault(m, []).append(acc); times.setdefault(m, []).append(ft)
            chosen.setdefault(m, []).append(pj)
            outer.append({"dataset": ds.name, "kernel": kernel, "method": m,
                          "outer_fold": fi, "val_accuracy": va, "test_accuracy": acc,
                          "fit_time_sec": ft, "params": pj})
    summary, cfg = [], []
    for m in methods:
        if m not in accs:
            continue
        modal = Counter(chosen[m]).most_common(1)[0][0]
        summary.append({"dataset": ds.name, "kernel": kernel, "method": m, "params": modal,
                        "mean_accuracy": float(np.mean(accs[m])),
                        "std_accuracy": float(np.std(accs[m])),
                        "time_sec": float(np.mean(times[m]))})
        cfg.append({"dataset": ds.name, "kernel": kernel, "method": m,
                    "modal_params": modal, "all_fold_params": " | ".join(chosen[m]),
                    "n_distinct": len(set(chosen[m]))})
    return summary, outer, cfg, pd.DataFrame()

def run_batch_flat(preloaded_map, todo, kernel, noise, seed, rob_arms=("Rob",), n_jobs=None):
    res = {}
    for name in todo:
        ds = preloaded_map[name]
        ds.seed = seed
        summ, outer, cfg, _ = run_benchmark_on_dataset(ds, kernel, noise, rob_arms, n_jobs)
        if summ:
            res[name] = (summ, outer, cfg)
    return res

def _tb_class_fit(self, X, y, K_tr=None):
    lin = self.kernel == 'linear'
    Z = X if lin else K_tr
    name, base = self._tb_id()
    c3 = getattr(self, 'c3', TB_C3_DEFAULT); c4 = getattr(self, 'c4', c3)
    s = _tb_fit(name, base, X, y, Z, [(self.c1, c3)], [(self.c2, c4)], lin,
                max_iter=getattr(self, 'max_iter', 50))
    if s is None:
        self.w1 = None
        return self
    self.w1, self.b1 = s[0][0, :-1], s[0][0, -1]
    self.w2, self.b2 = s[1][0, :-1], s[1][0, -1]
    return self

StdLSTSVM._tb_id = lambda self: ('Std', ())
RobustGMMLSTSVM._tb_id = lambda self: (TB_ROB_NAME[getattr(self, 'variant', 'full')],
                                       (self.bandwidth,))
LSWLTSVM._tb_id = lambda self: ('LSW', (self.k, self.t))
APL_RLSTSVM._tb_id = lambda self: ('APL', (self.theta, self.tau))
GBLSTSVM._tb_id = lambda self: ('GB', (self.pur,))
for _cls in (StdLSTSVM, RobustGMMLSTSVM, LSWLTSVM, APL_RLSTSVM, GBLSTSVM):
    _cls.fit = _tb_class_fit

def _tb_gb_predict(self, X, K_te=None):
    if self.w1 is None:
        return np.zeros(len(X), dtype=int)
    Z = X if self.kernel == 'linear' else K_te
    d1 = np.abs(Z @ self.w1 + self.b1) / max(np.linalg.norm(self.w1), 1e-300)
    d2 = np.abs(Z @ self.w2 + self.b2) / max(np.linalg.norm(self.w2), 1e-300)
    return np.where(d1 <= d2, 1, -1)

GBLSTSVM.predict = _tb_gb_predict

run_benchmark_on_dataset._tb_engine = True
run_batch_flat._tb_engine = True
_tb_class_fit._tb_engine = True
_tb_gb_predict._tb_engine = True

def tb_assert_engine(verbose=True):
    bad = []
    for nm in ("run_benchmark_on_dataset", "run_batch_flat"):
        if getattr(globals().get(nm), "_tb_engine", False) is not True:
            bad.append(nm)
    for cls in (StdLSTSVM, RobustGMMLSTSVM, LSWLTSVM, APL_RLSTSVM, GBLSTSVM):
        if getattr(cls.fit, "_tb_engine", False) is not True:
            bad.append(f"{cls.__name__}.fit")
    if getattr(GBLSTSVM.predict, "_tb_engine", False) is not True:
        bad.append("GBLSTSVM.predict")
    if TB_ROBUST_RIVAL != "GBF":
        bad.append(f"TB_ROBUST_RIVAL={TB_ROBUST_RIVAL!r}")
    if TB_NONLINEAR not in ("quad", "rbf"):
        bad.append(f"TB_NONLINEAR={TB_NONLINEAR!r}")
    if TB_PROTOCOL not in ("nested", "holdout7030"):
        bad.append(f"TB_PROTOCOL={TB_PROTOCOL!r}")
    if len(TB_PAIRS) != len(TB_C_GRID) * len(TB_C3_GRID):
        bad.append("TB_PAIRS با گریدهای c سازگار نیست")
    if bad:
        raise RuntimeError(
            "[GUARD] موتور TBSVM فعال نیست: " + "، ".join(bad) +
            "\nاحتمالاً سلولی پس از سلول موتور اجرا شده و تعریف‌ها را بازنویسی کرده است."
            "\nراه حل: سلول موتور TBSVM را دوباره اجرا کنید و بعد ادامه دهید."
            "\nاجرا متوقف شد تا نتیجه‌ی اشتباه تولید نشود.")
    if verbose:
        print(f"[GUARD] موتور فعال است | پروتکل={TB_PROTOCOL} | "
              f"ترکیب‌های c={len(TB_PAIRS)} | c3={[f'{v:g}' for v in TB_C3_GRID]}")
    return True

print(f"[TBSVM-ALL] فعال | پروتکل={TB_PROTOCOL} | TBSVM برای هر پنج روش | LS-GBLSTSVM | مقیاس‌بندی بدون نشت")

class GBFTSVM:
    def __init__(self, c1=1.0, c2=1.0, pur=0.985, c3=None, c4=None,
                 kernel='linear', gamma=None):
        self.c1, self.c2, self.pur = c1, c2, pur
        self.c3, self.c4, self.kernel, self.gamma = c3, c4, kernel, gamma
        self.w1 = None

    def fit(self, X, y, K_tr=None):
        Z = np.asarray(X, float) if self.kernel == 'linear' else K_tr
        c3 = TB_C3_DEFAULT if self.c3 is None else self.c3
        c4 = c3 if self.c4 is None else self.c4
        s = _tb_gbf_fit((self.pur,), np.asarray(y).ravel(), Z,
                        [(self.c1, c3)], [(self.c2, c4)])
        if s is None:
            self.w1 = None
            return self
        self.w1, self.b1 = s[0][0, :-1], s[0][0, -1]
        self.w2, self.b2 = s[1][0, :-1], s[1][0, -1]
        return self

    def predict(self, X, K_te=None):
        Z = np.asarray(X, float) if self.kernel == 'linear' else K_te
        if self.w1 is None:
            return np.ones(len(Z), dtype=int)
        d1, d2 = np.abs(Z @ self.w1 + self.b1), np.abs(Z @ self.w2 + self.b2)
        if TB_GBF_RULE == 'paper':
            d1 = d1 / max(np.linalg.norm(self.w1), 1e-300)
            d2 = d2 / max(np.linalg.norm(self.w2), 1e-300)
            return np.where(d1 <= d2, 1, -1)
        return np.where(d1 < d2, 1, -1)

def _tb_rival_label():
    return 'GBFTSVM' if TB_ROBUST_RIVAL == 'GBF' else 'APL-RLSTSVM'

def _tb_rival(c):
    if TB_ROBUST_RIVAL == 'GBF':
        return GBFTSVM(c1=c, c2=c, pur=0.985, kernel='linear')
    return APL_RLSTSVM(c1=c, c2=c, theta=1.0, tau=0.5, max_iter=50, kernel='linear')

def single_model_timing(preloaded_map, selected_cfg, noise=None, seed=None,
                        repeats=None, methods=None):
    noise = LABEL_NOISE_RATIOS[-1] if noise is None else noise
    seed = RANDOM_STATES[0] if seed is None else seed
    repeats = repeats or TIMING_REPEATS
    methods = methods or list(TB_METHODS) + ["Rob"]
    rows = []
    for name, ds in preloaded_map.items():
        ds.seed = seed
        for kernel in ("linear", TB_NONLINEAR):
            Xtr, ytr, Xte, yte, _ = _tb_split(ds, kernel, noise)[0]
            Xf, Xe, Dtr, Dte = _tb_prep_pair(Xtr, Xte, kernel)
            for m in methods:
                sub = selected_cfg[(selected_cfg.dataset == name) & (selected_cfg.kernel == kernel)
                                   & (selected_cfg.method == m)]
                if sub.empty:
                    continue
                p = _tbjson.loads(sub.iloc[0]["modal_params"])
                b, g = tuple(p["base"]), p["gamma"]
                pr1, pr2 = [(p["c1"], p["c3"])], [(p["c2"], p["c4"])]
                times = []
                for _ in range(repeats):
                    t0 = _tbtime.time()
                    if kernel == "quad":
                        _tb_fit_q(m, b, Xf, ytr, _tb_qmap(Xf), pr1, pr2)
                    else:
                        Z = Xf if kernel == "linear" else np.exp(-g * Dtr)
                        _tb_fit(m, b, Xf, ytr, Z, pr1, pr2, kernel == "linear")
                    times.append(_tbtime.time() - t0)
                for r_i, t_i in enumerate(times):
                    rows.append({"dataset": name, "kernel": kernel, "method": m,
                                 "noise": noise, "seed": seed, "n_train": len(ytr),
                                 "repeat": r_i, "fit_time_sec": float(t_i)})
    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(TIMING_CSV, index=False)
        print(f"[Timing] ذخیره شد: {TIMING_CSV}")
        print(df.groupby(["kernel", "method"])["fit_time_sec"].agg(["median", "mean", "std"]).round(5).to_string())
    return df

import re

_SI_SOURCES = {
    "hepatitis": ["https://archive.ics.uci.edu/ml/machine-learning-databases/hepatitis/hepatitis.data",
                  "https://raw.githubusercontent.com/renatopp/arff-datasets/master/classification/hepatitis.arff"],
    "vote":      ["https://archive.ics.uci.edu/ml/machine-learning-databases/voting-records/house-votes-84.data",
                  "https://raw.githubusercontent.com/renatopp/arff-datasets/master/classification/vote.arff"],
    "bands":     ["https://archive.ics.uci.edu/ml/machine-learning-databases/cylinder-bands/bands.data"],
}
_SI_LOCAL = {"hepatitis": ["hepatitis.data", "hepatitis.arff"],
             "vote": ["house-votes-84.data", "vote.arff"],
             "bands": ["bands.data", "cylinder-bands.data"]}

def _si_fetch(key):
    cached = CACHE_DIR / f"si_{key}.txt"
    if cached.exists():
        return cached.read_text(encoding="utf-8")
    for nm in _SI_LOCAL[key]:
        p = Path(DATA_DIR) / nm
        if p.exists():
            return p.read_text(encoding="utf-8", errors="replace")
    last = None
    for url in _SI_SOURCES[key]:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
            with urllib.request.urlopen(req, timeout=ONLINE_TIMEOUT) as r:
                raw = r.read().decode("utf-8", errors="replace")
            cached.write_text(raw, encoding="utf-8")
            print(f"  [SI-DATA] '{key}' از {url} دریافت شد.")
            return raw
        except Exception as e:
            last = e
    raise RuntimeError(f"دیتاست {key} دریافت نشد؛ فایل آن را در {DATA_DIR} بگذارید. ({last})")

def _si_rows(text):
    lines = [l.strip() for l in text.splitlines()]
    if any(l.lower().startswith("@data") for l in lines):
        noms, rows, data = [], [], False
        for l in lines:
            if not l or l.startswith("%"):
                continue
            if l.lower().startswith("@attribute"):
                m = re.search(r"\{(.*)\}", l)
                noms.append([v.strip().strip("'\"") for v in m.group(1).split(",")] if m else None)
            elif l.lower().startswith("@data"):
                data = True
            elif data:
                rows.append([v.strip().strip("'\"") for v in l.split(",")])
        return rows, noms, True
    return [[v.strip() for v in l.split(",")] for l in lines if l], None, False

def _si_num(v):
    return np.nan if v in ("?", "") else float(v)

def _raw_hepatitis():
    rows, noms, arff = _si_rows(_si_fetch("hepatitis"))
    if arff:
        X = [[(np.nan if v == "?" else (noms[j].index(v) + 1 if noms[j] else float(v)))
              for j, v in enumerate(r[:-1])] for r in rows]
        cls = [r[-1].upper() for r in rows]
    else:
        X = [[_si_num(v) for v in r[1:]] for r in rows]
        cls = ["DIE" if r[0] == "1" else "LIVE" for r in rows]
    return np.array(X, float), np.array(cls)

def _raw_vote():
    rows, noms, arff = _si_rows(_si_fetch("vote"))
    enc = {"y": 1.0, "n": -1.0, "?": 0.0}
    if arff:
        X = [[enc[v.lower()] for v in r[:-1]] for r in rows]; cls = [r[-1].lower() for r in rows]
    else:
        X = [[enc[v.lower()] for v in r[1:]] for r in rows]; cls = [r[0].lower() for r in rows]
    return np.array(X, float), np.array(cls)

def _raw_bands():
    rows, _, _ = _si_rows(_si_fetch("bands"))
    rows = [r for r in rows if len(r) == 40 and r[-1].strip().lower() in ("band", "noband")]
    feats = [r[4:39] for r in rows]; cls = [r[-1].strip().lower() for r in rows]
    cols = list(zip(*feats)); X = []
    for col in cols:
        vals = [v.strip().lower() for v in col]
        try:
            X.append([_si_num(v) for v in vals])
        except ValueError:
            cats = sorted({v for v in vals if v not in ("?", "")})
            X.append([np.nan if v in ("?", "") else float(cats.index(v) + 1) for v in vals])
    return np.array(X, float).T, np.array(cls)

BINARIZE_RULES.update({"Hepatitis": ("vs_rest", "DIE"),
                       "Vote": ("vs_rest", "democrat"),
                       "Cylinder_bands": ("vs_rest", "noband")})
for _n, _f in [("Cylinder_bands", _raw_bands), ("Hepatitis", _raw_hepatitis), ("Vote", _raw_vote)]:
    if _n not in {c["name"] for c in DATASETS}:
        _cfg = {"name": _n, "data": make_online_binary_loader(_n, _f), "source": "online",
                "kind": "binary", "rule": str(BINARIZE_RULES[_n])}
        UCI_ONLINE_DATASETS.append(_cfg); DATASETS.append(_cfg)

def _prepare_features(X):
    X = np.asarray(X, dtype=float)
    X = X[:, np.nan_to_num(np.nanstd(X, axis=0)) > 0]
    if X.shape[1] == 0:
        raise ValueError("پس از حذف ستون‌های با واریانس صفر، ویژگی‌ای نماند.")
    return X

def _tb_scaler_fit(X):
    X = np.asarray(X, float)
    keep = np.nan_to_num(np.nanstd(X, axis=0)) > 0
    if not np.any(keep):
        keep = np.ones(X.shape[1], bool)
    lo, hi = np.nanmin(X[:, keep], 0), np.nanmax(X[:, keep], 0)
    return keep, lo, np.where(hi - lo > 0, hi - lo, 1.0)

def _tb_prep_pair(Xa, Xb, kernel):
    sc = _tb_scaler_fit(Xa)
    A, B = _tb_scaler_apply(sc, Xa), _tb_scaler_apply(sc, Xb)
    if np.isnan(A).any() or np.isnan(B).any():
        med = np.nan_to_num(np.nanmedian(A, axis=0))
        A = np.where(np.isnan(A), med, A); B = np.where(np.isnan(B), med, B)
    if kernel == 'rbf':
        return A, B, pairwise_distances(A, squared=True), pairwise_distances(B, A, squared=True)
    return A, B, None, None

CQS_EPS, CQS_MAXIT, CQS_TOL = 1e-8, 30, 1e-8
CQS_DIAG = {"iters": [], "frac_capped": []}

def _cqs_w(r):
    a = np.abs(r)
    return np.where(a <= CQS_EPS, 1.0 / np.maximum(a, 1e-12), CQS_EPS)

def _tb_cqs_fit(X, y, Z, pairs1, pairs2):
    y = np.asarray(y).ravel()
    iA, iB = np.where(y == 1)[0], np.where(y == -1)[0]
    if len(iA) == 0 or len(iB) == 0:
        return None
    E = np.hstack([Z[iA], np.ones((len(iA), 1))]); F = np.hstack([Z[iB], np.ones((len(iB), 1))])
    oA, oB = np.ones(len(iA)), np.ones(len(iB))
    out = []
    for which, pairs in ((1, pairs1), (2, pairs2)):
        sol = np.empty((len(pairs), E.shape[1]))
        for k, pr in enumerate(pairs):
            if which == 1:
                th = _tb_ls_fac(F, oB, E, oA, oB, -1.0, [pr])[0]
            else:
                th = _tb_ls_fac(E, oA, F, oB, oA, +1.0, [pr])[0]
            it = 0
            for it in range(1, CQS_MAXIT + 1):
                if which == 1:
                    q, u = _cqs_w(E @ th), _cqs_w(1.0 + F @ th)
                    new = _tb_ls_fac(F, u, E, q, u, -1.0, [pr])[0]
                else:
                    f, g = _cqs_w(F @ th), _cqs_w(1.0 - E @ th)
                    new = _tb_ls_fac(E, g, F, f, g, +1.0, [pr])[0]
                done = np.max(np.abs(new - th)) <= CQS_TOL
                th = new
                if done:
                    break
            CQS_DIAG["iters"].append(it)
            if which == 1:
                CQS_DIAG["frac_capped"].append(float(np.mean(np.abs(E @ th) > CQS_EPS)))
            sol[k] = th
        out.append(sol)
    return out[0], out[1]

TB_ROBUST_RIVAL = 'GBF'
TB_METHODS = ['Std', 'LSW', 'GBF', 'GB', 'CQS']
ALL_METHODS = ["Std", "LSW", "GBF", "GB", "CQS", "Rob"]

def _raw_hepatitis():
    rows, noms, arff = _si_rows(_si_fetch("hepatitis"))
    if arff:
        X = [[(np.nan if v == "?" else (noms[j].index(v) + 1 if noms[j] else float(v)))
              for j, v in enumerate(r[:-1])] for r in rows]
        cls = [r[-1].upper() for r in rows]
    else:
        X = [[_si_num(v) for v in r[1:]] for r in rows]
        cls = ["DIE" if r[0] == "1" else "LIVE" for r in rows]
    X, cls = np.array(X, float), np.array(cls)
    keep = ~np.isnan(X).any(axis=1)
    return X[keep], cls[keep]

def _raw_vote():
    rows, noms, arff = _si_rows(_si_fetch("vote"))
    enc = {"y": 1.0, "n": -1.0, "?": np.nan}
    if arff:
        X = [[enc[v.lower()] for v in r[:-1]] for r in rows]; cls = [r[-1].lower() for r in rows]
    else:
        X = [[enc[v.lower()] for v in r[1:]] for r in rows]; cls = [r[0].lower() for r in rows]
    X, cls = np.array(X, float), np.array(cls)
    keep = ~np.isnan(X).any(axis=1)
    return X[keep], cls[keep]

def _bands_text():
    p = Path(DATA_DIR) / "bands.dat"
    if p.exists():
        return p.read_text(encoding="utf-8", errors="replace")
    try:
        import keel_ds, os as _os
        q = Path(_os.path.dirname(keel_ds.__file__)) / "data" / "balanced" / "raw" / "bands.dat"
        if q.exists():
            return q.read_text(encoding="utf-8", errors="replace")
    except ImportError:
        pass
    raise RuntimeError(f"فایل KEEL «bands.dat» پیدا نشد. آن را در {DATA_DIR} بگذارید "
                       "(یا: pip install keel-ds==0.2.5).")

def _raw_bands():
    rows, _, _ = _si_rows(_bands_text())
    rows = [r for r in rows if len(r) == 20]
    return np.array([[float(v) for v in r[:19]] for r in rows], float), \
           np.array([r[19].strip().lower() for r in rows])

_SI_EXPECT = {"Hepatitis": (80, 19, 13), "Vote": (232, 16, 124), "Cylinder_bands": (365, 19, 230)}

def _si_checked_loader(name, raw_fn):
    base = make_online_binary_loader(name, raw_fn)
    def _loader():
        X, y = base()
        n, d, npos = _SI_EXPECT[name]
        got = (X.shape[0], X.shape[1], int((y == 1).sum()))
        if got != (n, d, npos):
            raise RuntimeError(f"[DATA-CHECK] {name}: انتظار (n,d,n+)={(n, d, npos)} ولی {got}")
        print(f"  [DATA-CHECK] {name}: n={n}، d={d}، n+={npos} ✓")
        return X, y
    return _loader

for _c in DATASETS + UCI_ONLINE_DATASETS:
    if _c["name"] in _SI_EXPECT:
        _c["data"] = _si_checked_loader(_c["name"], {"Hepatitis": _raw_hepatitis, "Vote": _raw_vote,
                                                    "Cylinder_bands": _raw_bands}[_c["name"]])

def _tb_prep_pair(Xa, Xb, kernel):
    sc = _tb_scaler_fit(Xa)
    A, B = _tb_scaler_apply(sc, Xa), _tb_scaler_apply(sc, Xb)
    if kernel == 'rbf':
        return A, B, pairwise_distances(A, squared=True), pairwise_distances(B, A, squared=True)
    return A, B, None, None

_tb_split_inner = _tb_split
def _tb_split(ds, kernel, noise):
    if np.isnan(np.asarray(ds.X, float)).any():
        raise RuntimeError(f"[DATA-CHECK] {ds.name}: مقدار گمشده باقی مانده است.")
    return _tb_split_inner(ds, kernel, noise)

class CL1QTSVM:
    def __init__(self, c1=1.0, c2=1.0, c3=None, c4=None, kernel='linear', gamma=None):
        self.c1, self.c2, self.c3, self.c4, self.kernel = c1, c2, c3, c4, kernel
        self.w1 = None

    def _Z(self, X):
        X = np.asarray(X, float)
        return _tb_qmap(X) if self.kernel == 'quad' else X

    def fit(self, X, y, K_tr=None):
        c3 = TB_C3_DEFAULT if self.c3 is None else self.c3
        c4 = c3 if self.c4 is None else self.c4
        s = _tb_cqs_fit(np.asarray(X, float), np.asarray(y).ravel(), self._Z(X),
                        [(self.c1, c3)], [(self.c2, c4)])
        if s is not None:
            self.w1, self.b1 = s[0][0, :-1], s[0][0, -1]
            self.w2, self.b2 = s[1][0, :-1], s[1][0, -1]
        return self

    def predict(self, X, K_te=None):
        Z = self._Z(X)
        if self.w1 is None:
            return np.ones(len(Z), dtype=int)
        return np.where(np.abs(Z @ self.w1 + self.b1) < np.abs(Z @ self.w2 + self.b2), 1, -1)

def _tb_rival_label():
    return 'GBFTSVM'

def _tb_rival(c):
    return GBFTSVM(c1=c, c2=c, pur=0.985, kernel='linear')

TB_CODE_VERSION = "KDEWLSTSVM-final-2026-09-26" + ("-LOO" if TB_KDE_LOO else "")

def tb_run_fingerprint():
    cfg = {"version": TB_CODE_VERSION, "protocol": TB_PROTOCOL, "outer": TB_OUTER_FOLDS,
           "inner": TB_CV_FOLDS, "c_grid": [float(v) for v in TB_C_GRID],
           "c3_grid": [float(v) for v in TB_C3_GRID], "seeds": list(RANDOM_STATES),
           "noise": [float(v) for v in LABEL_NOISE_RATIOS], "methods": list(TB_METHODS),
           "rob_arms": ["Rob", "RobHua", "RobSelf"], "nonlinear": TB_NONLINEAR,
           "gb_rule": TB_GB_RULE, "gbf_rule": TB_GBF_RULE, "gbf_eps": TB_GBF_EPS,
           "cqs": [CQS_EPS, CQS_MAXIT, CQS_TOL], "datasets": sorted(d["name"] for d in DATASETS)}
    f = Path(OUTPUT_DIR) / "run_config.json"
    if f.exists():
        old = json.loads(f.read_text(encoding="utf-8"))
        diff = sorted(k for k in set(cfg) | set(old) if cfg.get(k) != old.get(k))

        if diff == ["noise"] and set(old.get("noise", [])) <= set(cfg["noise"]):
            added = sorted(set(cfg["noise"]) - set(old["noise"]))
            bak = Path(OUTPUT_DIR) / "run_config_before_noise_extension.json"
            if not bak.exists():
                bak.write_text(f.read_text(encoding="utf-8"), encoding="utf-8")
            f.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding="utf-8")
            print(f"[FINGERPRINT] سطح نویز تازه {added} به این پوشه اضافه شد؛ بقیه‌ی تنظیمات "
                  f"یکسان است و واحدهای کامل‌شده دوباره اجرا نمی‌شوند.")
            return
        if diff:
            raise RuntimeError(f"[FINGERPRINT] پوشه‌ی {OUTPUT_DIR} نتایج تنظیمات دیگری را دارد "
                               f"(تفاوت در: {diff}). یک پوشه‌ی خروجی تازه انتخاب کنید.")
        print("[FINGERPRINT] تنظیمات با اجرای قبلی همین پوشه یکسان است؛ ادامه از نقطه‌ی توقف.")
    else:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        f.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding="utf-8")
        print(f"[FINGERPRINT] تنظیمات اجرا در {f} ثبت شد.")

In [ ]:
import io as _io, gzip as _gzip, zipfile as _zipfile

_K6_EXPECT = {
    "Tic_tac_toe":  (958, 9, 626),
    "Monk_2":       (432, 6, 228),
    "Mammographic": (830, 5, 403),
    "SAheart":      (462, 9, 160),
    "Appendicitis": (106, 7, 21),
    "SPECTF_heart": (267, 44, 212),
}
_K6_KEEL = {"Tic_tac_toe": "tic-tac-toe", "Monk_2": "monk-2", "Mammographic": "mammographic",
            "SAheart": "saheart", "Appendicitis": "appendicitis", "SPECTF_heart": "spectfheart"}
_K6_POS = {"Tic_tac_toe": "positive", "Monk_2": "1", "Mammographic": "1",
           "SAheart": "1", "Appendicitis": "1", "SPECTF_heart": "1"}
_K6_DISPLAY = {"Tic_tac_toe": "Tic-tac-toe", "Monk_2": "Monk-2", "Mammographic": "Mammographic",
               "SAheart": "SAheart", "Appendicitis": "Appendicitis", "SPECTF_heart": "SPECTF heart"}
_K6_KEEL_ZIP = "https://sci2s.ugr.es/keel/dataset/data/classification/{}.zip"
_K6_UCI_SPECTF = ["https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.train",
                  "https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.test"]
_K6_PMLB_APP = ["https://github.com/EpistasisLab/pmlb/raw/master/datasets/appendicitis/appendicitis.tsv.gz"]

def _k6_get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
    with urllib.request.urlopen(req, timeout=ONLINE_TIMEOUT) as r:
        return r.read()

def _k6_text(keel):
    for p in (Path(DATA_DIR) / f"{keel}.dat", CACHE_DIR / f"keel_{keel}.dat"):
        if p.exists():
            return p.read_text(encoding="utf-8", errors="replace")
    cached = CACHE_DIR / f"keel_{keel}.dat"
    try:
        import keel_ds, os as _os
        q = Path(_os.path.dirname(keel_ds.__file__)) / "data" / "balanced" / "raw" / f"{keel}.dat"
        if q.exists():
            return q.read_text(encoding="utf-8", errors="replace")
    except ImportError:
        pass
    errs = []
    try:
        z = _zipfile.ZipFile(_io.BytesIO(_k6_get(_K6_KEEL_ZIP.format(keel))))
        nm = next(n for n in z.namelist() if n.endswith(".dat"))
        txt = z.read(nm).decode("utf-8", errors="replace")
        cached.write_text(txt, encoding="utf-8")
        print(f"  [KEEL6] '{keel}' از مخزن KEEL دریافت شد.")
        return txt
    except Exception as e:
        errs.append(f"KEEL: {e}")
    if keel == "spectfheart":
        try:
            lines = []
            for u in _K6_UCI_SPECTF:
                for l in _k6_get(u).decode("utf-8", errors="replace").splitlines():
                    v = [t.strip() for t in l.split(",") if t.strip() != ""]
                    if v:
                        lines.append(",".join(v[1:] + v[:1]))
            txt = "\n".join(lines)
            cached.write_text(txt, encoding="utf-8")
            print("  [KEEL6] 'spectfheart' از UCI (SPECTF.train + SPECTF.test) دریافت شد.")
            return txt
        except Exception as e:
            errs.append(f"UCI: {e}")
    if keel == "appendicitis":
        try:
            raw = _gzip.decompress(_k6_get(_K6_PMLB_APP[0])).decode("utf-8")
            rows = [l.split("\t") for l in raw.splitlines() if l.strip()][1:]
            txt = "\n".join(",".join(r) for r in rows)
            cached.write_text(txt, encoding="utf-8")
            print("  [KEEL6] 'appendicitis' از PMLB دریافت شد.")
            return txt
        except Exception as e:
            errs.append(f"PMLB: {e}")
    raise RuntimeError(f"دیتاست KEEL «{keel}» دریافت نشد. فایل {keel}.dat را از "
                       f"{_K6_KEEL_ZIP.format(keel)} بگیرید و در {DATA_DIR} بگذارید. ({errs})")

def _k6_raw(name):
    def _f():
        keel = _K6_KEEL[name]
        rows, _, _ = _si_rows(_k6_text(keel))
        d = _K6_EXPECT[name][1]
        rows = [r for r in rows if len(r) == d + 1]
        X, cls = [], []
        for r in rows:
            feats = r[:d]
            if name == "Tic_tac_toe":
                feats = [{"x": 1.0, "o": -1.0, "b": 0.0}[v.lower()] for v in feats]
            elif name == "SAheart":
                feats = [({"present": 1.0, "absent": 0.0}[v.lower()] if v.isalpha() else float(v))
                         for v in feats]
            else:
                feats = [float(v) for v in feats]
            X.append(feats)
            c = r[d].strip()
            try:
                c = str(int(float(c)))
            except ValueError:
                c = c.lower()
            cls.append(c)
        return np.array(X, float), np.array(cls)
    return _f

def _k6_checked_loader(name):
    base = make_online_binary_loader(name, _k6_raw(name))
    def _loader():
        X, y = base()
        n, d, npos = _K6_EXPECT[name]
        got = (X.shape[0], X.shape[1], int((y == 1).sum()))
        if got != (n, d, npos):
            raise RuntimeError(f"[DATA-CHECK] {name}: انتظار (n,d,n+)={(n, d, npos)} ولی {got}")
        print(f"  [DATA-CHECK] {name}: n={n}، d={d}، n+={npos} ✓")
        return X, y
    return _loader

DISPLAY_NAMES.update(_K6_DISPLAY)
BINARIZE_RULES.update({n: ("vs_rest", p) for n, p in _K6_POS.items()})
for _n in _K6_EXPECT:
    if _n not in {c["name"] for c in DATASETS}:
        _cfg = {"name": _n, "data": _k6_checked_loader(_n), "source": "online",
                "kind": "binary", "rule": str(BINARIZE_RULES[_n])}
        UCI_ONLINE_DATASETS.append(_cfg); DATASETS.append(_cfg)

In [ ]:
import io as _io, zipfile as _zipfile, gzip as _gzip, re as _re

EXT_INCLUDE_MONK1 = False
EXT_MIN_MINORITY  = 0.45

EXT_TIMEOUT       = 25
EXT_DEAD_SECONDS  = 600
EXT_USE_UCIMLREPO = True

_UCI_FTP = "https://archive.ics.uci.edu/ml/machine-learning-databases/"
_UCI_STATIC = "https://archive.ics.uci.edu/static/public/"
_GH = "https://raw.githubusercontent.com/"
_EXT_URL = {
    "penguins":   [_GH + "mwaskom/seaborn-data/master/penguins.csv"],
    "crabs":      [_GH + "vincentarelbundock/Rdatasets/master/csv/MASS/crabs.csv"],
    "synth":      [_GH + "vincentarelbundock/Rdatasets/master/csv/MASS/synth.tr.csv"],
    "italy_tr":   [_GH + "aeon-toolkit/aeon/main/aeon/datasets/data/ItalyPowerDemand/ItalyPowerDemand_TRAIN.ts"],
    "italy_te":   [_GH + "aeon-toolkit/aeon/main/aeon/datasets/data/ItalyPowerDemand/ItalyPowerDemand_TEST.ts"],
    "glass":      [_UCI_FTP + "glass/glass.data", _GH + "jbrownlee/Datasets/master/glass.csv"],
    "seeds":      [_UCI_FTP + "00236/seeds_dataset.txt", _GH + "jbrownlee/Datasets/master/wheat-seeds.csv"],
    "wq_red":     [_UCI_FTP + "wine-quality/winequality-red.csv",
                   _GH + "jbrownlee/Datasets/master/winequality-red.csv"],
    "wine":       [_UCI_FTP + "wine/wine.data", _GH + "jbrownlee/Datasets/master/wine.csv"],
    "yeast":      [_UCI_FTP + "yeast/yeast.data"],
    "vehicle_arff": [_GH + "renatopp/arff-datasets/master/classification/vehicle.arff"],
    "letter":     [_UCI_FTP + "letter-recognition/letter-recognition.data"],
    "optdigits":  [_UCI_FTP + "optdigits/optdigits.tra", _UCI_FTP + "optdigits/optdigits.tes"],
    "hill_zip":   [_UCI_STATIC + "166/hill+valley.zip"],
    "raisin_zip": [_UCI_STATIC + "850/raisin.zip"],
    "retino_zip": [_UCI_STATIC + "329/diabetic+retinopathy+debrecen.zip"],
    "cryo_zip":   [_UCI_STATIC + "429/cryotherapy+dataset.zip"],
}
_EXT_LOCAL = {
    "penguins": ["penguins.csv"], "crabs": ["crabs.csv"], "synth": ["synth.tr.csv"],
    "glass": ["glass.data", "glass.csv"], "seeds": ["seeds_dataset.txt", "wheat-seeds.csv"],
    "wq_red": ["winequality-red.csv"], "wine": ["wine.data", "wine.csv"],
    "yeast": ["yeast.data"], "letter": ["letter-recognition.data"],
    "raisin": ["Raisin_Dataset.xlsx", "Raisin_Dataset.csv"],
    "retino": ["messidor_features.arff"],
    "divorce": ["divorce.csv", "divorce.xlsx"],
    "somerville": ["SomervilleHappinessSurvey2015.csv"],
    "cryo": ["Cryotherapy.xlsx", "Cryotherapy.csv"],
    "hill": ["Hill_Valley_without_noise_Training.data", "Hill_Valley_without_noise_Testing.data"],
}

import socket as _socket, urllib.error as _uerr
_EXT_DEAD = {}

def _ext_host(url):
    return url.split("/")[2]

def _ext_dead(host):
    t = _EXT_DEAD.get(host)
    if t is not None and time.time() < t:
        return True
    _EXT_DEAD.pop(host, None)
    return False

def _ext_mark_dead(host, e):
    if isinstance(e, _uerr.HTTPError):
        return
    _EXT_DEAD[host] = time.time() + EXT_DEAD_SECONDS
    print(f"  [EXT][WARN] {host} پاسخ نداد ({type(e).__name__}). "
          f"{EXT_DEAD_SECONDS // 60} دقیقه کنار گذاشته می‌شود.")

def _ext_get(url):
    host = _ext_host(url)
    if _ext_dead(host):
        raise RuntimeError(f"{host} پاسخ نمی‌داد؛ رد شد")
    print(f"  [EXT] دریافت از {url} (حداکثر {EXT_TIMEOUT} ثانیه) ...", flush=True)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
        with urllib.request.urlopen(req, timeout=EXT_TIMEOUT) as r:
            return r.read()
    except Exception as e:
        _ext_mark_dead(host, e)
        raise

def _ext_bytes(key, urls=None):
    cached = CACHE_DIR / f"ext_{key}.bin"
    if cached.exists():
        return cached.read_bytes()
    for nm in _EXT_LOCAL.get(key, []):
        p = Path(DATA_DIR) / nm
        if p.exists():
            return p.read_bytes()
    errs = []
    for u in (urls if urls is not None else _EXT_URL[key]):
        try:
            raw = _ext_get(u)
            cached.write_bytes(raw)
            print(f"  [EXT] '{key}' از {u} دریافت شد.")
            return raw
        except Exception as e:
            errs.append(f"{u.split('/')[2]}: {type(e).__name__} {str(e)[:40]}")
    raise RuntimeError(f"منبعی برای «{key}» در دسترس نبود ({errs}). فایل را در {DATA_DIR} "
                       f"بگذارید: {_EXT_LOCAL.get(key, '')}")

def _ext_text(key, urls=None):
    raw = _ext_bytes(key, urls)
    for enc in ("utf-8-sig", "utf-16", "latin-1"):
        try:
            return raw.decode(enc)
        except UnicodeError:
            continue
    return raw.decode("utf-8", errors="replace")

def _ext_zip_member(zbytes, pattern):
    z = _zipfile.ZipFile(_io.BytesIO(zbytes))
    nm = next((n for n in z.namelist() if _re.search(pattern, n, _re.I)), None)
    if nm is None:
        raise RuntimeError(f"فایل مطابق «{pattern}» در zip نیست: {z.namelist()}")
    return z.read(nm)

def _ext_rows(text, sep=None):
    out = []
    for l in text.splitlines():
        l = l.strip()
        if not l or l.startswith(("#", "%", "@")):
            continue
        if sep is None:
            s = ";" if l.count(";") >= 2 else ("," if l.count(",") >= 1 else None)
        else:
            s = sep
        out.append([v.strip() for v in (l.split(s) if s else l.split())])
    return out

def _ext_ucirepo(uci_id):
    host = "archive.ics.uci.edu"
    if not EXT_USE_UCIMLREPO:
        raise RuntimeError("ucimlrepo غیرفعال است (EXT_USE_UCIMLREPO=False)")
    if _ext_dead(host):
        raise RuntimeError(f"{host} پاسخ نمی‌داد؛ رد شد")
    from ucimlrepo import fetch_ucirepo
    print(f"  [EXT] ucimlrepo id={uci_id} (حداکثر {EXT_TIMEOUT} ثانیه) ...", flush=True)
    old = _socket.getdefaulttimeout()
    _socket.setdefaulttimeout(EXT_TIMEOUT)
    try:
        r = fetch_ucirepo(id=uci_id)
    except Exception as e:
        _ext_mark_dead(host, e)
        raise
    finally:
        _socket.setdefaulttimeout(old)
    Xdf, Y = r.data.features, r.data.targets
    return Xdf.apply(pd.to_numeric, errors="raise"), Y.iloc[:, 0]

def _ext_try(providers, label):
    errs = []
    for nm, fn in providers:
        print(f"  [EXT] «{label}»: تلاش از {nm} ...", flush=True)
        try:
            X, cls = fn()
            print(f"  [EXT] «{label}» از {nm} خوانده شد.")
            return np.asarray(X, float), np.asarray(cls)
        except Exception as e:
            errs.append(f"{nm}: {type(e).__name__}: {str(e)[:70]}")
    raise RuntimeError(f"دیتاست «{label}» خوانده نشد. " + " | ".join(errs))

def _raw_penguins():
    d = pd.read_csv(_io.StringIO(_ext_text("penguins"))).dropna()
    X = d[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]].values
    return X, d["sex"].str.upper().values

def _raw_crabs():
    d = pd.read_csv(_io.StringIO(_ext_text("crabs")))
    d["sexn"] = (d["sex"] == "M").astype(float)
    return d[["sexn", "FL", "RW", "CL", "CW", "BD"]].values, d["sp"].values

def _raw_ripley():
    d = pd.read_csv(_io.StringIO(_ext_text("synth")))
    return d[["xs", "ys"]].values, d["yc"].astype(int).astype(str).values

def _ts_rows(text):
    X, cls = [], []
    for l in text.splitlines():
        l = l.strip()
        if not l or l.startswith(("#", "@")):
            continue
        a, b = l.rsplit(":", 1)
        X.append([float(v) for v in a.split(",")]); cls.append(b.strip())
    return X, cls

def _raw_italy():
    X1, c1 = _ts_rows(_ext_text("italy_tr")); X2, c2 = _ts_rows(_ext_text("italy_te"))
    return np.array(X1 + X2, float), np.array(c1 + c2)

def _raw_glass():
    rows = _ext_rows(_ext_text("glass"), ",")
    if len(rows[0]) == 11:
        rows = [r[1:] for r in rows]
    keep = [r for r in rows if r[-1] in ("1", "2", "3")]
    X = np.array([[float(v) for v in r[:-1]] for r in keep], float)
    cls = np.array(["float" if r[-1] in ("1", "3") else "nonfloat" for r in keep])
    return X, cls

def _raw_seeds():
    rows = [r for r in (_re.split(r"[,\s]+", l.strip()) for l in _ext_text("seeds").splitlines()
                        if l.strip()) if len(r) == 8]
    return np.array([[float(v) for v in r[:7]] for r in rows], float), \
           np.array([str(int(float(r[7]))) for r in rows])

def _raw_wq_red():
    rows = _ext_rows(_ext_text("wq_red"))
    rows = [r for r in rows if len(r) == 12]
    if not _re.match(r"^-?[\d.]+$", rows[0][0]):
        rows = rows[1:]
    q = np.array([int(float(r[11])) for r in rows])
    return np.array([[float(v) for v in r[:11]] for r in rows], float), \
           np.where(q >= 6, "good", "bad")

def _raw_wine12():
    def _keel():
        rows, _, _ = _si_rows(_k6_text("wine"))
        rows = [r for r in rows if len(r) == 14]
        return (np.array([[float(v) for v in r[:13]] for r in rows], float),
                np.array([str(int(float(r[13]))) for r in rows]))
    def _plain():
        rows = [r for r in _ext_rows(_ext_text("wine"), ",") if len(r) == 14]
        first_is_class = float(rows[0][0]) in (1.0, 2.0, 3.0) and float(rows[0][13]) > 3
        if first_is_class:
            return (np.array([[float(v) for v in r[1:]] for r in rows], float),
                    np.array([str(int(float(r[0]))) for r in rows]))
        return (np.array([[float(v) for v in r[:13]] for r in rows], float),
                np.array([str(int(float(r[13]))) for r in rows]))
    def _sk():
        from sklearn.datasets import load_wine
        d = load_wine()
        return d.data, (d.target + 1).astype(str)
    return _ext_try([("KEEL", _keel), ("UCI یا آینه", _plain), ("scikit-learn", _sk)], "Wine_12")

def _raw_yeast():
    rows = [r for r in (l.split() for l in _ext_text("yeast").splitlines() if l.strip()) if len(r) == 10]
    keep = [r for r in rows if r[9] in ("CYT", "NUC")]
    return np.array([[float(v) for v in r[1:9]] for r in keep], float), np.array([r[9] for r in keep])

def _raw_vehicle():
    def _keel():
        rows, _, _ = _si_rows(_k6_text("vehicle"))
        rows = [r for r in rows if len(r) == 19]
        return (np.array([[float(v) for v in r[:18]] for r in rows], float),
                np.array([r[18].strip().lower() for r in rows]))
    def _arff():
        rows, _, _ = _si_rows(_ext_text("vehicle_arff"))
        rows = [r for r in rows if len(r) == 19]
        return (np.array([[float(v) for v in r[:18]] for r in rows], float),
                np.array([r[18].strip().lower() for r in rows]))
    return _ext_try([("KEEL", _keel), ("آینه‌ی ARFF", _arff)], "Vehicle_bv")

def _raw_letter():
    def _keel():
        rows, _, _ = _si_rows(_k6_text("letter"))
        rows = [r for r in rows if len(r) == 17 and r[16].strip() in ("H", "Z")]
        return (np.array([[float(v) for v in r[:16]] for r in rows], float),
                np.array([r[16].strip() for r in rows]))
    def _uci():
        rows = [r for r in _ext_rows(_ext_text("letter"), ",") if len(r) == 17 and r[0] in ("H", "Z")]
        return (np.array([[float(v) for v in r[1:]] for r in rows], float),
                np.array([r[0] for r in rows]))
    return _ext_try([("KEEL", _keel), ("UCI", _uci)], "Letter_HZ")

def _raw_optdigits():
    def _keel():
        rows, _, _ = _si_rows(_k6_text("optdigits"))
        rows = [r for r in rows if len(r) == 65 and str(int(float(r[64]))) in ("3", "8")]
        return (np.array([[float(v) for v in r[:64]] for r in rows], float),
                np.array([str(int(float(r[64]))) for r in rows]))
    def _uci():
        rows = [r for r in _ext_rows(_ext_text("optdigits"), ",") if len(r) == 65 and r[64] in ("3", "8")]
        return (np.array([[float(v) for v in r[:64]] for r in rows], float),
                np.array([r[64] for r in rows]))
    return _ext_try([("KEEL", _keel), ("UCI", _uci)], "Optdigits_38")

def _ext_subsample(X, cls, n_total=1500, seed=0):
    order = np.lexsort(np.column_stack([X, cls.astype(str)]).T[::-1])
    X, cls = X[order], cls[order]
    rng = np.random.RandomState(seed)
    keep = []
    labs = np.unique(cls)
    for i, c in enumerate(labs):
        idx = np.where(cls == c)[0]
        k = n_total // len(labs) + (1 if i < n_total % len(labs) else 0)
        keep.extend(rng.choice(idx, size=k, replace=False).tolist())
    keep = np.sort(np.array(keep))
    return X[keep], cls[keep]

def _raw_keel_norm(keel, d):
    def _f():
        rows, _, _ = _si_rows(_k6_text(keel))
        rows = [r for r in rows if len(r) == d + 1]
        X = np.array([[float(v) for v in r[:d]] for r in rows], float)
        cls = np.array([str(int(float(r[d]))) for r in rows])
        return _ext_subsample(X, cls)
    return _f

def _read_any(name, data):
    n = name.lower()
    if n.endswith((".xlsx", ".xls")):
        return pd.read_excel(_io.BytesIO(data))
    txt = data.decode("utf-8-sig", errors="replace") if not data[:2] in (b"\xff\xfe", b"\xfe\xff") \
        else data.decode("utf-16")
    if n.endswith(".arff"):
        rows, _, _ = _si_rows(txt)
        return pd.DataFrame(rows)
    return pd.DataFrame(_ext_rows(txt))

def _raw_raisin():
    def _uci():
        Xd, y = _ext_ucirepo(850); return Xd.values, y.astype(str).values
    def _zip():
        df = _read_any("a.xlsx", _ext_zip_member(_ext_bytes("raisin_zip"), r"\.xlsx$"))
        return df.iloc[:, :7].astype(float).values, df.iloc[:, 7].astype(str).str.strip().values
    def _loc():
        for nm in _EXT_LOCAL["raisin"]:
            p = Path(DATA_DIR) / nm
            if p.exists():
                df = pd.read_excel(p) if nm.endswith("xlsx") else pd.read_csv(p)
                return df.iloc[:, :7].astype(float).values, df.iloc[:, 7].astype(str).str.strip().values
        raise FileNotFoundError("Raisin_Dataset در DATA_DIR نیست")
    return _ext_try([("فایل محلی", _loc), ("ucimlrepo", _uci), ("zip رسمی", _zip)], "Raisin")

def _raw_retino():
    def _uci():
        Xd, y = _ext_ucirepo(329); return Xd.values, y.astype(int).astype(str).values
    def _zip():
        rows, _, _ = _si_rows(_ext_zip_member(_ext_bytes("retino_zip"), r"\.arff$").decode("utf-8", "replace"))
        rows = [r for r in rows if len(r) == 20]
        return (np.array([[float(v) for v in r[:19]] for r in rows], float),
                np.array([str(int(float(r[19]))) for r in rows]))
    def _loc():
        p = Path(DATA_DIR) / "messidor_features.arff"
        rows, _, _ = _si_rows(p.read_text(encoding="utf-8", errors="replace"))
        rows = [r for r in rows if len(r) == 20]
        return (np.array([[float(v) for v in r[:19]] for r in rows], float),
                np.array([str(int(float(r[19]))) for r in rows]))
    return _ext_try([("فایل محلی", _loc), ("ucimlrepo", _uci), ("zip رسمی", _zip)], "Retinopathy")

def _raw_divorce():
    def _uci():
        Xd, y = _ext_ucirepo(539); return Xd.values, y.astype(int).astype(str).values
    def _loc():
        for nm in _EXT_LOCAL["divorce"]:
            p = Path(DATA_DIR) / nm
            if p.exists():
                df = pd.read_excel(p, header=None) if nm.endswith("xlsx") else \
                     pd.DataFrame(_ext_rows(p.read_text(encoding="utf-8-sig"), ";" if ";" in p.read_text(encoding="utf-8-sig")[:300] else ","))
                df = df.apply(pd.to_numeric, errors="coerce").dropna()
                return df.iloc[:, :54].values, df.iloc[:, 54].astype(int).astype(str).values
        raise FileNotFoundError("divorce.csv در DATA_DIR نیست")
    return _ext_try([("فایل محلی", _loc), ("ucimlrepo", _uci)], "Divorce")

def _raw_somerville():
    def _uci():
        Xd, y = _ext_ucirepo(479); return Xd.values, y.astype(int).astype(str).values
    def _loc():
        p = Path(DATA_DIR) / "SomervilleHappinessSurvey2015.csv"
        b = p.read_bytes()
        txt = b.decode("utf-16") if b[:2] in (b"\xff\xfe", b"\xfe\xff") else b.decode("utf-8-sig")
        df = pd.read_csv(_io.StringIO(txt))
        df.columns = [c.strip() for c in df.columns]
        return df[[f"X{i}" for i in range(1, 7)]].astype(float).values, df["D"].astype(int).astype(str).values
    return _ext_try([("فایل محلی", _loc), ("ucimlrepo", _uci)], "Somerville")

def _raw_cryo():
    def _uci():
        Xd, y = _ext_ucirepo(429); return Xd.values, y.astype(int).astype(str).values
    def _zip():
        df = _read_any("a.xlsx", _ext_zip_member(_ext_bytes("cryo_zip"), r"\.xlsx$"))
        return df.iloc[:, :6].astype(float).values, df.iloc[:, 6].astype(int).astype(str).values
    def _loc():
        for nm in _EXT_LOCAL["cryo"]:
            p = Path(DATA_DIR) / nm
            if p.exists():
                df = pd.read_excel(p) if nm.endswith("xlsx") else pd.read_csv(p)
                return df.iloc[:, :6].astype(float).values, df.iloc[:, 6].astype(int).astype(str).values
        raise FileNotFoundError("Cryotherapy در DATA_DIR نیست")
    return _ext_try([("فایل محلی", _loc), ("ucimlrepo", _uci), ("zip رسمی", _zip)], "Cryotherapy")

def _raw_hill():
    def _parse(txt):
        rows = [r for r in _ext_rows(txt, ",") if len(r) == 101]
        if rows and not _re.match(r"^-?[\d.eE+-]+$", rows[0][0]):
            rows = rows[1:]
        return [[float(v) for v in r[:100]] for r in rows], [str(int(float(r[100]))) for r in rows]
    def _zip():
        z = _ext_bytes("hill_zip"); Xs, cs = [], []
        for part in ("Training", "Testing"):
            a, b = _parse(_ext_zip_member(z, rf"without_noise_{part}\.data$").decode("utf-8", "replace"))
            Xs += a; cs += b
        return np.array(Xs, float), np.array(cs)
    def _loc():
        Xs, cs = [], []
        for nm in _EXT_LOCAL["hill"]:
            a, b = _parse((Path(DATA_DIR) / nm).read_text(encoding="utf-8", errors="replace"))
            Xs += a; cs += b
        return np.array(Xs, float), np.array(cs)
    return _ext_try([("فایل محلی", _loc), ("zip رسمی", _zip)], "HillValley")

def _raw_monk1():
    import itertools as _it
    dom = np.array(list(_it.product((1, 2, 3), (1, 2, 3), (1, 2), (1, 2, 3), (1, 2, 3, 4), (1, 2))), float)
    y = ((dom[:, 0] == dom[:, 1]) | (dom[:, 4] == 1)).astype(int)
    return dom, y.astype(str)

_EXT_META = {
    "Raisin":          ("Raisin",           _raw_raisin,    ("vs_rest", "Kecimen"), (900, 7, (450, 450))),
    "Retinopathy":     ("Retinopathy",      _raw_retino,    ("vs_rest", "1"),       (1151, 19, (540, 611))),
    "Penguins":        ("Penguins (sex)",   _raw_penguins,  ("vs_rest", "MALE"),    (333, 4, (165, 168))),
    "Divorce":         ("Divorce",          _raw_divorce,   ("vs_rest", "1"),       (170, 54, (84, 86))),
    "Somerville":      ("Somerville",       _raw_somerville, ("vs_rest", "1"),      (143, 6, None)),
    "Cryotherapy":     ("Cryotherapy",      _raw_cryo,      ("vs_rest", "1"),       (90, 6, (42, 48))),
    "Crabs":           ("Crabs",            _raw_crabs,     ("vs_rest", "O"),       (200, 6, (100, 100))),
    "ItalyPower":      ("ItalyPowerDemand", _raw_italy,     ("vs_rest", "1"),       (1096, 24, (547, 549))),
    "HillValley":      ("Hill-Valley",      _raw_hill,      ("vs_rest", "1"),       (1212, 100, (600, 612))),
    "Vehicle_bv":      ("Vehicle (bus vs van)", _raw_vehicle,  ("pair", "bus", "van"), (417, 18, (199, 218))),
    "Wine_12":         ("Wine (1 vs 2)",    _raw_wine12,    ("pair", "1", "2"),     (130, 13, (59, 71))),
    "Glass_win":       ("Glass (window)",   _raw_glass,     ("vs_rest", "float"),   (163, 9, (76, 87))),
    "Seeds_12":        ("Seeds (Kama vs Rosa)", _raw_seeds,    ("pair", "1", "2"),     (140, 7, (70, 70))),
    "Letter_HZ":       ("Letter (H vs Z)",     _raw_letter,    ("pair", "H", "Z"),     (1468, 16, (734, 734))),
    "Optdigits_38":    ("Optdigits (3 vs 8)",  _raw_optdigits, ("pair", "3", "8"),     (1126, 64, (554, 572))),
    "Yeast_CN":        ("Yeast (CYT vs NUC)",  _raw_yeast,     ("vs_rest", "CYT"),     (892, 8, (429, 463))),
    "WineQuality_red": ("Wine quality (red)", _raw_wq_red,  ("vs_rest", "good"),    (1599, 11, (744, 855))),
    "Twonorm":         ("Twonorm",          _raw_keel_norm("twonorm", 20), ("vs_rest", "1"), (1500, 20, None)),
    "Ringnorm":        ("Ringnorm",         _raw_keel_norm("ring", 20),    ("vs_rest", "1"), (1500, 20, None)),
    "Ripley":          ("Ripley",           _raw_ripley,    ("vs_rest", "1"),       (250, 2, (125, 125))),
}
if EXT_INCLUDE_MONK1:
    _EXT_META["Monk_1"] = ("Monk-1", _raw_monk1, ("vs_rest", "1"), (432, 6, (216, 216)))

def _ext_checked_loader(name):
    disp_name, raw_fn, rule, (n, d, cnt) = _EXT_META[name]
    def _loader():
        X0, cls = raw_fn()
        X, y, desc = binarize_and_finalize(X0, cls, rule)
        got = tuple(sorted((int((y == 1).sum()), int((y == -1).sum()))))
        if X.shape[0] != n or X0.shape[1] != d or (cnt is not None and got != cnt):
            raise RuntimeError(f"[DATA-CHECK] {name}: انتظار (n, d, کلاس‌ها)={(n, d, cnt)} "
                               f"ولی {(X.shape[0], X0.shape[1], got)}")
        minor = got[0] / len(y)
        if minor < EXT_MIN_MINORITY:
            raise RuntimeError(f"[DATA-CHECK] {name}: سهم کلاس کوچک {minor:.3f} < {EXT_MIN_MINORITY}")
        print(f"  [DATA-CHECK] {name}: n={X.shape[0]}، d={X0.shape[1]} (پس از حذف ثابت‌ها {X.shape[1]})، "
              f"کلاس‌ها={got}، کوچک={minor:.3f} ✓")
        return X, y
    return _loader

_BAD_FN_CHARS = set('\\/:*?"<>|')
for _n, (_disp, *_rest) in _EXT_META.items():
    if set(_disp) & _BAD_FN_CHARS:
        raise ValueError(f"[DATA-EXT] نام نمایشی «{_disp}» نویسه‌ی غیرمجاز برای نام فایل دارد")

EXT_DATASET_NAMES = []
for _n, (_disp, _raw, _rule, _exp) in _EXT_META.items():
    DISPLAY_NAMES[_n] = _disp
    BINARIZE_RULES[_n] = _rule
    if _n not in {c["name"] for c in DATASETS}:
        _cfg = {"name": _n, "data": _ext_checked_loader(_n), "source": "online",
                "kind": "binary", "rule": str(_rule)}
        UCI_ONLINE_DATASETS.append(_cfg); DATASETS.append(_cfg)
        EXT_DATASET_NAMES.append(_n)
print(f"[DATA-EXT] {len(EXT_DATASET_NAMES)} دیتاست تازه به رجیستری اضافه شد: {EXT_DATASET_NAMES}")

def ext_check_data(names=None):
    rows = []
    for n in (names or EXT_DATASET_NAMES):
        t0 = time.time()
        try:
            X, y = _ext_checked_loader(n)()
            rows.append(dict(dataset=n, status="OK", seconds=round(time.time() - t0, 1),
                             info=f"n={X.shape[0]}, d={X.shape[1]}"))
        except Exception as e:
            rows.append(dict(dataset=n, status="FAIL", seconds=round(time.time() - t0, 1),
                             info=str(e)[:150]))
    df = pd.DataFrame(rows)
    print("\n" + df.to_string(index=False))
    bad = df[df.status != "OK"].dataset.tolist()
    if bad:
        print(f"\n[EXT-CHECK] {len(bad)} دیتاست خوانده نشد: {bad}\n"
              f"فایل آن‌ها را دستی در {DATA_DIR} بگذارید (نام فایل‌ها در پیام هر خطا آمده) "
              f"یا نامشان را به EXCLUDE_DATASETS در سلول کنترل بیفزایید.")
    return df

In [ ]:
KERNELS_TO_RUN = ["linear", globals().get("TB_NONLINEAR", "rbf")]
TB_FINAL_REPORTS = True
TB_MAIN5 = ['Std', 'LSW', 'GBF', 'GB', 'CQS', 'Rob']
TB_LABEL = {'Std': 'LSTSVM', 'LSW': 'LSWLTSVM', 'APL': 'APL-RLSTSVM', 'GBF': 'GBFTSVM', 'CQS': 'CL1QTSVM',
            'GB': 'LS-GBLSTSVM', 'Rob': 'KDEWLSTSVM'}

def tb_kernel_summary(kernel):
    _repair_csv(PROGRESS_CSV)
    df = pd.read_csv(PROGRESS_CSV)
    df = df[(df.kernel == kernel) & (df.method.isin(TB_MAIN5))]
    if df.empty:
        print(f"[Summary] نتیجه‌ای برای kernel={kernel} نیست."); return None
    rows = []
    for noise, g in df.groupby('noise'):
        piv = g.groupby(['dataset', 'method']).mean_accuracy.mean().unstack()
        piv = piv.reindex(columns=TB_MAIN5).dropna(axis=0)
        rk = piv.rank(axis=1, ascending=False, method='average')
        for m in TB_MAIN5:
            rows.append({'noise': noise, 'method': TB_LABEL[m],
                         'mean_acc': 100 * piv[m].mean(), 'mean_rank': rk[m].mean(),
                         'n_datasets': len(piv)})
    out = pd.DataFrame(rows)
    acc = out.pivot(index='noise', columns='method', values='mean_acc')[[TB_LABEL[m] for m in TB_MAIN5]]
    rnk = out.pivot(index='noise', columns='method', values='mean_rank')[[TB_LABEL[m] for m in TB_MAIN5]]
    print(f"\n{'=' * 78}\n  خلاصه | kernel={kernel} | "
          f"دیتاست‌ها: {int(out.n_datasets.min())}\n{'=' * 78}")
    print("میانگین دقت (%):"); print(acc.round(2).to_string())
    print("\nمیانگین رتبه (کمتر بهتر):"); print(rnk.round(2).to_string())
    out.to_csv(OUTPUT_DIR / f"summary_{kernel}.csv", index=False)
    return out

def tb_run_linear_then_rbf():
    tb_assert_engine()
    tb_run_fingerprint()
    global KERNELS_TO_RUN, TB_FINAL_REPORTS
    tb_assert_engine()
    print("\n" + "#" * 78 + "\n#  مرحله‌ی ۱: کرنل خطی، همه‌ی نویزها\n" + "#" * 78)
    KERNELS_TO_RUN, TB_FINAL_REPORTS = ["linear"], False
    main_binary()
    tb_kernel_summary("linear")
    print(f"\n" + "#" * 78 + f"\n#  مرحله‌ی ۲: کرنل غیرخطی ({TB_NONLINEAR})، همه‌ی نویزها\n" + "#" * 78)
    KERNELS_TO_RUN, TB_FINAL_REPORTS = [TB_NONLINEAR], True
    col = main_binary()
    tb_kernel_summary(TB_NONLINEAR)
    return col

In [ ]:
import shutil as _shutil

SEED_FROM_PREVIOUS = False
_OLD_DIR = Path(r"C:\eshaghi\outputs_INS17")
_KEEP_DS = {"Australian", "Heart_Local", "spolice", "Sonar_UCI", "Iris_bin", "Vote",
            "Monk_2", "Mammographic"}
_UNIT_FILES = ["completed_units.csv", "progress_summary.csv", "nested_outer_folds.csv",
               "selected_hyperparameters.csv", "raw_selection_grid.csv"]

def _tb_current_cfg():
    return {"version": TB_CODE_VERSION, "protocol": TB_PROTOCOL, "outer": TB_OUTER_FOLDS,
            "inner": TB_CV_FOLDS, "c_grid": [float(v) for v in TB_C_GRID],
            "c3_grid": [float(v) for v in TB_C3_GRID], "seeds": list(RANDOM_STATES),
            "noise": [float(v) for v in LABEL_NOISE_RATIOS], "methods": list(TB_METHODS),
            "rob_arms": ["Rob", "RobHua", "RobSelf"], "nonlinear": TB_NONLINEAR,
            "gb_rule": TB_GB_RULE, "gbf_rule": TB_GBF_RULE, "gbf_eps": TB_GBF_EPS,
            "cqs": [CQS_EPS, CQS_MAXIT, CQS_TOL]}

def tb_seed_from_previous_run():
    new_dir = Path(OUTPUT_DIR)
    if TB_KDE_LOO or not SEED_FROM_PREVIOUS:
        print("[SEED-BAL8] کپی از اجرای قبلی غیرفعال است؛ همه‌ی واحدها اجرا می‌شوند.")
        return
    if (new_dir / "completed_units.csv").exists():
        print("[SEED-BAL8] پوشه‌ی خروجی از قبل پیشرفت دارد؛ کپی دوباره انجام نمی‌شود.")
        return
    old_cfg_f = _OLD_DIR / "run_config.json"
    if not old_cfg_f.exists():
        raise RuntimeError(f"[SEED-BAL8] {old_cfg_f} پیدا نشد.")
    old = json.loads(old_cfg_f.read_text(encoding="utf-8"))
    cur = _tb_current_cfg()
    diff = sorted(k for k in cur if cur[k] != old.get(k))
    if diff:
        raise RuntimeError(f"[SEED-BAL8] تنظیمات اجرای قبلی با تنظیمات فعلی فرق دارد "
                           f"(تفاوت در: {diff})؛ نتایج قبلی قابل استفاده نیست.")
    new_dir.mkdir(parents=True, exist_ok=True)
    for fn in _UNIT_FILES:
        src = _OLD_DIR / fn
        if not src.exists():
            continue
        df = pd.read_csv(src)
        n0 = len(df)
        if "dataset" in df.columns:
            df = df[df["dataset"].isin(_KEEP_DS)]
        df.to_csv(new_dir / fn, index=False, encoding="utf-8")
        print(f"  [SEED-BAL8] {fn}: {len(df)} سطر کپی شد ({n0 - len(df)} سطر حذف شد).")
    for src in _OLD_DIR.iterdir():
        if src.is_file() and src.name.startswith(("synthetic", "adversarial")):
            _shutil.copy2(src, new_dir / src.name)
    done = pd.read_csv(new_dir / "completed_units.csv")
    print(f"[SEED-BAL8] {len(done)} واحد کاری از اجرای قبلی منتقل شد؛ "
          f"دیتاست‌ها: {sorted(done['dataset'].unique())}")

tb_seed_from_previous_run()

In [ ]:
def _twin_line(ax, w, b, xlim, ylim, **kw):
    w = np.asarray(w, float).ravel()
    b = float(np.asarray(b).ravel()[0])
    if abs(w[1]) > 1e-12 * max(1.0, abs(w[0])):
        xs = np.linspace(xlim[0], xlim[1], 400)
        ax.plot(xs, -(w[0] * xs + b) / w[1], **kw)
    elif abs(w[0]) > 0:
        ax.plot([-b / w[0]] * 2, ylim, **kw)

def plot_twin_hyperplanes(n_per_class=150, seed=0, n_flip=1, c=1.0, bandwidth=0.7,
                          save_name="twin_hyperplanes_flipped_point", show=True):
    X, y = _adv_make_data(n_per_class, seed=seed)
    flip_idx = _adv_far_side_idx(X, y, n_flip)
    y_noisy = y.copy()
    y_noisy[flip_idx] *= -1

    models = [
        ("LSTSVM (without label noise)", lambda: StdLSTSVM(c1=c, c2=c, kernel='linear'), 'blue', y),
        ("LSTSVM",       lambda: StdLSTSVM(c1=c, c2=c, kernel='linear'), 'green', y_noisy),
        ("KDEWLSTSVM",   lambda: RobustGMMLSTSVM(c1=c, c2=c, kernel='linear', method='kde',
                                                 bandwidth=bandwidth), 'red', y_noisy),
        ("LSWLTSVM",     lambda: LSWLTSVM(c1=c, c2=c, k_neighbor=5, t_param=1.0,
                                          kernel='linear'), 'orange', y_noisy),
        ("GBFTSVM",      lambda: _tb_rival(c), 'purple', y_noisy),
        ("LS-GBLSTSVM",  lambda: GBLSTSVM(c1=c, c2=c, pur=0.985, delbals=2,
                                          kernel='linear'), 'saddlebrown', y_noisy),
        ("CL1QTSVM",     lambda: CL1QTSVM(c1=c, c2=c, kernel='linear'), 'deeppink', y_noisy),
    ]

    pad = 1.0
    xlim = (X[:, 0].min() - pad, X[:, 0].max() + pad)
    ylim = (X[:, 1].min() - pad, X[:, 1].max() + pad)

    fig, ax = plt.subplots(figsize=(12, 8))
    pos, neg = (y_noisy == 1), (y_noisy == -1)
    ax.scatter(X[pos, 0], X[pos, 1], marker='*', s=45, c='red', alpha=0.6, label='Class +1')
    ax.scatter(X[neg, 0], X[neg, 1], marker='o', s=25, c='blue', alpha=0.5, label='Class -1')
    ax.scatter(X[flip_idx, 0], X[flip_idx, 1], s=220, facecolors='none', edgecolors='lime',
               linewidths=2.2, label='The Flipped Point' if n_flip == 1 else 'Flipped Points')

    for name, build, col, yy in models:
        m = build()
        m.fit(X, yy)
        _twin_line(ax, m.w1, m.b1, xlim, ylim, color=col, lw=2.0, ls='-', label=name)
        _twin_line(ax, m.w2, m.b2, xlim, ylim, color=col, lw=1.4, ls='--', alpha=0.8, label=name)

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(r'$x_1$', fontsize=12)
    ax.set_ylabel(r'$x_2$', fontsize=12)
    ax.grid(True, ls=':', alpha=0.6)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=9,
              title='solid: class +1 plane\ndashed: class -1 plane', title_fontsize=9)
    fig.tight_layout()
    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{save_name}.pdf", bbox_inches='tight')
    fig.savefig(out / f"{save_name}.png", dpi=150, bbox_inches='tight')
    print(f"[TWIN-PLANES] ذخیره شد: {out / (save_name + '.pdf')}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig

plot_twin_hyperplanes()

In [ ]:
SMOKE = False

RUN_SYNTHETIC   = True
RUN_ADVERSARIAL = True
RUN_BINARY      = True

EXCLUDE_DATASETS = ["Banknote", "Cylinder_bands", "Hepatitis",

                    "Diabets_Local", "Breastcancer", "Ionosphere", "liver-disorders",
                    "Haberman", "Tic_tac_toe", "SAheart", "Appendicitis", "SPECTF_heart"]
DATASETS = [d for d in DATASETS if d["name"] not in EXCLUDE_DATASETS]

if any(d["name"] == "Sonar" for d in DATASETS):
    DATASETS = [d for d in DATASETS if d["name"] != "Sonar_UCI"]
_EXPECTED_BASE = {"Australian", "Heart_Local", "spolice", "Iris_bin", "Vote",
                  "Monk_2", "Mammographic"}
_EXPECTED_EXT = set(EXT_DATASET_NAMES) - set(EXCLUDE_DATASETS)
_names = [d["name"] for d in DATASETS]
_sonar = [n for n in _names if n in ("Sonar", "Sonar_UCI")]
_rest = set(_names) - {"Sonar", "Sonar_UCI"}
if len(_sonar) != 1 or _rest != _EXPECTED_BASE | _EXPECTED_EXT:
    raise RuntimeError(f"[DATA-CHECK] فهرست دیتاست‌ها با انتظار نمی‌خواند؛ "
                       f"اضافه: {sorted(_rest - _EXPECTED_BASE - _EXPECTED_EXT)} | "
                       f"کم: {sorted((_EXPECTED_BASE | _EXPECTED_EXT) - _rest)} | Sonar: {_sonar}")
print(f"[DATA-CHECK] {len(_names)} دیتاست: {sorted(_names)}")

if SMOKE:
    RANDOM_STATES      = RANDOM_STATES[:1]
    LABEL_NOISE_RATIOS = LABEL_NOISE_RATIOS[:1]
    CURRENT_LABEL_NOISE = LABEL_NOISE_RATIOS[0]
    DATASETS = DATASETS[:2] if len(DATASETS) >= 2 else DATASETS
    print('!'*68)
    print('!!  حالت SMOKE روشن است: نتایج این اجرا برای مقاله معتبر نیست.')
    print(f'!!  دیتاست‌ها: {[c["name"] for c in DATASETS]}')
    print('!'*68)

print("دیتاست‌های فعال:", [d["name"] for d in DATASETS])

if RUN_SYNTHETIC:
    print('\n' + '='*60 + '\n  داده‌ی مصنوعی (بخش ۴.۱ مقاله)\n' + '='*60)
    if (OUTPUT_DIR / 'synthetic_decision_boundaries.pdf').exists():
        print('[RESUME] خروجی موجود است؛ رد شد.')
    else:
        synthetic_results = run_synthetic_experiment()

if RUN_ADVERSARIAL:
    print('\n' + '='*60 + '\n  نویز خصمانه\n' + '='*60)
    df_adv, summary_adv = run_adversarial_experiment()

if RUN_BINARY:
    print('\n' + '='*60 + '\n  بنچمارک دوکلاسه\n' + '='*60)
    progress_report()
    binary_collector = tb_run_linear_then_rbf()

print('\nDone.')

In [ ]:
import shutil
import matplotlib
import matplotlib.pyplot as plt

DROP_FROM_REPORTS = ["Banknote", "Sonar"]

FILTERED_DIR = OUTPUT_DIR / "filtered_reports"
FILTERED_DIR.mkdir(exist_ok=True, parents=True)

_src = pd.read_csv(PROGRESS_CSV)
print(f"سطرهای اولیه: {len(_src)} | دیتاست‌ها: {sorted(_src['dataset'].unique())}")

_missing = [d for d in DROP_FROM_REPORTS if d not in set(_src["dataset"])]
if _missing:
    print(f"[توجه] این نام‌ها در داده یافت نشدند و چیزی حذف نشد: {_missing}")

_filt = _src[~_src["dataset"].isin(DROP_FROM_REPORTS)].copy()
print(f"سطرهای پس از فیلتر: {len(_filt)} | "
      f"دیتاست‌ها: {sorted(_filt['dataset'].unique())}")
print(f"تعداد دیتاست: {_src['dataset'].nunique()} -> {_filt['dataset'].nunique()}")
if "Sonar_UCI" in set(_filt["dataset"]):
    print("[توجه] Sonar_UCI در جدول‌ها و شکل‌ها با نام نمایشی «Sonar» ظاهر "
          "می‌شود؛ این همان دیتاست استاندارد Sonar است و با نسخه‌ی محلیِ "
          "حذف‌شده اشتباه نشود.")

if _filt.empty:
    raise SystemExit("پس از فیلتر هیچ داده‌ای نماند؛ نام دیتاست‌ها را بررسی کنید.")

_col = ResultsCollector()
_col.rows = _filt.drop_duplicates(
    subset=["seed", "noise", "dataset", "kernel", "method"], keep="last"
)[["seed", "noise", "dataset", "kernel", "method",
   "mean_accuracy", "std_accuracy", "time_sec"]].to_dict("records")
print(f"Collector ساخته شد با {len(_col.rows)} سطر.")

_orig_set_title = matplotlib.axes.Axes.set_title
_orig_suptitle = matplotlib.figure.Figure.suptitle

_KEEP_TITLES = set()
for _d in _src["dataset"].unique():
    _KEEP_TITLES.add(str(_d).strip())
    _KEEP_TITLES.add(str(disp(str(_d))).strip())

def _selective_set_title(self, label="", *a, **k):
    if isinstance(label, str) and label.strip() in _KEEP_TITLES:

        return _orig_set_title(self, disp(label.strip()), *a, **k)
    return None

def _no_suptitle(self, *a, **k):
    return None

matplotlib.axes.Axes.set_title = _selective_set_title
matplotlib.figure.Figure.suptitle = _no_suptitle

try:

    print("\n" + "=" * 60)
    print("  جدول‌های سبک-مقاله (بدون دیتاست‌های حذف‌شده)")
    print("=" * 60)
    for _noise in sorted(_col.to_dataframe()["noise"].unique()):
        for _kernel in ("linear", globals().get("TB_NONLINEAR", "rbf")):
            try:
                _col.paper_style_table(_kernel, _noise)
            except Exception as e:
                print(f"  [SKIP] جدول {_kernel}/{_noise}: {e}")

    print("\n" + "=" * 60)
    print("  آزمون‌های آماری")
    print("=" * 60)
    for _fn, _name in ((lambda: _col.wilcoxon_report_by_dataset(reference="Rob"),
                        "Wilcoxon سطح دیتاست"),
                       (lambda: _col.wilcoxon_two_sided_holm(reference="Rob"),
                        "Wilcoxon دوطرفه + Holm"),
                       (lambda: _col.friedman_report(), "Friedman"),
                       (lambda: _col.friedman_posthoc_holm(reference="Rob"),
                        "Friedman post-hoc")):
        try:
            _fn()
        except Exception as e:
            print(f"  [SKIP] {_name}: {e}")

    print("\n" + "=" * 60)
    print("  شکل‌ها (بدون عنوان بالای شکل)")
    print("=" * 60)
    for _fn, _name in (
        (lambda: _col.noise_curve_plot(FILTERED_DIR), "accuracy vs noise"),
        (lambda: _col.noise_curve_per_dataset(FILTERED_DIR), "accuracy vs noise per dataset"),
        (lambda: _col.critical_difference_plot(FILTERED_DIR / "critical_difference"), "CD diagrams"),
        (lambda: _col.boxplots(FILTERED_DIR / "boxplots_binary"), "boxplots"),
        (lambda: _col.bandwidth_sensitivity_plot(FILTERED_DIR), "bandwidth sensitivity"),
    ):
        try:
            _fn()
        except Exception as e:
            print(f"  [SKIP] {_name}: {type(e).__name__}: {e}")

    _orig_results_dir = RESULTS_DIR
    try:
        globals()["RESULTS_DIR"] = FILTERED_DIR
        _col.ablation_table()
    except Exception as e:
        print(f"  [SKIP] ablation: {type(e).__name__}: {e}")
    finally:
        globals()["RESULTS_DIR"] = _orig_results_dir

    try:
        _col.to_excel(FILTERED_DIR / "binary_results_summary_filtered.xlsx")
    except Exception as e:
        print(f"  [SKIP] Excel: {e}")

    try:
        _jdir = FILTERED_DIR / "json"
        _jdir.mkdir(exist_ok=True, parents=True)
        _summary = {}
        for (_k, _n), _sub in _filt.groupby(["kernel", "noise"]):
            _summary.setdefault(str(_k), {})[str(_n)] = {}
            for (_ds, _m), _g in _sub.groupby(["dataset", "method"]):
                _per_seed = _g.groupby("seed")["mean_accuracy"].mean().values
                _summary[str(_k)][str(_n)].setdefault(disp(_ds), {})[
                    METHOD_LABELS.get(_m, _m)] = {
                        "mean_accuracy": float(np.mean(_per_seed)),
                        "std_accuracy": float(np.std(_per_seed)),
                        "n_seeds": int(len(_per_seed)),
                }
        (_jdir / "results_filtered.json").write_text(
            json.dumps({"excluded_datasets": DROP_FROM_REPORTS,
                        "summary": _summary}, indent=1, ensure_ascii=False),
            encoding="utf-8")
        print(f"[JSON] نوشته شد: {_jdir / 'results_filtered.json'}")
    except Exception as e:
        print(f"  [SKIP] JSON: {e}")

finally:

    matplotlib.axes.Axes.set_title = _orig_set_title
    matplotlib.figure.Figure.suptitle = _orig_suptitle

print(f"\nتمام شد. خروجی‌ها در: {FILTERED_DIR}")
print("نتایج و شکل‌های قبلی دست‌نخورده باقی مانده‌اند.")